# NPC DSC 2026 — Final Triple V4 Submission

Notebook ringkas untuk membuat ulang kandidat submission terbaik yang sudah diuji (public score 0.64871): Triple V4 + confidence/disagreement, tepat 245 Unknown.

**Tidak ada training ulang.** Tambahkan output notebook sumber sebagai Input bila file probabilitas belum ada di `/kaggle/working`. Tambahkan juga dataset kompetisi NPC DSC 2026. Jalankan satu cell kode di bawah.

Output: `submission_triple_v4_rebuilt_confidence_disagreement_u245.csv` dan file diagnostik. Jika probabilitas yang ditemukan berbeda dari run sebelumnya, skor 0.64871 tidak dijamin berulang.


In [1]:
import os

required = [
    "test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy",
    "tta_test_crop_probabilities.npz",
    "test_effnet_b2_audio8s_tta3.npy",
    "test_convnext_tiny_pretrained_v4_resize224_tta3.npy",
]

found = {}

for base in ["/kaggle/input", "/kaggle/working"]:
    if not os.path.exists(base):
        continue

    for root, dirs, files in os.walk(base):
        for name in required:
            if name in files and name not in found:
                found[name] = os.path.join(root, name)

print("HASIL PEMERIKSAAN OUTPUT TRIPLE V4")
print("=" * 55)

for name in required:
    if name in found:
        print("✅ DITEMUKAN:", name)
        print("   Path:", found[name])
    else:
        print("❌ BELUM DITEMUKAN:", name)

print(f"\nTotal: {len(found)}/{len(required)} file ditemukan.")

HASIL PEMERIKSAAN OUTPUT TRIPLE V4
✅ DITEMUKAN: test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy
   Path: /kaggle/input/notebooks/radithiafatir/testttrisan/test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy
✅ DITEMUKAN: tta_test_crop_probabilities.npz
   Path: /kaggle/input/notebooks/radithiafatir/testttrisan/tta_test_crop_probabilities.npz
✅ DITEMUKAN: test_effnet_b2_audio8s_tta3.npy
   Path: /kaggle/input/notebooks/radithiafatir/testttrisan/test_effnet_b2_audio8s_tta3.npy
✅ DITEMUKAN: test_convnext_tiny_pretrained_v4_resize224_tta3.npy
   Path: /kaggle/input/notebooks/radithiafatir/testttrisan/test_convnext_tiny_pretrained_v4_resize224_tta3.npy

Total: 4/4 file ditemukan.


In [2]:
# ================================================================
# NPC DSC 2026 — FINAL TRIPLE V4 SUBMISSION (MINIMAL)
# Rebuild only the best-known submission ranking:
#   Triple V4 probabilities + confidence/disagreement -> exactly 245 Unknown
# NO model training. Reuses saved .npy/.npz outputs from the notebook input.
# ================================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd

LABELS = ["Bondol Jawa", "Cendrawasih", "Kutilang"]
N_UNKNOWN = 245
CONF_WEIGHT = 0.50
DISAGREE_WEIGHT = 0.50
OUT = "/kaggle/working"

# ---- Locate competition dataset ----
def find_competition_root():
    preferred = "/kaggle/input/competitions/schematics-2026"
    if (os.path.isfile(os.path.join(preferred, "sample_submission.csv"))
        and os.path.isfile(os.path.join(preferred, "train", "metadata.csv"))):
        return preferred

    for root, dirs, files in os.walk("/kaggle/input"):
        if "sample_submission.csv" in files and os.path.isfile(
            os.path.join(root, "train", "metadata.csv")
        ):
            return root
    raise FileNotFoundError(
        "Dataset kompetisi tidak ditemukan. Tambahkan NPC DSC 2026 melalui Add Input."
    )

ROOT = find_competition_root()
SAMPLE_PATH = os.path.join(ROOT, "sample_submission.csv")
META_PATH = os.path.join(ROOT, "train", "metadata.csv")

# ---- Locate one consistent saved-output bundle ----
ARTIFACT_NAMES = [
    "test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy",
    "tta_test_crop_probabilities.npz",
    "test_effnet_b2_audio8s_tta3.npy",
    "test_convnext_tiny_pretrained_v4_resize224_tta3.npy",
]
OPTIONAL_BASELINE = "submission_triple_v4_rebuilt_u245.csv"


def collect_in_root(root, target_names):
    found = {}
    target_set = set(target_names)
    # Check direct files first.
    for name in target_names:
        p = os.path.join(root, name)
        if os.path.isfile(p):
            found[name] = p
    if len(found) == len(target_set):
        return found
    for current, dirs, files in os.walk(root):
        for name in target_set.intersection(files):
            found.setdefault(name, os.path.join(current, name))
        if len(found) == len(target_set):
            break
    return found

# Same /kaggle/working is preferred when running in the original session.
artifact_paths = collect_in_root(OUT, ARTIFACT_NAMES + [OPTIONAL_BASELINE])
if not set(ARTIFACT_NAMES).issubset(artifact_paths):
    artifact_paths = {}
    input_root = "/kaggle/input"
    top_roots = [str(p) for p in Path(input_root).iterdir() if p.is_dir()]
    for candidate_root in top_roots:
        found = collect_in_root(candidate_root, ARTIFACT_NAMES + [OPTIONAL_BASELINE])
        if set(ARTIFACT_NAMES).issubset(found):
            artifact_paths = found
            break

missing = [name for name in ARTIFACT_NAMES if name not in artifact_paths]
if missing:
    raise FileNotFoundError(
        "File probabilitas yang diperlukan belum ditemukan:\n- "
        + "\n- ".join(missing)
        + "\nSimpan notebook sumber dengan output, lalu tambahkan output-nya melalui Add Input."
    )

ENSEMBLE_PATH = artifact_paths[ARTIFACT_NAMES[0]]
RESNET_PATH = artifact_paths[ARTIFACT_NAMES[1]]
B2_PATH = artifact_paths[ARTIFACT_NAMES[2]]
CONVNEXT_PATH = artifact_paths[ARTIFACT_NAMES[3]]
BASELINE_PATH = artifact_paths.get(OPTIONAL_BASELINE)

print("Competition root:", ROOT)
print("Saved artifact files:")
for name in ARTIFACT_NAMES:
    print("  ", name, "->", artifact_paths[name])

# ---- Load sample/meta and validate labels ----
sample = pd.read_csv(SAMPLE_PATH)
meta = pd.read_csv(META_PATH)
if "file_id" not in sample.columns or "label" not in meta.columns:
    raise ValueError("Metadata/sample submission tidak memiliki kolom yang diharapkan.")
if sorted(meta["label"].astype(str).unique().tolist()) != LABELS:
    raise ValueError("Daftar kelas train berbeda dari kelas Triple V4 yang diharapkan.")
sample["file_id"] = sample["file_id"].astype(str)
if sample["file_id"].duplicated().any():
    raise ValueError("sample_submission berisi file_id duplikat.")
n_test = len(sample)


def normalize_probs(P, name):
    P = np.asarray(P, dtype=np.float64)
    if P.ndim != 2 or not np.isfinite(P).all():
        raise ValueError(f"{name}: probabilitas tidak valid, shape={P.shape}")
    if (P < -1e-5).any() or (P > 1.00001).any():
        raise ValueError(f"{name}: nilai berada di luar rentang 0-1.")
    P = np.clip(P, 1e-12, None)
    return P / P.sum(axis=1, keepdims=True)

# ---- Ensemble probabilities and per-model probabilities ----
P_ensemble = normalize_probs(np.load(ENSEMBLE_PATH), "Triple V4 ensemble")
if P_ensemble.shape != (n_test, len(LABELS)):
    raise ValueError(f"Shape ensemble {P_ensemble.shape}; expected {(n_test, len(LABELS))}")

with np.load(RESNET_PATH) as z:
    if not {"test_r5", "test_r8"}.issubset(z.files):
        raise ValueError("ResNet archive harus memiliki test_r5 dan test_r8.")
    r5 = np.asarray(z["test_r5"], dtype=np.float64)
    r8 = np.asarray(z["test_r8"], dtype=np.float64)

if (r5.ndim != 3 or r8.ndim != 3 or r5.shape[0] != n_test or r8.shape[0] != n_test
    or r5.shape[1] < 5 or r8.shape[1] < 5 or r5.shape[2] != 3 or r8.shape[2] != 3):
    raise ValueError(f"ResNet shapes tidak sesuai: r5={r5.shape}, r8={r8.shape}")

selected_crops = [0, 2, 4]
P_r5 = normalize_probs(r5[:, selected_crops, :].mean(axis=1), "ResNet 5s")
P_r8 = normalize_probs(r8[:, selected_crops, :].mean(axis=1), "ResNet 8s")
P_resnet = normalize_probs(0.25 * P_r5 + 0.75 * P_r8, "ResNet combined")
P_b2 = normalize_probs(np.load(B2_PATH), "EfficientNet-B2")
P_convnext = normalize_probs(np.load(CONVNEXT_PATH), "ConvNeXt V4")

for name, P in [("ResNet", P_resnet), ("EfficientNet-B2", P_b2), ("ConvNeXt", P_convnext)]:
    if P.shape != P_ensemble.shape:
        raise ValueError(f"{name}: shape {P.shape} tidak cocok dengan ensemble {P_ensemble.shape}.")

P_rebuilt = normalize_probs(
    0.55 * P_resnet + 0.05 * P_b2 + 0.40 * P_convnext,
    "Reconstructed ensemble",
)
max_diff = float(np.max(np.abs(P_rebuilt - P_ensemble)))
print("Max difference saved/rebuilt ensemble:", round(max_diff, 8))
if max_diff > 0.005:
    raise ValueError("Probabilitas komponen tidak cocok dengan ensemble tersimpan. Hentikan untuk menghindari submission yang salah.")

# ---- Confidence + disagreement ranking (same algorithm as best score 0.64871) ----
confidence = P_ensemble.max(axis=1)
v4_pred = np.asarray(LABELS)[P_ensemble.argmax(axis=1)]

def tv_distance(Pa, Pb):
    return 0.5 * np.abs(Pa - Pb).sum(axis=1)

disagreement = (
    tv_distance(P_resnet, P_b2)
    + tv_distance(P_resnet, P_convnext)
    + tv_distance(P_b2, P_convnext)
) / 3.0

def percentile_rank(values):
    return pd.Series(np.asarray(values)).rank(method="average", pct=True).to_numpy(dtype=np.float64)

confidence_rank = percentile_rank(confidence)
disagreement_reverse_rank = percentile_rank(-disagreement)
combined_score = CONF_WEIGHT * confidence_rank + DISAGREE_WEIGHT * disagreement_reverse_rank
if not np.isfinite(combined_score).all():
    raise ValueError("Combined score mengandung NaN/inf.")

unknown_order = np.argsort(combined_score, kind="stable")
selected_indices = unknown_order[:N_UNKNOWN]
unknown_mask = np.zeros(n_test, dtype=bool)
unknown_mask[selected_indices] = True

if N_UNKNOWN >= n_test:
    raise ValueError("N_UNKNOWN harus lebih kecil dari jumlah sampel test.")
final_labels = v4_pred.astype(object)
final_labels[unknown_mask] = "Unknown"
if int(np.sum(final_labels == "Unknown")) != N_UNKNOWN:
    raise AssertionError("Jumlah Unknown bukan tepat 245.")

# ---- Optional baseline comparison; does not affect generated labels ----
baseline_labels = None
if BASELINE_PATH and os.path.isfile(BASELINE_PATH):
    baseline = pd.read_csv(BASELINE_PATH)
    if {"file_id", "label"}.issubset(baseline.columns):
        baseline["file_id"] = baseline["file_id"].astype(str)
        baseline = baseline.set_index("file_id").reindex(sample["file_id"])
        if not baseline["label"].isna().any():
            baseline_labels = baseline["label"].astype(str).to_numpy()
            base_unknown = baseline_labels == "Unknown"
            known_base = ~base_unknown
            mismatches = int(np.sum(baseline_labels[known_base] != v4_pred[known_base]))
            print("Baseline argmax mismatches:", mismatches)
            print("Baseline Unknown count:", int(base_unknown.sum()))

# ---- Save latest submission + diagnostics ----
SUBMISSION_OUT = os.path.join(OUT, "submission_triple_v4_rebuilt_confidence_disagreement_u245.csv")
DIAGNOSTIC_OUT = os.path.join(OUT, "diagnostic_triple_v4_rebuilt_confidence_disagreement_u245.csv")

submission = pd.DataFrame({"file_id": sample["file_id"].to_numpy(), "label": final_labels})
submission.to_csv(SUBMISSION_OUT, index=False)

diag = pd.DataFrame({
    "file_id": sample["file_id"].to_numpy(),
    "v4_predicted_label": v4_pred,
    "v4_confidence": confidence,
    "model_disagreement": disagreement,
    "confidence_rank": confidence_rank,
    "disagreement_reverse_rank": disagreement_reverse_rank,
    "combined_score": combined_score,
    "candidate_unknown": unknown_mask,
    "final_label": final_labels,
})
if baseline_labels is not None:
    diag["baseline_label"] = baseline_labels
    diag["newly_unknown"] = unknown_mask & (baseline_labels != "Unknown")
    diag["released_from_unknown"] = (~unknown_mask) & (baseline_labels == "Unknown")

diag.to_csv(DIAGNOSTIC_OUT, index=False)

print("\n" + "=" * 68)
print("FINAL SUBMISSION CREATED — NOT SUBMITTED")
print("Submission:", SUBMISSION_OUT)
print("Diagnostic:", DIAGNOSTIC_OUT)
print("Rows:", len(submission))
print("Unknown count:", int((submission["label"] == "Unknown").sum()))
print("Label distribution:\n", submission["label"].value_counts().to_string())
print("Reference public score from previous run: 0.64871 (must be rechecked if source probabilities differ).")


Competition root: /kaggle/input/competitions/schematics-2026
Saved artifact files:
   test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy -> /kaggle/input/notebooks/radithiafatir/testttrisan/test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy
   tta_test_crop_probabilities.npz -> /kaggle/input/notebooks/radithiafatir/testttrisan/tta_test_crop_probabilities.npz
   test_effnet_b2_audio8s_tta3.npy -> /kaggle/input/notebooks/radithiafatir/testttrisan/test_effnet_b2_audio8s_tta3.npy
   test_convnext_tiny_pretrained_v4_resize224_tta3.npy -> /kaggle/input/notebooks/radithiafatir/testttrisan/test_convnext_tiny_pretrained_v4_resize224_tta3.npy
Max difference saved/rebuilt ensemble: 5e-08
Baseline argmax mismatches: 0
Baseline Unknown count: 245

FINAL SUBMISSION CREATED — NOT SUBMITTED
Submission: /kaggle/working/submission_triple_v4_rebuilt_confidence_disagreement_u245.csv
Diagnostic: /kaggle/working/diagnostic_triple_v4_rebuilt_confidence_disagreement_u24

In [4]:
import os

ROOTS = [
    "/kaggle/working",
    "/kaggle/input/notebooks/radithiafatir/testttrisan",
]

OOF_FILES = [
    "oof_resnet50_ensemble_tta3_seed16.npy",
    "oof_efficientnet_b2_audio8s_tta3_seed16.npy",
    "oof_convnext_tiny_audio8s_tta3_seed16_pretrained_v4_resize224.npy",
]

print("=" * 65)
print("PEMERIKSAAN OOF TRIPLE V4")
print("=" * 65)

for name in OOF_FILES:
    matches = []

    for root in ROOTS:
        path = os.path.join(root, name)
        if os.path.isfile(path):
            matches.append(path)

    if matches:
        print(f"\n✅ {name}")
        for path in matches:
            print("   ", path)
    else:
        print(f"\n❌ BELUM DITEMUKAN: {name}")

PEMERIKSAAN OOF TRIPLE V4

✅ oof_resnet50_ensemble_tta3_seed16.npy
    /kaggle/input/notebooks/radithiafatir/testttrisan/oof_resnet50_ensemble_tta3_seed16.npy

✅ oof_efficientnet_b2_audio8s_tta3_seed16.npy
    /kaggle/input/notebooks/radithiafatir/testttrisan/oof_efficientnet_b2_audio8s_tta3_seed16.npy

✅ oof_convnext_tiny_audio8s_tta3_seed16_pretrained_v4_resize224.npy
    /kaggle/input/notebooks/radithiafatir/testttrisan/oof_convnext_tiny_audio8s_tta3_seed16_pretrained_v4_resize224.npy


In [5]:
# ================================================================
# OFFLINE EVALUATION — TRIPLE V4 OOF
#
# Tujuan:
# 1. Memuat ketiga OOF probability array.
# 2. Menghitung kembali performa ensemble Triple V4.
# 3. Membandingkan confidence-only, disagreement-only,
#    dan confidence + disagreement.
# 4. Mengukur apakah sampel yang ditolak lebih banyak
#    mengandung kesalahan klasifikasi pada OOF.
#
# Tidak training model.
# Tidak membuat submission baru.
# Tidak memakai jatah submission.
# ================================================================

import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

ROOT = "/kaggle/input/competitions/schematics-2026"
OUT = "/kaggle/working"

OOF_PATHS = {
    "ResNet50": os.path.join(
        "/kaggle/input/notebooks/radithiafatir/testttrisan",
        "oof_resnet50_ensemble_tta3_seed16.npy"
    ),
    "EfficientNet-B2": os.path.join(
        "/kaggle/input/notebooks/radithiafatir/testttrisan",
        "oof_efficientnet_b2_audio8s_tta3_seed16.npy"
    ),
    "ConvNeXt-v4": os.path.join(
        "/kaggle/input/notebooks/radithiafatir/testttrisan",
        "oof_convnext_tiny_audio8s_tta3_seed16_pretrained_v4_resize224.npy"
    ),
}

LABELS = ["Bondol Jawa", "Cendrawasih", "Kutilang"]

# Bobot ensemble yang dipakai pada submission terbaik.
W_R, W_B2, W_C = 0.55, 0.05, 0.40

# Proporsi test yang dijadikan Unknown pada submission terbaik.
N_UNKNOWN = 245
N_TEST = 814
REJECT_FRACTION = N_UNKNOWN / N_TEST

# ------------------------------------------------
# 1. VALIDASI DAN LOAD FILE
# ------------------------------------------------
meta_path = os.path.join(ROOT, "train", "metadata.csv")
sample_path = os.path.join(ROOT, "sample_submission.csv")

for p in [meta_path, sample_path, *OOF_PATHS.values()]:
    if not os.path.isfile(p):
        raise FileNotFoundError(f"File tidak ditemukan: {p}")

meta = pd.read_csv(meta_path)
sample = pd.read_csv(sample_path)

y_text = meta["label"].astype(str).to_numpy()
classes = np.sort(np.unique(y_text))

if list(classes) != LABELS:
    raise ValueError(f"Urutan kelas tidak sesuai: {classes.tolist()}")

def normalize_probs(P, name):
    P = np.asarray(P, dtype=np.float64)

    if P.ndim != 2 or not np.isfinite(P).all():
        raise ValueError(f"{name}: shape/nilai tidak valid {P.shape}")

    if (P < -1e-5).any() or (P > 1.00001).any():
        raise ValueError(f"{name}: nilai di luar rentang probabilitas.")

    P = np.clip(P, 1e-12, None)
    return P / P.sum(axis=1, keepdims=True)

P = {}

for name, path in OOF_PATHS.items():
    P[name] = normalize_probs(np.load(path), name)

    if P[name].shape != (len(meta), len(LABELS)):
        raise ValueError(
            f"{name}: shape {P[name].shape}, "
            f"target {(len(meta), len(LABELS))}"
        )

print("=" * 70)
print("OOF INPUT VALIDATION")
print("=" * 70)

print("Jumlah train:", len(meta))
print("Jumlah test:", len(sample))
print("Proporsi test Unknown:", round(REJECT_FRACTION, 4))

for name, probs in P.items():
    print(f"{name:18s}: {probs.shape}")

# ------------------------------------------------
# 2. BANGUN ENSEMBLE OOF
# ------------------------------------------------
P_ensemble = normalize_probs(
    W_R * P["ResNet50"]
    + W_B2 * P["EfficientNet-B2"]
    + W_C * P["ConvNeXt-v4"],
    "Triple V4 ensemble"
)

y_true = np.array([LABELS.index(label) for label in y_text])
y_pred = P_ensemble.argmax(axis=1)

correct = (y_pred == y_true)
confidence = P_ensemble.max(axis=1)

print("\n" + "=" * 70)
print("TRIPLE V4 OOF PERFORMANCE")
print("=" * 70)

print(f"Accuracy : {accuracy_score(y_true, y_pred):.5f}")
print(
    "Macro F1 :",
    f"{f1_score(y_true, y_pred, labels=np.arange(3), average='macro', zero_division=0):.5f}"
)

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=np.arange(3),
        target_names=LABELS,
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------
# 3. HITUNG DISAGREEMENT ANTARMODEL
# ------------------------------------------------
def tv_distance(A, B):
    # Total variation distance untuk tiap sampel.
    return 0.5 * np.abs(A - B).sum(axis=1)

disagreement = (
    tv_distance(P["ResNet50"], P["EfficientNet-B2"])
    + tv_distance(P["ResNet50"], P["ConvNeXt-v4"])
    + tv_distance(P["EfficientNet-B2"], P["ConvNeXt-v4"])
) / 3.0

def rank01(values):
    return pd.Series(np.asarray(values)).rank(
        method="average",
        pct=True
    ).to_numpy(dtype=np.float64)

# Confidence tinggi -> rank tinggi.
confidence_rank = rank01(confidence)

# Disagreement tinggi -> reverse rank rendah.
disagreement_reverse_rank = rank01(-disagreement)

# Ini sama dengan prinsip ranking yang dipakai pada kandidat terbaik.
combined_score = (
    0.50 * confidence_rank
    + 0.50 * disagreement_reverse_rank
)

# ------------------------------------------------
# 4. SIMULASIKAN REJECTION PADA PROPORSI YANG SAMA
# ------------------------------------------------
n_reject = int(np.ceil(REJECT_FRACTION * len(meta)))

scores = {
    "Confidence only": confidence,
    "Disagreement only": -disagreement,
    "Confidence + disagreement": combined_score,
}

masks = {}

for name, score in scores.items():
    # Skor kecil = lebih diprioritaskan untuk ditolak.
    mask = np.zeros(len(meta), dtype=bool)
    mask[np.argsort(score, kind="stable")[:n_reject]] = True
    masks[name] = mask

# ------------------------------------------------
# 5. BANDINGKAN ERROR RATE DAN KOMPOSISI KELAS
# ------------------------------------------------
all_error_rate = float((~correct).mean())
result_rows = []
class_rows = []

for name, mask in masks.items():
    rejected = mask
    retained = ~mask

    reject_error_rate = float((~correct[rejected]).mean())
    keep_error_rate = float((~correct[retained]).mean())

    error_lift = (
        reject_error_rate / all_error_rate
        if all_error_rate > 0 else np.nan
    )

    result_rows.append({
        "Strategy": name,
        "Rejected": int(rejected.sum()),
        "OOF_errors_rejected": int((~correct[rejected]).sum()),
        "Rejected_error_rate": reject_error_rate,
        "Retained_error_rate": keep_error_rate,
        "Overall_error_rate": all_error_rate,
        "Error_enrichment": error_lift,
        "Rejected_macro_F1_conditional": f1_score(
            y_true[retained],
            y_pred[retained],
            labels=np.arange(3),
            average="macro",
            zero_division=0
        ),
    })

    for i, label in enumerate(LABELS):
        class_mask = (y_true == i)

        class_rows.append({
            "Strategy": name,
            "Class": label,
            "Class_support": int(class_mask.sum()),
            "Class_rejected": int((rejected & class_mask).sum()),
            "Class_reject_rate": float(
                (rejected & class_mask).sum() / max(class_mask.sum(), 1)
            ),
            "Rejected_correct": int(
                (rejected & class_mask & correct).sum()
            ),
            "Rejected_wrong": int(
                (rejected & class_mask & ~correct).sum()
            ),
        })

results = pd.DataFrame(result_rows)
class_results = pd.DataFrame(class_rows)

print("\n" + "=" * 70)
print("REJECTION STRATEGY COMPARISON")
print("=" * 70)

with pd.option_context("display.max_columns", None, "display.width", 220):
    print(results.round(4).to_string(index=False))

print("\n" + "=" * 70)
print("CLASS-WISE REJECTION RATES")
print("=" * 70)

with pd.option_context("display.max_columns", None, "display.width", 180):
    print(
        class_results.pivot(
            index="Class",
            columns="Strategy",
            values="Class_reject_rate"
        ).round(4).to_string()
    )

# ------------------------------------------------
# 6. SIMPAN HASIL DIAGNOSTIK
# ------------------------------------------------
diagnostic_path = os.path.join(
    OUT, "triple_v4_oof_rejection_comparison.csv"
)

results_path = os.path.join(
    OUT, "triple_v4_oof_class_rejection_rates.csv"
)

results.to_csv(diagnostic_path, index=False)
class_results.to_csv(results_path, index=False)

print("\nFile diagnostik tersimpan:")
print(diagnostic_path)
print(results_path)

print("\nCatatan:")
print(
    "OOF ini hanya berisi tiga kelas yang dikenal. "
    "Analisis mengukur apakah strategi rejection menemukan sampel yang "
    "sulit atau salah diklasifikasikan, bukan membuktikan deteksi Unknown "
    "yang sesungguhnya. Belum ada submission baru yang dibuat."
)

OOF INPUT VALIDATION
Jumlah train: 1826
Jumlah test: 814
Proporsi test Unknown: 0.301
ResNet50          : (1826, 3)
EfficientNet-B2   : (1826, 3)
ConvNeXt-v4       : (1826, 3)

TRIPLE V4 OOF PERFORMANCE
Accuracy : 0.98412
Macro F1 : 0.94204

Classification report:
              precision    recall  f1-score   support

 Bondol Jawa     0.9911    0.9867    0.9889      1124
 Cendrawasih     0.8182    0.8675    0.8421        83
    Kutilang     0.9952    0.9952    0.9952       619

    accuracy                         0.9841      1826
   macro avg     0.9348    0.9498    0.9420      1826
weighted avg     0.9846    0.9841    0.9843      1826


REJECTION STRATEGY COMPARISON
                 Strategy  Rejected  OOF_errors_rejected  Rejected_error_rate  Retained_error_rate  Overall_error_rate  Error_enrichment  Rejected_macro_F1_conditional
          Confidence only       550                   29               0.0527               0.0000              0.0159            3.3200                   

In [6]:
# ================================================================
# OFFLINE WEIGHT SENSITIVITY TEST
# Triple V4 | Confidence + Disagreement | Fixed U245
#
# Tidak melatih model.
# Tidak mengubah submission terbaik.
# Tidak menggunakan jatah leaderboard.
# ================================================================

import os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score

ROOT = "/kaggle/input/competitions/schematics-2026"
WORK = "/kaggle/working"
SRC = "/kaggle/input/notebooks/radithiafatir/testttrisan"

LABELS = ["Bondol Jawa", "Cendrawasih", "Kutilang"]
N_UNKNOWN = 245
TEST_SIZE = 814
REJECT_RATE = N_UNKNOWN / TEST_SIZE

# (bobot confidence, bobot disagreement)
WEIGHT_GRID = [
    (1.0, 0.0),
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5),  # Metode terbaik sebelumnya: public 0.64871
    (0.4, 0.6),
    (0.3, 0.7),
    (0.0, 1.0),
]

def get_path(name):
    for base in [WORK, SRC]:
        path = os.path.join(base, name)
        if os.path.isfile(path):
            return path
    raise FileNotFoundError(
        f"{name} tidak ditemukan di {WORK} maupun {SRC}"
    )

def normalize_probs(P, name):
    P = np.asarray(P, dtype=np.float64)
    if P.ndim != 2 or not np.isfinite(P).all():
        raise ValueError(f"{name}: shape/nilai tidak valid {P.shape}")
    if (P < -1e-5).any() or (P > 1.00001).any():
        raise ValueError(f"{name}: nilai di luar rentang probabilitas")
    P = np.clip(P, 1e-12, None)
    return P / P.sum(axis=1, keepdims=True)

def rank01(x):
    return pd.Series(np.asarray(x)).rank(
        method="average", pct=True
    ).to_numpy(dtype=np.float64)

def tv(A, B):
    return 0.5 * np.abs(A - B).sum(axis=1)

def composite_score(confidence, disagreement, wc, wd):
    # Skor kecil => lebih diprioritaskan menjadi Unknown.
    return (
        wc * rank01(confidence)
        + wd * rank01(-disagreement)
    )

def select_lowest(score, n):
    idx = np.argsort(score, kind="stable")[:n]
    mask = np.zeros(len(score), dtype=bool)
    mask[idx] = True
    return mask

# ------------------------------------------------
# 1. PATHS DAN INPUT
# ------------------------------------------------
oof_paths = {
    "ResNet50": "oof_resnet50_ensemble_tta3_seed16.npy",
    "EfficientNet-B2": "oof_efficientnet_b2_audio8s_tta3_seed16.npy",
    "ConvNeXt-v4": (
        "oof_convnext_tiny_audio8s_tta3_seed16_pretrained_v4_resize224.npy"
    ),
}
test_paths = {
    "ResNetArchive": "tta_test_crop_probabilities.npz",
    "B2": "test_effnet_b2_audio8s_tta3.npy",
    "ConvNeXt": "test_convnext_tiny_pretrained_v4_resize224_tta3.npy",
}

meta = pd.read_csv(os.path.join(ROOT, "train", "metadata.csv"))
sample = pd.read_csv(os.path.join(ROOT, "sample_submission.csv"))
y_text = meta["label"].astype(str).to_numpy()

if sorted(np.unique(y_text).tolist()) != LABELS:
    raise ValueError("Label train tidak sesuai.")

sample_ids = sample["file_id"].astype(str).to_numpy()
n_test = len(sample_ids)

if n_test != TEST_SIZE:
    raise ValueError(f"Test size berubah: {n_test}, ekspektasi {TEST_SIZE}")

# ------------------------------------------------
# 2. OOF COMPONENTS
# ------------------------------------------------
O = {}
for name, filename in oof_paths.items():
    O[name] = normalize_probs(np.load(get_path(filename)), name)
    if O[name].shape != (len(meta), 3):
        raise ValueError(f"Shape OOF {name}: {O[name].shape}")

P_oof = normalize_probs(
    0.55 * O["ResNet50"]
    + 0.05 * O["EfficientNet-B2"]
    + 0.40 * O["ConvNeXt-v4"],
    "OOF ensemble"
)

y_true = np.array([LABELS.index(v) for v in y_text])
y_pred = P_oof.argmax(axis=1)
correct = (y_true == y_pred)
confidence_oof = P_oof.max(axis=1)

disagree_oof = (
    tv(O["ResNet50"], O["EfficientNet-B2"])
    + tv(O["ResNet50"], O["ConvNeXt-v4"])
    + tv(O["EfficientNet-B2"], O["ConvNeXt-v4"])
) / 3.0

n_reject_oof = int(np.ceil(REJECT_RATE * len(meta)))

print("=" * 72)
print("OOF BASELINE")
print("=" * 72)
print("Train rows:", len(meta))
print("Jumlah sampel yang ditolak untuk simulasi:", n_reject_oof)
print("Accuracy:", round(accuracy_score(y_true, y_pred), 5))
print(
    "Macro F1:",
    round(f1_score(y_true, y_pred, labels=np.arange(3),
                   average="macro", zero_division=0), 5)
)
print("Total OOF errors:", int((~correct).sum()))

# ------------------------------------------------
# 3. SIMULASI PENOLAKAN PADA OOF
# ------------------------------------------------
oof_rows = []
class_rows = []
oof_masks = {}

overall_error = float((~correct).mean())
all_weights = []

for wc, wd in WEIGHT_GRID:
    name = f"C{int(wc*100):02d}_D{int(wd*100):02d}"
    score = composite_score(confidence_oof, disagree_oof, wc, wd)
    mask = select_lowest(score, n_reject_oof)
    oof_masks[name] = mask

    n_errors_rejected = int((~correct[mask]).sum())
    n_errors_retained = int((~correct[~mask]).sum())
    error_rate_rejected = float((~correct[mask]).mean())
    error_rate_retained = float((~correct[~mask]).mean())

    oof_rows.append({
        "weights": name,
        "confidence_weight": wc,
        "disagreement_weight": wd,
        "rejected": int(mask.sum()),
        "errors_rejected": n_errors_rejected,
        "fraction_of_all_errors_captured": (
            n_errors_rejected / max(int((~correct).sum()), 1)
        ),
        "rejected_error_rate": error_rate_rejected,
        "retained_errors": n_errors_retained,
        "retained_error_rate": error_rate_retained,
        "error_enrichment": (
            error_rate_rejected / overall_error
            if overall_error > 0 else np.nan
        ),
        "retained_macro_f1": f1_score(
            y_true[~mask], y_pred[~mask],
            labels=np.arange(3), average="macro", zero_division=0
        ),
    })

    for i, cls in enumerate(LABELS):
        cls_mask = (y_true == i)
        class_rows.append({
            "weights": name,
            "class": cls,
            "support": int(cls_mask.sum()),
            "rejected": int((mask & cls_mask).sum()),
            "rejected_rate": float(
                (mask & cls_mask).sum() / max(cls_mask.sum(), 1)
            ),
            "rejected_correct": int(
                (mask & cls_mask & correct).sum()
            ),
            "rejected_wrong": int(
                (mask & cls_mask & ~correct).sum()
            ),
        })

    all_weights.append(name)

oof_result = pd.DataFrame(oof_rows)
class_result = pd.DataFrame(class_rows)

print("\n" + "=" * 72)
print("OOF REJECTION COMPARISON")
print("=" * 72)
print(oof_result.round(4).to_string(index=False))

print("\n" + "=" * 72)
print("OOF CLASS REJECTION RATES")
print("=" * 72)
print(
    class_result.pivot(
        index="class",
        columns="weights",
        values="rejected_rate"
    ).round(3).to_string()
)

# ------------------------------------------------
# 4. TEST COMPONENTS
# ------------------------------------------------
P_saved = normalize_probs(
    np.load(get_path(
        "test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy"
    )),
    "Saved test ensemble"
)

with np.load(get_path(test_paths["ResNetArchive"])) as z:
    if not {"test_r5", "test_r8"}.issubset(z.files):
        raise ValueError("ResNet archive tidak mempunyai test_r5/test_r8")
    r5 = np.asarray(z["test_r5"], dtype=np.float64)
    r8 = np.asarray(z["test_r8"], dtype=np.float64)

if (
    r5.ndim != 3 or r8.ndim != 3
    or r5.shape[0] != n_test or r8.shape[0] != n_test
    or r5.shape[1] < 5 or r8.shape[1] < 5
    or r5.shape[2] != 3 or r8.shape[2] != 3
):
    raise ValueError(f"Shape ResNet test salah: {r5.shape}, {r8.shape}")

crops = [0, 2, 4]
p5 = normalize_probs(r5[:, crops, :].mean(axis=1), "ResNet 5s")
p8 = normalize_probs(r8[:, crops, :].mean(axis=1), "ResNet 8s")
T = {
    "ResNet50": normalize_probs(0.25 * p5 + 0.75 * p8, "ResNet test"),
    "EfficientNet-B2": normalize_probs(
        np.load(get_path(test_paths["B2"])), "B2 test"
    ),
    "ConvNeXt-v4": normalize_probs(
        np.load(get_path(test_paths["ConvNeXt"])), "ConvNeXt test"
    ),
}

P_rebuilt = normalize_probs(
    0.55 * T["ResNet50"]
    + 0.05 * T["EfficientNet-B2"]
    + 0.40 * T["ConvNeXt-v4"],
    "Rebuilt test ensemble"
)

maxdiff = float(np.max(np.abs(P_saved - P_rebuilt)))
print("\nMax difference saved/rebuilt test ensemble:", maxdiff)

if maxdiff > 0.005:
    raise ValueError("Test component probabilities tidak konsisten.")

confidence_test = P_saved.max(axis=1)
pred_test = np.asarray(LABELS)[P_saved.argmax(axis=1)]

disagree_test = (
    tv(T["ResNet50"], T["EfficientNet-B2"])
    + tv(T["ResNet50"], T["ConvNeXt-v4"])
    + tv(T["EfficientNet-B2"], T["ConvNeXt-v4"])
) / 3.0

# Baseline score 0.64871 berasal dari 50:50 ranking.
score_5050 = composite_score(
    confidence_test, disagree_test, 0.5, 0.5
)
best_mask = select_lowest(score_5050, N_UNKNOWN)

# ------------------------------------------------
# 5. EVALUATE TEST VARIANTS; SAVE ALTERNATIVES
# ------------------------------------------------
test_rows = []
candidate_paths = []

for wc, wd in WEIGHT_GRID:
    name = f"C{int(wc*100):02d}_D{int(wd*100):02d}"
    score = composite_score(confidence_test, disagree_test, wc, wd)
    mask = select_lowest(score, N_UNKNOWN)

    test_rows.append({
        "weights": name,
        "confidence_weight": wc,
        "disagreement_weight": wd,
        "unknown_count": int(mask.sum()),
        "overlap_with_best_50_50": int((mask & best_mask).sum()),
        "changed_unknown_status_vs_best": int(
            2 * (N_UNKNOWN - (mask & best_mask).sum())
        ),
        "unknown_from_bondol": int((mask & (pred_test == LABELS[0])).sum()),
        "unknown_from_cendrawasih": int((mask & (pred_test == LABELS[1])).sum()),
        "unknown_from_kutilang": int((mask & (pred_test == LABELS[2])).sum()),
    })

    # Jangan overwrite 50:50, karena itu metode dengan public score 0.64871.
    if (wc, wd) != (0.5, 0.5):
        labels = pred_test.astype(object)
        labels[mask] = "Unknown"

        out_path = os.path.join(
            WORK,
            f"offline_candidate_triple_v4_{name.lower()}_u245.csv"
        )
        pd.DataFrame({
            "file_id": sample_ids,
            "label": labels
        }).to_csv(out_path, index=False)
        candidate_paths.append(out_path)

test_result = pd.DataFrame(test_rows)

print("\n" + "=" * 72)
print("TEST UNKNOWN SELECTION SENSITIVITY")
print("=" * 72)
print(test_result.to_string(index=False))

print("\n" + "=" * 72)
print("ALTERNATIVE CSV FILES — NOT SUBMITTED")
print("=" * 72)
for path in candidate_paths:
    print(path)

# ------------------------------------------------
# 6. SAVE DIAGNOSTICS
# ------------------------------------------------
oof_result.to_csv(
    os.path.join(WORK, "offline_oof_weight_comparison.csv"),
    index=False
)
class_result.to_csv(
    os.path.join(WORK, "offline_oof_class_rejection_rates.csv"),
    index=False
)
test_result.to_csv(
    os.path.join(WORK, "offline_test_weight_sensitivity.csv"),
    index=False
)

print("\nDiagnostik tersimpan:")
print(os.path.join(WORK, "offline_oof_weight_comparison.csv"))
print(os.path.join(WORK, "offline_oof_class_rejection_rates.csv"))
print(os.path.join(WORK, "offline_test_weight_sensitivity.csv"))

print("\nSelesai. Tidak ada submission baru yang dikirim.")
print("Public score 0.64871 tetap merupakan hasil eksperimen sebelumnya.")

OOF BASELINE
Train rows: 1826
Jumlah sampel yang ditolak untuk simulasi: 550
Accuracy: 0.98412
Macro F1: 0.94204
Total OOF errors: 29

OOF REJECTION COMPARISON
 weights  confidence_weight  disagreement_weight  rejected  errors_rejected  fraction_of_all_errors_captured  rejected_error_rate  retained_errors  retained_error_rate  error_enrichment  retained_macro_f1
C100_D00                1.0                  0.0       550               29                           1.0000               0.0527                0               0.0000            3.3200             1.0000
 C70_D30                0.7                  0.3       550               28                           0.9655               0.0509                1               0.0008            3.2055             0.9932
 C60_D40                0.6                  0.4       550               28                           0.9655               0.0509                1               0.0008            3.2055             0.9932
 C50_D50            

In [7]:
# ================================================================
# NEXT CANDIDATE: C70_D30 | EXACTLY U245
# Current best public score: 0.65057 (C60_D40)
# ================================================================

import os
import numpy as np
import pandas as pd

ROOT = "/kaggle/input/competitions/schematics-2026"
WORK = "/kaggle/working"
SRC = "/kaggle/input/notebooks/radithiafatir/testttrisan"

LABELS = ["Bondol Jawa", "Cendrawasih", "Kutilang"]
N_UNKNOWN = 245

FILES = {
    "ensemble": "test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy",
    "resnet": "tta_test_crop_probabilities.npz",
    "b2": "test_effnet_b2_audio8s_tta3.npy",
    "convnext": "test_convnext_tiny_pretrained_v4_resize224_tta3.npy",
}

def find_file(name):
    for base in [WORK, SRC]:
        path = os.path.join(base, name)
        if os.path.isfile(path):
            return path
    raise FileNotFoundError(f"File tidak ditemukan: {name}")

def probs(P, name):
    P = np.asarray(P, dtype=np.float64)
    if P.ndim != 2 or not np.isfinite(P).all():
        raise ValueError(f"{name}: probabilitas tidak valid")
    if (P < -1e-5).any() or (P > 1.00001).any():
        raise ValueError(f"{name}: nilai di luar rentang 0-1")
    P = np.clip(P, 1e-12, None)
    return P / P.sum(axis=1, keepdims=True)

def rank01(x):
    return pd.Series(np.asarray(x)).rank(
        method="average", pct=True
    ).to_numpy(dtype=np.float64)

# 1. Load data
sample = pd.read_csv(os.path.join(ROOT, "sample_submission.csv"))
sample["file_id"] = sample["file_id"].astype(str)

if sample["file_id"].duplicated().any():
    raise ValueError("file_id duplikat pada sample_submission")

n = len(sample)
P = probs(np.load(find_file(FILES["ensemble"])), "ensemble")

if P.shape != (n, 3):
    raise ValueError(f"Shape ensemble salah: {P.shape}")

# 2. Load probabilitas tiga model untuk disagreement
with np.load(find_file(FILES["resnet"])) as z:
    r5, r8 = z["test_r5"], z["test_r8"]

if (
    r5.ndim != 3 or r8.ndim != 3
    or r5.shape[0] != n or r8.shape[0] != n
    or r5.shape[1] < 5 or r8.shape[1] < 5
    or r5.shape[2] != 3 or r8.shape[2] != 3
):
    raise ValueError("Shape probabilitas ResNet tidak sesuai")

crops = [0, 2, 4]
p5 = probs(r5[:, crops, :].mean(axis=1), "ResNet 5s")
p8 = probs(r8[:, crops, :].mean(axis=1), "ResNet 8s")
P_r = probs(0.25 * p5 + 0.75 * p8, "ResNet gabungan")

P_b = probs(np.load(find_file(FILES["b2"])), "EfficientNet-B2")
P_c = probs(np.load(find_file(FILES["convnext"])), "ConvNeXt")

# Validasi konsistensi probabilitas ensemble
P_rebuilt = probs(
    0.55 * P_r + 0.05 * P_b + 0.40 * P_c,
    "ensemble rebuilt"
)

max_diff = float(np.max(np.abs(P - P_rebuilt)))
print("Max difference ensemble:", max_diff)

if max_diff > 0.005:
    raise ValueError("Probabilitas komponen tidak cocok dengan ensemble.")

# 3. Hitung confidence dan disagreement
confidence = P.max(axis=1)

def tv(A, B):
    return 0.5 * np.abs(A - B).sum(axis=1)

disagreement = (
    tv(P_r, P_b) + tv(P_r, P_c) + tv(P_b, P_c)
) / 3.0

# Confidence 70%; disagreement 30%
# Confidence rendah -> rank rendah.
# Disagreement tinggi -> rank01(-disagreement) rendah.
score = (
    0.70 * rank01(confidence)
    + 0.30 * rank01(-disagreement)
)

if not np.isfinite(score).all():
    raise ValueError("Skor ranking mengandung NaN/inf.")

# 4. Pilih tepat 245 Unknown
selected = np.argsort(score, kind="stable")[:N_UNKNOWN]

mask = np.zeros(n, dtype=bool)
mask[selected] = True

pred = np.asarray(LABELS)[P.argmax(axis=1)]
final_labels = pred.astype(object)
final_labels[mask] = "Unknown"

if int(np.sum(final_labels == "Unknown")) != N_UNKNOWN:
    raise AssertionError("Jumlah Unknown tidak tepat 245")

# 5. Simpan kandidat
out_path = os.path.join(
    WORK, "offline_candidate_triple_v4_c70_d30_u245.csv"
)

submission = pd.DataFrame({
    "file_id": sample["file_id"],
    "label": final_labels,
})
submission.to_csv(out_path, index=False)

# 6. Diagnostik
print("\nDistribusi label:")
print(submission["label"].value_counts().to_string())

print("\nKelas prediksi awal yang dipilih menjadi Unknown:")
print(pd.Series(pred[mask]).value_counts().to_string())

print("\nFile kandidat:", out_path)
print("Jumlah baris:", len(submission))
print("Unknown:", int((submission["label"] == "Unknown").sum()))
print("\nBelum disubmit. Public score kandidat belum diketahui.")

Max difference ensemble: 4.931897412685515e-08

Distribusi label:
label
Unknown        245
Kutilang       236
Bondol Jawa    216
Cendrawasih    117

Kelas prediksi awal yang dipilih menjadi Unknown:
Bondol Jawa    105
Kutilang        75
Cendrawasih     65

File kandidat: /kaggle/working/offline_candidate_triple_v4_c70_d30_u245.csv
Jumlah baris: 814
Unknown: 245

Belum disubmit. Public score kandidat belum diketahui.


In [8]:
# =============================================================================
# SCHEMATICS NPC DSC 2026 — EfficientNet-B3 audio 8s, 4-fold Group CV
#
# Purpose:
#   1) Train EfficientNet-B3 from scratch on the competition's train audio only.
#   2) Reuse the existing 8-second log-mel cache if it is available.
#   3) Use the same seed=16, four group folds, mel preprocessing and TTA3 style
#      as the earlier EfficientNet-B2 experiment.
#   4) Save OOF and test probabilities for later comparison with Triple V4.
#   5) Search four-model ensemble weights on OOF only; create no submission.
#
# Inputs expected in the attached notebook output (or /kaggle/working):
#   oof_resnet50_ensemble_tta3_seed16.npy
#   oof_efficientnet_b2_audio8s_tta3_seed16.npy
#   oof_convnext_tiny_audio8s_tta3_seed16_pretrained_v4_resize224.npy
#   tta_test_crop_probabilities.npz
#   test_effnet_b2_audio8s_tta3.npy
#   test_convnext_tiny_pretrained_v4_resize224_tta3.npy
#
# This script does not submit to Kaggle and does not overwrite the best CSV.
# =============================================================================

import os
import glob
import json
import math
import random
import warnings
from contextlib import nullcontext
from itertools import product

import numpy as np
import pandas as pd
import librosa

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b3

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report

warnings.filterwarnings("ignore", category=UserWarning)

# =============================================================================
# 1. CONFIGURATION
# =============================================================================
SEED = 16
N_FOLDS = 4
SR = 32000
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
FMIN = 100
FMAX = 14000
DURATION = 8.0
EPOCHS = 12
BATCH_SIZE = 4           # Safer starting point for EfficientNet-B3 on Tesla T4.
VAL_BATCH_SIZE = 4
NUM_WORKERS = 2
LR = 3e-4
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 4
USE_AMP_REQUESTED = True

ROOT = "/kaggle/input/competitions/schematics-2026"
OUT = "/kaggle/working"
SOURCE_OUTPUT = "/kaggle/input/notebooks/radithiafatir/testttrisan"

TRAIN_META_PATH = os.path.join(ROOT, "train", "metadata.csv")
SAMPLE_PATH = os.path.join(ROOT, "sample_submission.csv")
TRAIN_CACHE_DIR = os.path.join(OUT, "bird_mel_cache_effnet_b3")
TEST_CACHE_DIR = os.path.join(OUT, "bird_mel_cache_test_effnet_b3")
MODEL_DIR = os.path.join(OUT, "effnet_b3_audio8s_group4_v1")

OOF_B3_PATH = os.path.join(OUT, "oof_efficientnet_b3_audio8s_tta3_seed16.npy")
TEST_B3_PATH = os.path.join(OUT, "test_effnet_b3_audio8s_tta3.npy")
OOF_SEARCH_PATH = os.path.join(OUT, "effnet_b3_quad_ensemble_oof_weight_search.csv")
OOF_METRICS_PATH = os.path.join(OUT, "effnet_b3_oof_metrics.json")
BEST_BLEND_TEST_PATH = os.path.join(OUT, "test_resnet_b2_convnext_b3_best_oof_blend_probabilities.npy")

TRAIN_CACHE_READ_DIRS = [
    TRAIN_CACHE_DIR,
    os.path.join(OUT, "bird_mel_cache_effnet"),
    os.path.join(SOURCE_OUTPUT, "bird_mel_cache_effnet"),
]
TEST_CACHE_READ_DIRS = [
    TEST_CACHE_DIR,
    os.path.join(OUT, "bird_mel_cache_test_convnext_v4"),
    os.path.join(OUT, "bird_mel_cache_test_effnet"),
    os.path.join(SOURCE_OUTPUT, "bird_mel_cache_test_convnext_v4"),
    os.path.join(SOURCE_OUTPUT, "bird_mel_cache_test_effnet"),
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = bool(USE_AMP_REQUESTED and DEVICE.type == "cuda")
LABELS_EXPECTED = ["Bondol Jawa", "Cendrawasih", "Kutilang"]

os.makedirs(OUT, exist_ok=True)
os.makedirs(TRAIN_CACHE_DIR, exist_ok=True)
os.makedirs(TEST_CACHE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# =============================================================================
# 2. REPRODUCIBILITY AND HELPERS
# =============================================================================
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def autocast_context():
    if USE_AMP:
        return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True)
    return nullcontext()


def make_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


def find_artifact(filename, required=True):
    for base in [OUT, SOURCE_OUTPUT]:
        candidate = os.path.join(base, filename)
        if os.path.isfile(candidate):
            return candidate
    if required:
        raise FileNotFoundError(
            f"Artefak tidak ditemukan: {filename}\n"
            f"Sudah dicari di {OUT} dan {SOURCE_OUTPUT}. "
            "Pastikan output notebook testttrisan telah ditambahkan sebagai input."
        )
    return None


def base_id(file_id):
    return os.path.splitext(os.path.basename(str(file_id)))[0]


def find_cached_mel(file_id, is_test=False):
    name = base_id(file_id) + ".npy"
    dirs = TEST_CACHE_READ_DIRS if is_test else TRAIN_CACHE_READ_DIRS
    for directory in dirs:
        path = os.path.join(directory, name)
        if os.path.isfile(path):
            return path
    return None


def resolve_train_audio(row):
    raw = str(row["audio_path"])
    file_id = str(row["file_id"])
    raw_base = os.path.basename(raw)
    file_base = base_id(file_id)
    audio_dir = os.path.join(ROOT, "train", "audio")

    candidates = []
    if os.path.isabs(raw):
        candidates.append(raw)
    else:
        candidates.extend([
            os.path.join(ROOT, raw),
            os.path.join(ROOT, "train", raw),
        ])

    candidates.extend([
        os.path.join(audio_dir, file_id),
        os.path.join(audio_dir, raw_base),
    ])
    for ext in [".flac", ".wav", ".mp3", ".ogg", ".m4a"]:
        candidates.append(os.path.join(audio_dir, file_base + ext))
        candidates.append(os.path.join(audio_dir, base_id(raw_base) + ext))

    for path in candidates:
        if os.path.isfile(path):
            return path

    matches = glob.glob(os.path.join(audio_dir, file_base + ".*"))
    if matches:
        return matches[0]
    matches = glob.glob(os.path.join(audio_dir, base_id(raw_base) + ".*"))
    if matches:
        return matches[0]

    raise FileNotFoundError(
        f"Audio train tidak ditemukan untuk file_id={file_id}; audio_path={raw}"
    )


def resolve_test_audio(file_id):
    file_id = str(file_id)
    file_base = base_id(file_id)
    audio_dir = os.path.join(ROOT, "test", "audio")
    candidates = [os.path.join(audio_dir, file_id)]
    for ext in [".flac", ".wav", ".mp3", ".ogg", ".m4a"]:
        candidates.append(os.path.join(audio_dir, file_base + ext))
    for path in candidates:
        if os.path.isfile(path):
            return path
    matches = glob.glob(os.path.join(audio_dir, file_base + ".*"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Audio test tidak ditemukan untuk file_id={file_id}")


def compute_mel(audio_path):
    audio, _ = librosa.load(audio_path, sr=SR, mono=True)
    if audio.size == 0:
        raise ValueError(f"Audio kosong: {audio_path}")
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=2.0,
    )
    db = librosa.power_to_db(mel, ref=np.max, top_db=80.0)
    # Keep the exact normalization used by the previous B2 pipeline.
    return ((db + 40.0) / 20.0).astype(np.float16)


def build_cache_if_missing(frame, is_test=False):
    cache_dir = TEST_CACHE_DIR if is_test else TRAIN_CACHE_DIR
    os.makedirs(cache_dir, exist_ok=True)
    newly_computed = 0
    total = len(frame)

    for i, (_, row) in enumerate(frame.iterrows(), start=1):
        file_id = str(row["file_id"])
        existing = find_cached_mel(file_id, is_test=is_test)
        if existing is None:
            target = os.path.join(cache_dir, base_id(file_id) + ".npy")
            audio_path = (
                resolve_test_audio(file_id) if is_test else resolve_train_audio(row)
            )
            np.save(target, compute_mel(audio_path))
            newly_computed += 1
        else:
            # Validate cache shape; regenerate only if the cached file is bad.
            try:
                arr = np.load(existing, mmap_mode="r")
                valid = arr.ndim == 2 and arr.shape[0] == N_MELS and arr.shape[1] > 0
            except Exception:
                valid = False
            if not valid:
                target = os.path.join(cache_dir, base_id(file_id) + ".npy")
                audio_path = (
                    resolve_test_audio(file_id) if is_test else resolve_train_audio(row)
                )
                np.save(target, compute_mel(audio_path))
                newly_computed += 1

        if i % 200 == 0 or i == total:
            split = "TEST" if is_test else "TRAIN"
            print(f"  {split} mel cache: {i}/{total}; newly computed={newly_computed}")

    # Record exact path for each ID after cache creation/reuse.
    mapping = {}
    for file_id in frame["file_id"].astype(str):
        path = find_cached_mel(file_id, is_test=is_test)
        if path is None:
            raise FileNotFoundError(f"Cache mel belum ada untuk {file_id}")
        mapping[file_id] = path
    return mapping


def frames_for(seconds):
    return int(math.ceil(SR * seconds / HOP_LENGTH))

TARGET_FRAMES = frames_for(DURATION)


def crop_one(spec, target_frames, mode="center"):
    t = spec.shape[1]
    if t <= target_frames:
        return np.pad(
            spec, ((0, 0), (0, target_frames - t)), mode="constant"
        ).astype(np.float32)
    max_start = t - target_frames
    if mode == "random":
        start = random.randint(0, max_start)
    elif mode == "start":
        start = 0
    elif mode == "end":
        start = max_start
    else:
        start = max_start // 2
    return spec[:, start:start + target_frames].astype(np.float32)


def make_three_crops(spec):
    t = spec.shape[1]
    if t <= TARGET_FRAMES:
        padded = np.pad(
            spec, ((0, 0), (0, TARGET_FRAMES - t)), mode="constant"
        )
        return np.stack([padded, padded, padded]).astype(np.float32)
    d = t - TARGET_FRAMES
    starts = [0, d // 2, d]
    return np.stack([
        spec[:, start:start + TARGET_FRAMES] for start in starts
    ]).astype(np.float32)


def spec_augment(x):
    x = x.copy()
    if random.random() < 0.5:
        width = random.randint(1, min(12, x.shape[0]))
        f0 = random.randint(0, x.shape[0] - width)
        x[f0:f0 + width, :] = 0.0
    if random.random() < 0.5:
        width = random.randint(1, min(45, x.shape[1]))
        t0 = random.randint(0, x.shape[1] - width)
        x[:, t0:t0 + width] = 0.0
    return x

# =============================================================================
# 3. DATASETS
# =============================================================================
class TrainBirdMelDataset(Dataset):
    def __init__(self, frame, cache_map, mode="train"):
        self.frame = frame.reset_index(drop=True).copy()
        self.cache_map = cache_map
        self.mode = mode

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        spec = np.load(self.cache_map[str(row["file_id"])]).astype(np.float32)
        global_idx = int(row["row_idx"])
        label = int(row["label_idx"])

        if self.mode == "crops":
            crops = make_three_crops(spec)
            x = torch.from_numpy(np.ascontiguousarray(crops[:, None, :, :]))
            return x, torch.tensor(label, dtype=torch.long), global_idx

        if self.mode == "train":
            x = crop_one(spec, TARGET_FRAMES, mode="random")
            x = spec_augment(x)
        else:
            x = crop_one(spec, TARGET_FRAMES, mode="center")

        x = torch.from_numpy(np.ascontiguousarray(x[None, :, :]))
        return x, torch.tensor(label, dtype=torch.long), global_idx


class TestBirdMelDataset(Dataset):
    def __init__(self, ids, cache_map):
        self.ids = list(ids)
        self.cache_map = cache_map

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        file_id = str(self.ids[index])
        spec = np.load(self.cache_map[file_id]).astype(np.float32)
        crops = make_three_crops(spec)
        x = torch.from_numpy(np.ascontiguousarray(crops[:, None, :, :]))
        return x, index


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# =============================================================================
# 4. MODEL / PREDICTION HELPERS
# =============================================================================
def build_effnet_b3_audio(num_classes):
    # Train from scratch, no pretrained external weights.
    model = efficientnet_b3(weights=None)
    old_conv = model.features[0][0]
    model.features[0][0] = nn.Conv2d(
        in_channels=1,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        dilation=old_conv.dilation,
        groups=old_conv.groups,
        bias=(old_conv.bias is not None),
        padding_mode=old_conv.padding_mode,
    )
    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features, num_classes
    )
    return model


def load_weights(model, path):
    try:
        state = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        state = torch.load(path, map_location="cpu")
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model.load_state_dict(state, strict=True)
    return model


def evaluate_center(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y, _ in loader:
            x = x.to(DEVICE, non_blocking=True)
            with autocast_context():
                logits = model(x)
            y_true.extend(y.numpy().tolist())
            y_pred.extend(logits.argmax(1).cpu().numpy().tolist())
    return f1_score(
        y_true, y_pred, labels=list(range(NUM_CLASSES)),
        average="macro", zero_division=0
    )


def predict_three_crop(model, loader, n_rows):
    model.eval()
    output = np.zeros((n_rows, NUM_CLASSES), dtype=np.float32)
    test_mode = isinstance(loader.dataset, TestBirdMelDataset)
    with torch.no_grad():
        for batch in loader:
            # Test loader yields (crops, local_index); train crops loader yields
            # (crops, label, global_index).
            if test_mode:
                crops, indices = batch
            else:
                crops, _, indices = batch
            if crops.ndim != 5:
                raise ValueError(f"Expected (B,3,1,H,W), got {tuple(crops.shape)}")
            idx = indices.numpy()
            b, nc, ch, h, w = crops.shape
            x = crops.reshape(b * nc, ch, h, w).to(DEVICE, non_blocking=True)
            with autocast_context():
                logits = model(x)
            probs = torch.softmax(logits.float(), dim=1).reshape(
                b, nc, NUM_CLASSES
            ).cpu().numpy().mean(axis=1)
            output[idx] = probs
    return output

# =============================================================================
# 5. LOAD METADATA AND THE EXACT EXISTING FOLDS
# =============================================================================
seed_everything(SEED)

if not os.path.isfile(TRAIN_META_PATH) or not os.path.isfile(SAMPLE_PATH):
    raise FileNotFoundError(
        "Dataset kompetisi tidak ditemukan. Tambahkan competition dataset sebagai input."
    )

meta = pd.read_csv(TRAIN_META_PATH).copy()
sample = pd.read_csv(SAMPLE_PATH).copy()
required_cols = {"file_id", "audio_path", "label", "source_group"}
missing_cols = required_cols - set(meta.columns)
if missing_cols:
    raise ValueError(f"Kolom metadata train hilang: {missing_cols}")
if "file_id" not in sample.columns:
    raise ValueError("sample_submission.csv tidak memiliki file_id")

meta["file_id"] = meta["file_id"].astype(str)
meta["label"] = meta["label"].astype(str)
meta["source_group"] = meta["source_group"].astype(str)
sample["file_id"] = sample["file_id"].astype(str)

LABELS = sorted(meta["label"].unique().tolist())
if LABELS != LABELS_EXPECTED:
    raise ValueError(f"Urutan kelas tidak sesuai: {LABELS}")
NUM_CLASSES = len(LABELS)
LABEL_TO_IDX = {label: i for i, label in enumerate(LABELS)}
meta["label_idx"] = meta["label"].map(LABEL_TO_IDX).astype(int)
meta["row_idx"] = np.arange(len(meta), dtype=int)

# Prefer stored folds from the previous B2/ResNet experiment so OOF arrays align.
fold_file = find_artifact("bird_group_folds_seed16.csv", required=False)
if fold_file:
    saved_folds = pd.read_csv(fold_file)
    if not {"file_id", "fold"}.issubset(saved_folds.columns):
        raise ValueError("bird_group_folds_seed16.csv tidak memiliki file_id/fold")
    saved_folds["file_id"] = saved_folds["file_id"].astype(str)
    if saved_folds["file_id"].duplicated().any():
        raise ValueError("file_id duplikat pada saved folds")
    fold_map = saved_folds.set_index("file_id")["fold"]
    mapped = meta["file_id"].map(fold_map)
    if mapped.isna().any() or len(saved_folds) != len(meta):
        raise ValueError("Fold lama tidak cocok persis dengan metadata train")
    meta["fold"] = mapped.astype(int)
    print("Using saved group folds:", fold_file)
else:
    print("Saved fold file not found; recreating StratifiedGroupKFold seed=16.")
    sgkf = StratifiedGroupKFold(
        n_splits=N_FOLDS, shuffle=True, random_state=SEED
    )
    meta["fold"] = -1
    for fold, (_, val_idx) in enumerate(
        sgkf.split(meta, meta["label_idx"], groups=meta["source_group"])
    ):
        meta.loc[val_idx, "fold"] = fold
    if (meta["fold"] < 0).any():
        raise RuntimeError("Ada data tidak memiliki fold")

if sorted(meta["fold"].unique().tolist()) != list(range(N_FOLDS)):
    raise ValueError("Fold tidak lengkap; perlu fold 0,1,2,3")

fold_out = os.path.join(OUT, "effnet_b3_group_folds_seed16.csv")
meta[["file_id", "label", "source_group", "row_idx", "fold"]].to_csv(fold_out, index=False)

print("=" * 78)
print("SCHEMATICS NPC DSC 2026 — EfficientNet-B3 8s GroupCV")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("AMP:", USE_AMP)
print("Train rows:", len(meta))
print("Test rows:", len(sample))
print("Labels:", LABELS)
print("Fold counts:", meta["fold"].value_counts().sort_index().to_dict())
print("Architecture: torchvision efficientnet_b3(weights=None), 1-channel input")
print("Checkpoint directory:", MODEL_DIR)
print("=" * 78)

if sample["file_id"].duplicated().any():
    raise ValueError("sample_submission memiliki file_id duplikat")

# =============================================================================
# 6. REUSE / BUILD MEL CACHE
# =============================================================================
print("\nPreparing/reusing train mel cache...")
train_cache_map = build_cache_if_missing(meta, is_test=False)
print("Train mel paths ready:", len(train_cache_map))

print("\nPreparing/reusing test mel cache...")
test_frame = pd.DataFrame({"file_id": sample["file_id"].astype(str)})
test_cache_map = build_cache_if_missing(test_frame, is_test=True)
print("Test mel paths ready:", len(test_cache_map))

# =============================================================================
# 7. TRAIN/LOAD FOUR FOLDS; MAKE OOF + TEST PREDICTIONS
# =============================================================================
oof_b3 = np.zeros((len(meta), NUM_CLASSES), dtype=np.float32)
test_sum = np.zeros((len(sample), NUM_CLASSES), dtype=np.float32)
fold_metrics = []

test_ds = TestBirdMelDataset(sample["file_id"].tolist(), test_cache_map)
test_loader = DataLoader(
    test_ds,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

for fold in range(N_FOLDS):
    print("\n" + "-" * 70)
    print(f"EfficientNet-B3 fold {fold + 1}/{N_FOLDS}")
    print("-" * 70)
    seed_everything(SEED + fold)

    train_frame = meta.loc[meta["fold"] != fold].copy()
    val_frame = meta.loc[meta["fold"] == fold].copy()

    ckpt_path = os.path.join(MODEL_DIR, f"efficientnet_b3_audio8s_fold{fold}.pth")
    done_path = os.path.join(MODEL_DIR, f"efficientnet_b3_audio8s_fold{fold}.done.json")

    train_ds = TrainBirdMelDataset(train_frame, train_cache_map, mode="train")
    val_center_ds = TrainBirdMelDataset(val_frame, train_cache_map, mode="center")
    val_crops_ds = TrainBirdMelDataset(val_frame, train_cache_map, mode="crops")

    gen = torch.Generator()
    gen.manual_seed(SEED + fold)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        worker_init_fn=seed_worker,
        generator=gen,
        persistent_workers=(NUM_WORKERS > 0),
        drop_last=False,
    )
    val_center_loader = DataLoader(
        val_center_ds,
        batch_size=VAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
    )
    val_crops_loader = DataLoader(
        val_crops_ds,
        batch_size=VAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
    )

    model = build_effnet_b3_audio(NUM_CLASSES).to(DEVICE)
    reuse_checkpoint = os.path.isfile(ckpt_path) and os.path.isfile(done_path)

    if reuse_checkpoint:
        print("Reusing completed fold checkpoint:", ckpt_path)
        load_weights(model, ckpt_path)
        with open(done_path, "r", encoding="utf-8") as f:
            done_info = json.load(f)
        best_f1 = float(done_info.get("best_f1_center", np.nan))
        best_epoch = int(done_info.get("best_epoch", -1))
    else:
        counts = np.bincount(
            train_frame["label_idx"].to_numpy(), minlength=NUM_CLASSES
        ).astype(np.float32)
        weights = 1.0 / np.sqrt(np.maximum(counts, 1.0))
        weights = weights / weights.mean()
        class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
        criterion = nn.CrossEntropyLoss(
            weight=class_weights, label_smoothing=0.05
        )
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-6
        )
        scaler = make_grad_scaler()
        best_f1 = -1.0
        best_epoch = 0
        epochs_without_improve = 0

        for epoch in range(EPOCHS):
            model.train()
            total_loss = 0.0
            seen = 0

            for x, y, _ in train_loader:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)

                with autocast_context():
                    logits = model(x)
                    loss = criterion(logits, y)

                if USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

                total_loss += float(loss.item()) * len(y)
                seen += len(y)

            val_f1 = evaluate_center(model, val_center_loader)
            scheduler.step(val_f1)
            mean_loss = total_loss / max(seen, 1)
            current_lr = optimizer.param_groups[0]["lr"]
            print(
                f"Epoch {epoch + 1:02d}/{EPOCHS} | loss={mean_loss:.4f} | "
                f"val_center_macroF1={val_f1:.5f} | lr={current_lr:.2e}"
            )

            if val_f1 > best_f1 + 1e-5:
                best_f1 = float(val_f1)
                best_epoch = epoch + 1
                epochs_without_improve = 0
                torch.save(model.state_dict(), ckpt_path)
                print(f"  -> saved best checkpoint (epoch {best_epoch}, F1={best_f1:.5f})")
            else:
                epochs_without_improve += 1

            if epochs_without_improve >= EARLY_STOPPING_PATIENCE:
                print("  Early stopping.")
                break

        if not os.path.isfile(ckpt_path):
            raise RuntimeError(f"Checkpoint tidak berhasil dibuat: {ckpt_path}")
        with open(done_path, "w", encoding="utf-8") as f:
            json.dump({
                "model": "efficientnet_b3",
                "seed": SEED,
                "fold": fold,
                "best_f1_center": best_f1,
                "best_epoch": best_epoch,
                "duration_seconds": DURATION,
                "pretrained": False,
                "labels": LABELS,
            }, f, indent=2)
        load_weights(model, ckpt_path)

    # OOF probabilities are written only to each fold's held-out rows.
    fold_oof_all = predict_three_crop(model, val_crops_loader, len(meta))
    val_indices = val_frame["row_idx"].to_numpy(dtype=int)
    oof_b3[val_indices] = fold_oof_all[val_indices]
    np.save(OOF_B3_PATH, oof_b3)

    # Test predictions are three-crop average for this fold. Average folds at end.
    fold_test = predict_three_crop(model, test_loader, len(sample))
    test_sum += fold_test / N_FOLDS

    fold_pred = oof_b3[val_indices].argmax(axis=1)
    fold_y = val_frame["label_idx"].to_numpy(dtype=int)
    fold_f1 = f1_score(
        fold_y, fold_pred, labels=list(range(NUM_CLASSES)),
        average="macro", zero_division=0
    )
    fold_metrics.append({
        "fold": fold,
        "n_train": len(train_frame),
        "n_valid": len(val_frame),
        "best_epoch": best_epoch,
        "best_center_macro_f1": best_f1,
        "valid_three_crop_macro_f1": float(fold_f1),
    })
    print(f"Fold {fold} three-crop OOF Macro F1: {fold_f1:.5f}")

    del model, train_loader, val_center_loader, val_crops_loader
    del train_ds, val_center_ds, val_crops_ds
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Validate populated OOF and test predictions.
if not np.all(oof_b3.sum(axis=1) > 0):
    raise RuntimeError("OOF B3 tidak lengkap: ada baris tanpa probabilitas")
if not np.all(test_sum.sum(axis=1) > 0):
    raise RuntimeError("Test B3 probability tidak lengkap")

oof_b3 = oof_b3 / np.clip(oof_b3.sum(axis=1, keepdims=True), 1e-12, None)
test_b3 = test_sum / np.clip(test_sum.sum(axis=1, keepdims=True), 1e-12, None)
np.save(OOF_B3_PATH, oof_b3.astype(np.float32))
np.save(TEST_B3_PATH, test_b3.astype(np.float32))
pd.DataFrame(fold_metrics).to_csv(
    os.path.join(OUT, "effnet_b3_fold_metrics.csv"), index=False
)

# =============================================================================
# 8. EVALUATE B3 AND SEARCH QUAD-ENSEMBLE WEIGHTS ON OOF
# =============================================================================
y_true = meta["label_idx"].to_numpy(dtype=int)
y_pred_b3 = oof_b3.argmax(axis=1)
b3_macro_f1 = f1_score(
    y_true, y_pred_b3, labels=list(range(NUM_CLASSES)),
    average="macro", zero_division=0
)
b3_acc = accuracy_score(y_true, y_pred_b3)

print("\n" + "=" * 78)
print("EFFICIENTNET-B3 OOF PERFORMANCE")
print("=" * 78)
print(f"Accuracy : {b3_acc:.5f}")
print(f"Macro F1 : {b3_macro_f1:.5f}")
print(classification_report(
    y_true, y_pred_b3, labels=list(range(NUM_CLASSES)),
    target_names=LABELS, digits=4, zero_division=0
))

# Load existing OOF predictions in the exact same row order.
P_R = np.asarray(np.load(find_artifact(
    "oof_resnet50_ensemble_tta3_seed16.npy"
)), dtype=np.float64)
P_B2 = np.asarray(np.load(find_artifact(
    "oof_efficientnet_b2_audio8s_tta3_seed16.npy"
)), dtype=np.float64)
P_C = np.asarray(np.load(find_artifact(
    "oof_convnext_tiny_audio8s_tta3_seed16_pretrained_v4_resize224.npy"
)), dtype=np.float64)

for name, arr in [("ResNet50", P_R), ("EfficientNet-B2", P_B2), ("ConvNeXt-v4", P_C)]:
    if arr.shape != (len(meta), NUM_CLASSES):
        raise ValueError(f"OOF {name} shape tidak sesuai: {arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"OOF {name} memiliki NaN/inf")
    arr /= np.clip(arr.sum(axis=1, keepdims=True), 1e-12, None)

P_R = P_R.astype(np.float64)
P_B2 = P_B2.astype(np.float64)
P_C = P_C.astype(np.float64)
P_B3 = oof_b3.astype(np.float64)

print("Individual OOF Macro F1 comparison:")
for name, arr in [
    ("ResNet50 5s+8s", P_R),
    ("EfficientNet-B2", P_B2),
    ("ConvNeXt-v4", P_C),
    ("EfficientNet-B3", P_B3),
]:
    pred = arr.argmax(axis=1)
    print(
        f"  {name:20s}: F1="
        f"{f1_score(y_true, pred, labels=list(range(NUM_CLASSES)), average='macro', zero_division=0):.5f}; "
        f"accuracy={accuracy_score(y_true, pred):.5f}"
    )

print("\nSearching 4-model ensemble weights at 0.05 increments...")
search_rows = []
units = 20  # weights in increments of 1/20 = 0.05
for i_r in range(units + 1):
    for i_b2 in range(units - i_r + 1):
        for i_c in range(units - i_r - i_b2 + 1):
            i_b3 = units - i_r - i_b2 - i_c
            wr, wb2, wc, wb3 = [v / units for v in (i_r, i_b2, i_c, i_b3)]
            blend = wr * P_R + wb2 * P_B2 + wc * P_C + wb3 * P_B3
            blend /= np.clip(blend.sum(axis=1, keepdims=True), 1e-12, None)
            pred = blend.argmax(axis=1)
            f1 = f1_score(
                y_true, pred, labels=list(range(NUM_CLASSES)),
                average="macro", zero_division=0
            )
            acc = accuracy_score(y_true, pred)
            search_rows.append({
                "resnet50_weight": wr,
                "efficientnet_b2_weight": wb2,
                "convnext_v4_weight": wc,
                "efficientnet_b3_weight": wb3,
                "oof_macro_f1": float(f1),
                "oof_accuracy": float(acc),
            })

weight_df = pd.DataFrame(search_rows).sort_values(
    ["oof_macro_f1", "oof_accuracy"], ascending=False
).reset_index(drop=True)
weight_df.to_csv(OOF_SEARCH_PATH, index=False)

print("\nTop 15 OOF blend configurations:")
print(weight_df.head(15).to_string(index=False, float_format=lambda x: f"{x:.5f}"))

# Explicitly report the original 3-model baseline and B3 replacing B2 at 5%.
def score_weights(wr, wb2, wc, wb3):
    blend = wr * P_R + wb2 * P_B2 + wc * P_C + wb3 * P_B3
    blend /= np.clip(blend.sum(axis=1, keepdims=True), 1e-12, None)
    pred = blend.argmax(axis=1)
    return float(f1_score(
        y_true, pred, labels=list(range(NUM_CLASSES)),
        average="macro", zero_division=0
    )), float(accuracy_score(y_true, pred)), blend

base_f1, base_acc, _ = score_weights(0.55, 0.05, 0.40, 0.0)
replace_b2_f1, replace_b2_acc, _ = score_weights(0.55, 0.0, 0.40, 0.05)
best = weight_df.iloc[0]

print("\n" + "=" * 78)
print("B3 CONTRIBUTION SUMMARY")
print("=" * 78)
print(f"Baseline triple (R55/B2-05/C40): F1={base_f1:.5f}, acc={base_acc:.5f}")
print(f"Replace B2 5% with B3 5%       : F1={replace_b2_f1:.5f}, acc={replace_b2_acc:.5f}")
print("Best searched OOF configuration:")
for col in [
    "resnet50_weight", "efficientnet_b2_weight",
    "convnext_v4_weight", "efficientnet_b3_weight"
]:
    print(f"  {col}: {float(best[col]):.2f}")
print(f"  Macro F1: {float(best['oof_macro_f1']):.5f}")
print(f"  Accuracy: {float(best['oof_accuracy']):.5f}")

# Save the test blend from best OOF-selected four-model weights for offline analysis.
P_R_TEST = np.load(find_artifact("test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy"))
# The saved 3-model ensemble probabilities are retained as a consistency reference.
if P_R_TEST.shape != (len(sample), NUM_CLASSES):
    raise ValueError(f"Saved test ensemble shape unexpected: {P_R_TEST.shape}")

# Individual test probabilities for R, B2 and C are reloaded to apply the searched weights.
with np.load(find_artifact("tta_test_crop_probabilities.npz")) as z:
    test_r5 = np.asarray(z["test_r5"], dtype=np.float64)
    test_r8 = np.asarray(z["test_r8"], dtype=np.float64)
if test_r5.shape[0] != len(sample) or test_r8.shape[0] != len(sample):
    raise ValueError("ResNet test probabilities do not align with sample submission")
test_p5 = test_r5[:, [0, 2, 4], :].mean(axis=1)
test_p8 = test_r8[:, [0, 2, 4], :].mean(axis=1)
P_R_TEST_COMP = 0.25 * test_p5 + 0.75 * test_p8
P_R_TEST_COMP /= np.clip(P_R_TEST_COMP.sum(axis=1, keepdims=True), 1e-12, None)
P_B2_TEST = np.asarray(np.load(find_artifact("test_effnet_b2_audio8s_tta3.npy")), dtype=np.float64)
P_C_TEST = np.asarray(np.load(find_artifact("test_convnext_tiny_pretrained_v4_resize224_tta3.npy")), dtype=np.float64)
P_B3_TEST = test_b3.astype(np.float64)

for name, arr in [("ResNet test", P_R_TEST_COMP), ("B2 test", P_B2_TEST), ("ConvNeXt test", P_C_TEST), ("B3 test", P_B3_TEST)]:
    if arr.shape != (len(sample), NUM_CLASSES):
        raise ValueError(f"{name} shape mismatch: {arr.shape}")
    arr /= np.clip(arr.sum(axis=1, keepdims=True), 1e-12, None)

saved_ensemble = np.asarray(P_R_TEST, dtype=np.float64)
saved_ensemble /= np.clip(saved_ensemble.sum(axis=1, keepdims=True), 1e-12, None)
reconstructed_triple = 0.55 * P_R_TEST_COMP + 0.05 * P_B2_TEST + 0.40 * P_C_TEST
reconstructed_triple /= np.clip(reconstructed_triple.sum(axis=1, keepdims=True), 1e-12, None)
test_diff = float(np.max(np.abs(saved_ensemble - reconstructed_triple)))
print("\nSaved vs reconstructed existing triple test probabilities max diff:", test_diff)

best_test = (
    float(best["resnet50_weight"]) * P_R_TEST_COMP
    + float(best["efficientnet_b2_weight"]) * P_B2_TEST
    + float(best["convnext_v4_weight"]) * P_C_TEST
    + float(best["efficientnet_b3_weight"]) * P_B3_TEST
)
best_test /= np.clip(best_test.sum(axis=1, keepdims=True), 1e-12, None)
np.save(BEST_BLEND_TEST_PATH, best_test.astype(np.float32))

metrics = {
    "seed": SEED,
    "n_folds": N_FOLDS,
    "labels": LABELS,
    "pretrained": False,
    "effnet_b3_oof_macro_f1": float(b3_macro_f1),
    "effnet_b3_oof_accuracy": float(b3_acc),
    "baseline_triple_oof_macro_f1": float(base_f1),
    "baseline_triple_oof_accuracy": float(base_acc),
    "replace_b2_5pct_with_b3_5pct_oof_macro_f1": float(replace_b2_f1),
    "replace_b2_5pct_with_b3_5pct_oof_accuracy": float(replace_b2_acc),
    "best_four_model_weights": {
        "resnet50": float(best["resnet50_weight"]),
        "efficientnet_b2": float(best["efficientnet_b2_weight"]),
        "convnext_v4": float(best["convnext_v4_weight"]),
        "efficientnet_b3": float(best["efficientnet_b3_weight"]),
    },
    "best_four_model_oof_macro_f1": float(best["oof_macro_f1"]),
    "best_four_model_oof_accuracy": float(best["oof_accuracy"]),
    "existing_saved_ensemble_test_max_abs_difference": test_diff,
    "artifacts": {
        "oof_b3": OOF_B3_PATH,
        "test_b3": TEST_B3_PATH,
        "weight_search": OOF_SEARCH_PATH,
        "best_oof_blend_test_probabilities": BEST_BLEND_TEST_PATH,
    },
    "note": (
        "OOF metrics cover only the three known classes. Best OOF weights are selected "
        "on the same OOF set and should be treated as exploratory. No submission CSV "
        "is generated or uploaded by this script. The previous public best score 0.65057 "
        "remains the reference until a new submission is separately tested."
    ),
}
with open(OOF_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("\n" + "=" * 78)
print("EFFICIENTNET-B3 EXPERIMENT COMPLETE")
print("=" * 78)
print("OOF B3 probabilities:", OOF_B3_PATH)
print("Test B3 probabilities:", TEST_B3_PATH)
print("OOF weight search:", OOF_SEARCH_PATH)
print("Best OOF blend test probabilities:", BEST_BLEND_TEST_PATH)
print("Metrics JSON:", OOF_METRICS_PATH)
print("Fold metrics CSV:", os.path.join(OUT, "effnet_b3_fold_metrics.csv"))
print("No submission was created or submitted.")


Using saved group folds: /kaggle/input/notebooks/radithiafatir/testttrisan/bird_group_folds_seed16.csv
SCHEMATICS NPC DSC 2026 — EfficientNet-B3 8s GroupCV
Device: cuda
GPU: Tesla T4
AMP: True
Train rows: 1826
Test rows: 814
Labels: ['Bondol Jawa', 'Cendrawasih', 'Kutilang']
Fold counts: {0: 284, 1: 495, 2: 553, 3: 494}
Architecture: torchvision efficientnet_b3(weights=None), 1-channel input
Checkpoint directory: /kaggle/working/effnet_b3_audio8s_group4_v1

Preparing/reusing train mel cache...
  TRAIN mel cache: 200/1826; newly computed=0
  TRAIN mel cache: 400/1826; newly computed=0
  TRAIN mel cache: 600/1826; newly computed=0
  TRAIN mel cache: 800/1826; newly computed=0
  TRAIN mel cache: 1000/1826; newly computed=0
  TRAIN mel cache: 1200/1826; newly computed=0
  TRAIN mel cache: 1400/1826; newly computed=0
  TRAIN mel cache: 1600/1826; newly computed=0
  TRAIN mel cache: 1800/1826; newly computed=0
  TRAIN mel cache: 1826/1826; newly computed=0
Train mel paths ready: 1826

Prepar

In [9]:
import os
import numpy as np
import pandas as pd

ROOT = "/kaggle/input/competitions/schematics-2026"
SRC = "/kaggle/input/notebooks/radithiafatir/testttrisan"
OUT = "/kaggle/working"

N_UNKNOWN = 245
W_CONFIDENCE = 0.60
W_DISAGREEMENT = 0.40

# Pastikan file probabilitas dari eksperimen B3 masih ada.
blend_path = os.path.join(
    OUT, "test_resnet_b2_convnext_b3_best_oof_blend_probabilities.npy"
)
b3_path = os.path.join(
    OUT, "test_effnet_b3_audio8s_tta3.npy"
)
b2_path = os.path.join(
    SRC, "test_effnet_b2_audio8s_tta3.npy"
)
convnext_path = os.path.join(
    SRC, "test_convnext_tiny_pretrained_v4_resize224_tta3.npy"
)
crop_path = os.path.join(SRC, "tta_test_crop_probabilities.npz")
sample_path = os.path.join(ROOT, "sample_submission.csv")

for path in [blend_path, b3_path, b2_path, convnext_path, crop_path, sample_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"File tidak ditemukan: {path}\n"
            "Periksa apakah output B3 sudah tersimpan dan input notebook terpasang."
        )

sample = pd.read_csv(sample_path)
n = len(sample)

# Ensemble empat model terbaik berdasarkan hasil pencarian OOF.
P_BLEND = np.asarray(np.load(blend_path), dtype=np.float64)

# Probabilitas individual untuk mengukur disagreement antar-model.
P_B2 = np.asarray(np.load(b2_path), dtype=np.float64)
P_CONVNEXT = np.asarray(np.load(convnext_path), dtype=np.float64)
P_B3 = np.asarray(np.load(b3_path), dtype=np.float64)

# Rekonstruksi probabilitas ResNet menggunakan resep eksperimen sebelumnya.
with np.load(crop_path) as z:
    test_r5 = np.asarray(z["test_r5"], dtype=np.float64)
    test_r8 = np.asarray(z["test_r8"], dtype=np.float64)

P_R5 = test_r5[:, [0, 2, 4], :].mean(axis=1)
P_R8 = test_r8[:, [0, 2, 4], :].mean(axis=1)
P_RESNET = 0.25 * P_R5 + 0.75 * P_R8

arrays = {
    "Ensemble": P_BLEND,
    "ResNet50": P_RESNET,
    "EfficientNet-B2": P_B2,
    "ConvNeXt": P_CONVNEXT,
    "EfficientNet-B3": P_B3,
}

for name, p in arrays.items():
    if p.shape != (n, 3):
        raise ValueError(f"{name} memiliki shape {p.shape}, seharusnya {(n, 3)}")
    if not np.isfinite(p).all():
        raise ValueError(f"{name} memiliki nilai NaN/inf")
    arrays[name] = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)

P_BLEND = arrays["Ensemble"]
P_RESNET = arrays["ResNet50"]
P_B2 = arrays["EfficientNet-B2"]
P_CONVNEXT = arrays["ConvNeXt"]
P_B3 = arrays["EfficientNet-B3"]

# Disagreement rata-rata: Total Variation Distance dari keenam pasangan model.
models = [P_RESNET, P_B2, P_CONVNEXT, P_B3]
pairwise_tv = []

for i in range(len(models)):
    for j in range(i + 1, len(models)):
        tv = 0.5 * np.abs(models[i] - models[j]).sum(axis=1)
        pairwise_tv.append(tv)

disagreement = np.mean(pairwise_tv, axis=0)

# Confidence ensemble = probabilitas kelas terbesar.
confidence = P_BLEND.max(axis=1)

# Persentil rendah berarti confidence rendah atau disagreement tinggi.
confidence_rank = pd.Series(confidence).rank(pct=True).to_numpy()
disagreement_rank = pd.Series(-disagreement).rank(pct=True).to_numpy()

unknown_score = (
    W_CONFIDENCE * confidence_rank
    + W_DISAGREEMENT * disagreement_rank
)

if not 0 < N_UNKNOWN < n:
    raise ValueError(f"Jumlah Unknown tidak valid: {N_UNKNOWN} dari {n}")

unknown_idx = np.argsort(unknown_score)[:N_UNKNOWN]

# Prediksi tiga kelas dari ensemble, lalu ubah 245 baris terpilih menjadi Unknown.
labels = np.array(["Bondol Jawa", "Cendrawasih", "Kutilang"])
prediction = labels[P_BLEND.argmax(axis=1)]
prediction[unknown_idx] = "Unknown"

# Pertahankan bentuk dan urutan kolom dari sample_submission.
target_cols = [c for c in sample.columns if c != "file_id"]
if len(target_cols) != 1:
    raise ValueError(
        f"Kolom target tidak bisa ditentukan otomatis: {sample.columns.tolist()}"
    )

submission = sample.copy()
submission[target_cols[0]] = prediction

output_path = os.path.join(
    OUT, "submission_effnet_b3_conf60_dis40_u245.csv"
)
submission.to_csv(output_path, index=False)

print("=" * 65)
print("KANDIDAT EMPAT MODEL SELESAI — BELUM DISUBMIT")
print("=" * 65)
print("Blend weights: ResNet 45%, B2 5%, ConvNeXt 35%, B3 15%")
print("Jumlah Unknown:", int((prediction == "Unknown").sum()))
print("\nDistribusi prediksi:")
print(pd.Series(prediction).value_counts().to_string())
print("\nContoh skor Unknown terpilih:")
print(
    pd.DataFrame({
        "file_id": sample["file_id"],
        "confidence": confidence,
        "disagreement": disagreement,
        "unknown_score": unknown_score,
        "prediction": prediction,
    })
    .iloc[unknown_idx]
    .sort_values("unknown_score")
    .head(10)
    .to_string(index=False)
)
print("\nCSV tersimpan:", output_path)

KANDIDAT EMPAT MODEL SELESAI — BELUM DISUBMIT
Blend weights: ResNet 45%, B2 5%, ConvNeXt 35%, B3 15%
Jumlah Unknown: 245

Distribusi prediksi:
Unknown        245
Kutilang       232
Bondol Jawa    212
Cendrawasih    125

Contoh skor Unknown terpilih:
                                                         file_id  confidence  disagreement  unknown_score prediction
2cf80b20a6fdbf08fe53c72d8111b89482fa82cd95da5b43437e4cc7db22c40a    0.399797      0.514162       0.009582    Unknown
a2c8d490fd07f8a7d8cda75fd8b1a78281bb78f4936459def50e2152598908eb    0.401949      0.526802       0.009828    Unknown
ebc4da25b22680cb81d608dbcd9220f8cdfa8a583247c09ef3716bcb0f43cd05    0.412401      0.503039       0.016708    Unknown
7da66f41da1dd9f6235078b69fcc019fc317553500419a44dd59826f08a8574b    0.376398      0.467992       0.018428    Unknown
81890d827fb44ea6f1579fbd5ce1b371063d6013c663e22fb65d20786c86c90b    0.411067      0.477973       0.024324    Unknown
dc0ef4cd307378f38a8787943ba0df1cd7970f38ae215f51

In [10]:
import os
import numpy as np
import pandas as pd

ROOT = "/kaggle/input/competitions/schematics-2026"
SRC = "/kaggle/input/notebooks/radithiafatir/testttrisan"
OUT = "/kaggle/working"

# Konfigurasi eksperimen terakhir
W_RESNET = 0.50
W_CONVNEXT = 0.35
W_B3 = 0.15

W_CONFIDENCE = 0.60
W_DISAGREEMENT = 0.40
N_UNKNOWN = 245

# Periksa berkas
paths = {
    "sample": os.path.join(ROOT, "sample_submission.csv"),
    "resnet_crops": os.path.join(SRC, "tta_test_crop_probabilities.npz"),
    "convnext": os.path.join(
        SRC, "test_convnext_tiny_pretrained_v4_resize224_tta3.npy"
    ),
    "b3": os.path.join(OUT, "test_effnet_b3_audio8s_tta3.npy"),
}

for name, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"{name} tidak ditemukan: {path}")

sample = pd.read_csv(paths["sample"])
n = len(sample)

# 1. ResNet50: gunakan kombinasi crop yang sama dengan eksperimen sebelumnya
with np.load(paths["resnet_crops"]) as z:
    r5 = np.asarray(z["test_r5"], dtype=np.float64)
    r8 = np.asarray(z["test_r8"], dtype=np.float64)

P_R5 = r5[:, [0, 2, 4], :].mean(axis=1)
P_R8 = r8[:, [0, 2, 4], :].mean(axis=1)
P_RESNET = 0.25 * P_R5 + 0.75 * P_R8

# 2. ConvNeXt-v4 dan EfficientNet-B3
P_CONVNEXT = np.asarray(np.load(paths["convnext"]), dtype=np.float64)
P_B3 = np.asarray(np.load(paths["b3"]), dtype=np.float64)

models = {
    "ResNet50": P_RESNET,
    "ConvNeXt-v4": P_CONVNEXT,
    "EfficientNet-B3": P_B3,
}

# Validasi dan normalisasi probabilitas
for name, p in models.items():
    if p.shape != (n, 3):
        raise ValueError(
            f"{name}: shape {p.shape}; seharusnya {(n, 3)}"
        )
    if not np.isfinite(p).all() or np.any(p < 0):
        raise ValueError(f"Probabilitas {name} tidak valid.")

    models[name] = p / np.clip(
        p.sum(axis=1, keepdims=True), 1e-12, None
    )

P_RESNET = models["ResNet50"]
P_CONVNEXT = models["ConvNeXt-v4"]
P_B3 = models["EfficientNet-B3"]

# 3. Ensemble tiga model, B2 dikeluarkan sepenuhnya
P_BLEND = (
    W_RESNET * P_RESNET
    + W_CONVNEXT * P_CONVNEXT
    + W_B3 * P_B3
)
P_BLEND /= np.clip(P_BLEND.sum(axis=1, keepdims=True), 1e-12, None)

# 4. Hitung disagreement rata-rata antar tiga model
pairs = [
    (P_RESNET, P_CONVNEXT),
    (P_RESNET, P_B3),
    (P_CONVNEXT, P_B3),
]

pairwise_tv = [
    0.5 * np.abs(a - b).sum(axis=1)
    for a, b in pairs
]
disagreement = np.mean(pairwise_tv, axis=0)

# 5. Confidence rendah + disagreement tinggi memperoleh skor terendah
confidence = P_BLEND.max(axis=1)

confidence_rank = pd.Series(confidence).rank(pct=True).to_numpy()
disagreement_rank = pd.Series(-disagreement).rank(pct=True).to_numpy()

unknown_score = (
    W_CONFIDENCE * confidence_rank
    + W_DISAGREEMENT * disagreement_rank
)

if N_UNKNOWN >= n:
    raise ValueError("Jumlah Unknown harus lebih kecil daripada jumlah test.")

unknown_idx = np.argsort(unknown_score)[:N_UNKNOWN]

labels = np.array(["Bondol Jawa", "Cendrawasih", "Kutilang"])
prediction = labels[P_BLEND.argmax(axis=1)]
prediction[unknown_idx] = "Unknown"

# 6. Bentuk CSV sesuai sample_submission
target_cols = [c for c in sample.columns if c != "file_id"]
if len(target_cols) != 1:
    raise ValueError(
        f"Kolom target tidak dikenali: {sample.columns.tolist()}"
    )

submission = sample.copy()
submission[target_cols[0]] = prediction

output_path = os.path.join(
    OUT, "submission_b3_replace_b2_conf60_dis40_u245.csv"
)
submission.to_csv(output_path, index=False)

# 7. Pemeriksaan sebelum submission
check = pd.read_csv(output_path)

assert len(check) == len(sample)
assert check["file_id"].equals(sample["file_id"])
assert check[target_cols[0]].notna().all()
assert (check[target_cols[0]] == "Unknown").sum() == N_UNKNOWN
assert set(check[target_cols[0]].unique()).issubset(
    {"Bondol Jawa", "Cendrawasih", "Kutilang", "Unknown"}
)

print("=" * 64)
print("KANDIDAT TERAKHIR: B3 MENGGANTIKAN B2")
print("=" * 64)
print("Bobot: ResNet50=50%, ConvNeXt-v4=35%, B3=15%")
print("Unknown:", int((prediction == "Unknown").sum()))
print("\nDistribusi label:")
print(pd.Series(prediction).value_counts().to_string())
print("\nCSV:", output_path)
print("\nValidasi format: LULUS")
print("Status: BELUM DISUBMIT")

KANDIDAT TERAKHIR: B3 MENGGANTIKAN B2
Bobot: ResNet50=50%, ConvNeXt-v4=35%, B3=15%
Unknown: 245

Distribusi label:
Unknown        245
Kutilang       229
Bondol Jawa    214
Cendrawasih    126

CSV: /kaggle/working/submission_b3_replace_b2_conf60_dis40_u245.csv

Validasi format: LULUS
Status: BELUM DISUBMIT


In [11]:
# -*- coding: utf-8 -*-
"""
SCHEMATICS NPC DSC 2026 — BEATs pretrained frozen embeddings + group-CV classifier

- Uses the public Microsoft BEATs_iter3_plus_AS2M pretrained checkpoint (not gated).
- Extracts frozen BEATs embeddings for fixed 8-second, 16 kHz mono audio.
- Evaluates lightweight classifiers using the saved seed-16 group folds.
- Compares BEATs OOF probabilities to the known best triple OOF blend if artifacts exist.
- Saves probability arrays only. It DOES NOT create or submit a Kaggle submission.

Internet may be required the first time to download official source code and checkpoint.
If internet is disabled, attach BEATs_iter3_plus_AS2M.pt as a Kaggle input and ensure the
three official source files (BEATs.py, backbone.py, modules.py) are available locally.
"""

import os
import sys
import glob
import json
import time
import random
import shutil
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score, classification_report

warnings.filterwarnings("ignore", category=FutureWarning)

# =============================================================================
# CONFIG
# =============================================================================
SEED = 16
N_FOLDS = 4
SAMPLE_RATE = 16000
DURATION_SECONDS = 8.0
NUM_SAMPLES = int(SAMPLE_RATE * DURATION_SECONDS)
BATCH_SIZE = 8
NUM_WORKERS = 2
USE_AMP = True

ROOT = "/kaggle/input/competitions/schematics-2026"
OUT = "/kaggle/working"
SOURCE_OUTPUT = "/kaggle/input/notebooks/radithiafatir/testttrisan"
CHECKPOINT_FILENAME = "BEATs_iter3_plus_AS2M.pt"
CHECKPOINT_REPO_ID = "lpepino/beats_ckpts"  # public mirror of official Microsoft checkpoints
BEATS_CODE_DIR = os.path.join(OUT, "beats_official_code")

TRAIN_META_PATH = os.path.join(ROOT, "train", "metadata.csv")
SAMPLE_PATH = os.path.join(ROOT, "sample_submission.csv")
TRAIN_EMB_PATH = os.path.join(OUT, "beats_iter3plus_as2m_8s_train_embeddings.npy")
TEST_EMB_PATH = os.path.join(OUT, "beats_iter3plus_as2m_8s_test_embeddings.npy")
OOF_BEST_PATH = os.path.join(OUT, "beats_iter3plus_best_oof_probabilities.npy")
TEST_BEST_PATH = os.path.join(OUT, "beats_iter3plus_best_test_probabilities.npy")
CLASSIFIER_RESULTS_PATH = os.path.join(OUT, "beats_iter3plus_classifier_oof_results.csv")
BLEND_RESULTS_PATH = os.path.join(OUT, "beats_vs_triple_oof_blend_search.csv")
BEST_BLEND_TEST_PATH = os.path.join(OUT, "test_beats_best_oof_blend_probabilities.npy")
METRICS_JSON_PATH = os.path.join(OUT, "beats_iter3plus_metrics.json")

EXPECTED_LABELS = ["Bondol Jawa", "Cendrawasih", "Kutilang"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(OUT, exist_ok=True)
os.makedirs(BEATS_CODE_DIR, exist_ok=True)


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def find_first_file(filename, search_dirs=None):
    """Find exact-name artifact without assuming arbitrary Kaggle input IDs."""
    dirs = search_dirs or [OUT, SOURCE_OUTPUT, "/kaggle/input"]
    for directory in dirs:
        candidate = os.path.join(directory, filename)
        if os.path.isfile(candidate):
            return candidate
    for directory in dirs:
        if os.path.isdir(directory):
            matches = glob.glob(os.path.join(directory, "**", filename), recursive=True)
            if matches:
                return matches[0]
    return None


def download_file(url, destination):
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    tmp = destination + ".part"
    print(f"Downloading: {url}")
    try:
        urllib.request.urlretrieve(url, tmp)
        if os.path.getsize(tmp) < 1000:
            raise RuntimeError(f"Download looks too small ({os.path.getsize(tmp)} bytes)")
        os.replace(tmp, destination)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)


def ensure_beats_source():
    """Use a complete local official source bundle when mounted; otherwise download 3 files."""
    source_files = ["BEATs.py", "backbone.py", "modules.py"]
    candidate_dirs = [BEATS_CODE_DIR]
    # Support fully offline runs if the three official source files were attached as a Kaggle input.
    for beats_py in glob.glob(os.path.join("/kaggle/input", "**", "BEATs.py"), recursive=True):
        candidate_dirs.append(os.path.dirname(beats_py))
    for directory in candidate_dirs:
        if all(os.path.isfile(os.path.join(directory, name)) for name in source_files):
            if directory not in sys.path:
                sys.path.insert(0, directory)
            return directory

    source_base = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"
    for name in source_files:
        destination = os.path.join(BEATS_CODE_DIR, name)
        if os.path.isfile(destination):
            continue
        try:
            download_file(source_base + name, destination)
        except Exception as exc:
            raise RuntimeError(
                f"Tidak bisa memperoleh kode resmi BEATs ({name}). Aktifkan Internet di Kaggle, "
                "atau lampirkan satu folder input yang berisi ketiga file resmi: "
                "BEATs.py, backbone.py, dan modules.py."
            ) from exc
    if BEATS_CODE_DIR not in sys.path:
        sys.path.insert(0, BEATS_CODE_DIR)
    return BEATS_CODE_DIR


def ensure_checkpoint():
    # First: environment override, then mounted Kaggle inputs, then Hugging Face public mirror.
    env_path = os.environ.get("BEATS_CHECKPOINT_PATH", "").strip()
    if env_path and os.path.isfile(env_path):
        return env_path

    existing = find_first_file(CHECKPOINT_FILENAME, search_dirs=["/kaggle/input", OUT])
    if existing:
        return existing

    destination = os.path.join(OUT, "pretrained", CHECKPOINT_FILENAME)
    if os.path.isfile(destination) and os.path.getsize(destination) > 100_000_000:
        return destination

    # Prefer huggingface_hub; fallback to a direct public resolve URL.
    try:
        from huggingface_hub import hf_hub_download
        downloaded = hf_hub_download(
            repo_id=CHECKPOINT_REPO_ID,
            filename=CHECKPOINT_FILENAME,
            local_dir=os.path.dirname(destination),
        )
        if os.path.isfile(downloaded) and os.path.getsize(downloaded) > 100_000_000:
            return downloaded
    except Exception as exc:
        print("huggingface_hub download failed; trying direct URL. Reason:", repr(exc))

    url = f"https://huggingface.co/{CHECKPOINT_REPO_ID}/resolve/main/{CHECKPOINT_FILENAME}"
    try:
        download_file(url, destination)
        return destination
    except Exception as exc:
        raise RuntimeError(
            "Checkpoint BEATs belum ditemukan dan unduhan gagal. Aktifkan Internet Kaggle, "
            "atau unduh file publik BEATs_iter3_plus_AS2M.pt lalu tambahkan sebagai Kaggle Input. "
            "Sesudah ditambahkan, jalankan ulang notebook."
        ) from exc


def load_beats_model(checkpoint_path):
    ensure_beats_source()
    # The checkpoint is a public, expected BEATs state dictionary; weights_only=False is needed
    # because the official checkpoint stores both cfg and model entries.
    try:
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:  # compatibility with older PyTorch
        checkpoint = torch.load(checkpoint_path, map_location="cpu")

    from BEATs import BEATs, BEATsConfig

    cfg = BEATsConfig(checkpoint["cfg"])
    model = BEATs(cfg)
    model.load_state_dict(checkpoint["model"], strict=True)
    model.eval()
    model.to(DEVICE)

    # BEATs' official extract_features() computes Kaldi filterbanks internally.
    # torchaudio.compliance.kaldi expects CPU waveforms. Run that preprocessing on CPU,
    # then move the filterbank tensor to the model device before the neural network layers.
    original_preprocess = model.preprocess

    def preprocess_to_model_device(self, source, fbank_mean=15.41663, fbank_std=6.55582):
        source_cpu = source.detach().to("cpu")
        fbanks = original_preprocess(source_cpu, fbank_mean=fbank_mean, fbank_std=fbank_std)
        return fbanks.to(DEVICE, non_blocking=True)

    import types
    model.preprocess = types.MethodType(preprocess_to_model_device, model)

    # This checkpoint is the pretrained encoder (not the AudioSet output classifier).
    # Defensive handling allows embeddings to be extracted if a classifier head exists.
    if getattr(model, "predictor", None) is not None:
        model.predictor = None

    print("BEATs cfg encoder dim:", getattr(cfg, "encoder_embed_dim", "unknown"))
    print("BEATs encoder layers:", getattr(cfg, "encoder_layers", "unknown"))
    print("BEATs checkpoint:", checkpoint_path)
    return model


def base_id(file_id):
    return os.path.splitext(os.path.basename(str(file_id)))[0]


def resolve_train_audio(row):
    raw = str(row.get("audio_path", ""))
    file_id = str(row["file_id"])
    raw_base = os.path.basename(raw)
    file_base = base_id(file_id)
    audio_dir = os.path.join(ROOT, "train", "audio")
    candidates = []
    if os.path.isabs(raw):
        candidates.append(raw)
    else:
        candidates.extend([os.path.join(ROOT, raw), os.path.join(ROOT, "train", raw)])
    candidates.extend([os.path.join(audio_dir, file_id), os.path.join(audio_dir, raw_base)])
    for ext in [".flac", ".wav", ".mp3", ".ogg", ".m4a"]:
        candidates.append(os.path.join(audio_dir, file_base + ext))
        candidates.append(os.path.join(audio_dir, base_id(raw_base) + ext))
    for path in candidates:
        if path and os.path.isfile(path):
            return path
    matches = glob.glob(os.path.join(audio_dir, file_base + ".*"))
    if matches:
        return matches[0]
    matches = glob.glob(os.path.join(audio_dir, base_id(raw_base) + ".*"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Audio train tidak ditemukan: file_id={file_id}; audio_path={raw}")


def resolve_test_audio(file_id):
    file_id = str(file_id)
    file_base = base_id(file_id)
    audio_dir = os.path.join(ROOT, "test", "audio")
    candidates = [os.path.join(audio_dir, file_id)]
    for ext in [".flac", ".wav", ".mp3", ".ogg", ".m4a"]:
        candidates.append(os.path.join(audio_dir, file_base + ext))
    for path in candidates:
        if os.path.isfile(path):
            return path
    matches = glob.glob(os.path.join(audio_dir, file_base + ".*"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Audio test tidak ditemukan: file_id={file_id}")


class WaveDataset(Dataset):
    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        try:
            y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
        except Exception as exc:
            raise RuntimeError(f"Gagal membaca audio: {path}") from exc
        if y.size == 0:
            raise ValueError(f"Audio kosong: {path}")
        if len(y) < NUM_SAMPLES:
            y = np.pad(y, (0, NUM_SAMPLES - len(y)))
        else:
            y = y[:NUM_SAMPLES]
        y = np.asarray(y, dtype=np.float32)
        return torch.from_numpy(y), idx


def extract_embeddings(model, paths, output_path, split_name):
    if os.path.isfile(output_path):
        try:
            old = np.load(output_path, mmap_mode="r")
            if old.ndim == 2 and old.shape[0] == len(paths) and old.shape[1] >= 256:
                print(f"Reusing cached {split_name} embeddings: {old.shape} at {output_path}")
                return np.asarray(old, dtype=np.float32)
        except Exception:
            pass

    dataset = WaveDataset(paths)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        persistent_workers=(NUM_WORKERS > 0),
    )
    chunks = []
    model.eval()
    start = time.time()
    with torch.inference_mode():
        for batch_idx, (waves, indices) in enumerate(loader):
            # Keep waveforms on CPU because Kaldi fbank runs on CPU; patched preprocessing
            # moves the fbank representations to GPU after computation.
            waves = waves.float()
            if DEVICE.type == "cuda" and USE_AMP:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    features, _ = model.extract_features(waves)
            else:
                features, _ = model.extract_features(waves)
            if features.ndim != 3:
                raise RuntimeError(f"Unexpected BEATs output dimensions: {tuple(features.shape)}")
            # Temporal mean pooling -> one fixed-size embedding per audio clip.
            embedding = features.float().mean(dim=1).cpu().numpy().astype(np.float32)
            chunks.append(embedding)
            done = min((batch_idx + 1) * BATCH_SIZE, len(dataset))
            if done % 200 < BATCH_SIZE or done == len(dataset):
                print(f"  {split_name} embeddings: {done}/{len(dataset)}")

    embeddings = np.concatenate(chunks, axis=0)
    if embeddings.shape[0] != len(paths):
        raise RuntimeError(f"Embedding row count mismatch: {embeddings.shape} for {len(paths)} paths")
    np.save(output_path, embeddings)
    print(f"Saved {split_name} embeddings: {embeddings.shape} -> {output_path}; {time.time()-start:.1f}s")
    return embeddings


def build_classifier(name):
    if name.startswith("logreg"):
        c_value = float(name.split("C")[-1])
        clf = LogisticRegression(
            C=c_value,
            class_weight="balanced",
            max_iter=2500,
            solver="lbfgs",
            random_state=SEED,
        )
    elif name.startswith("rbf_svc"):
        c_value = float(name.split("C")[-1])
        clf = SVC(
            C=c_value,
            kernel="rbf",
            gamma="scale",
            class_weight="balanced",
            probability=True,
            cache_size=1000,
            random_state=SEED,
        )
    else:
        raise ValueError(f"Unknown classifier config: {name}")
    return make_pipeline(StandardScaler(), clf)


def aligned_predict_proba(estimator, X, n_classes):
    probs = estimator.predict_proba(X)
    result = np.zeros((len(X), n_classes), dtype=np.float32)
    for col, class_id in enumerate(estimator[-1].classes_):
        result[:, int(class_id)] = probs[:, col]
    result /= np.clip(result.sum(axis=1, keepdims=True), 1e-12, None)
    return result


def normalize_probs(p):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, 1e-9, None)
    return p / p.sum(axis=1, keepdims=True)


def metric_row(name, y, probs):
    preds = probs.argmax(axis=1)
    return {
        "classifier": name,
        "oof_macro_f1": float(f1_score(y, preds, average="macro")),
        "oof_accuracy": float(accuracy_score(y, preds)),
    }


def load_previous_oof_and_test(meta, sample):
    """Rebuild the current best triple baseline, if its existing artifacts are attached."""
    names = {
        "resnet": "oof_resnet50_ensemble_tta3_seed16.npy",
        "b2": "oof_efficientnet_b2_audio8s_tta3_seed16.npy",
        "convnext": "oof_convnext_tiny_audio8s_tta3_seed16_pretrained_v4_resize224.npy",
    }
    found = {k: find_first_file(v) for k, v in names.items()}
    missing = [k for k, p in found.items() if p is None]
    if missing:
        print("Previous OOF files absent; BEATs-vs-triple OOF comparison will be skipped:", missing)
        return None, None

    probs = {k: np.asarray(np.load(path), dtype=np.float64) for k, path in found.items()}
    expected_shape = (len(meta), len(EXPECTED_LABELS))
    for key, p in probs.items():
        if p.shape != expected_shape:
            print(f"OOF shape mismatch for {key}: {p.shape} vs {expected_shape}; skipping blend comparison")
            return None, None
        probs[key] = normalize_probs(p)

    # Existing best triple probability blend from the earlier experiment: R55 / B2-05 / ConvNeXt-40.
    baseline_oof = normalize_probs(
        0.55 * probs["resnet"] + 0.05 * probs["b2"] + 0.40 * probs["convnext"]
    ).astype(np.float32)

    test_name = "test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy"
    test_path = find_first_file(test_name)
    baseline_test = None
    if test_path:
        arr = np.asarray(np.load(test_path), dtype=np.float64)
        if arr.shape == (len(sample), len(EXPECTED_LABELS)):
            baseline_test = normalize_probs(arr).astype(np.float32)
        else:
            print("Baseline test probability shape mismatch; cannot save test blend:", arr.shape)
    else:
        print("Baseline triple test probability file absent; OOF comparison still available.")
    return baseline_oof, baseline_test


def main():
    seed_everything()
    print("=" * 78)
    print("SCHEMATICS NPC DSC 2026 — BEATs Iter3+ AS2M Frozen Embedding GroupCV")
    print("Device:", DEVICE)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    print("Train/test audio length:", DURATION_SECONDS, "sec at", SAMPLE_RATE, "Hz")
    print("Backbone: BEATs_iter3_plus_AS2M.pt (public pretrained checkpoint)")
    print("Output:", OUT)
    print("=" * 78)

    if not os.path.isfile(TRAIN_META_PATH) or not os.path.isfile(SAMPLE_PATH):
        raise FileNotFoundError(
            f"Dataset kompetisi belum ditemukan. Pastikan input '{ROOT}' ditambahkan ke notebook."
        )
    meta = pd.read_csv(TRAIN_META_PATH).copy()
    sample = pd.read_csv(SAMPLE_PATH).copy()
    required = {"file_id", "audio_path", "label", "source_group"}
    missing = required - set(meta.columns)
    if missing:
        raise ValueError(f"Kolom metadata train hilang: {missing}")
    if "file_id" not in sample.columns:
        raise ValueError("sample_submission.csv tidak memiliki file_id")

    meta["file_id"] = meta["file_id"].astype(str)
    meta["label"] = meta["label"].astype(str)
    meta["source_group"] = meta["source_group"].astype(str)
    sample["file_id"] = sample["file_id"].astype(str)
    labels = sorted(meta["label"].unique().tolist())
    if labels != EXPECTED_LABELS:
        raise ValueError(f"Label tidak sesuai ekspektasi: {labels}")
    label_to_idx = {label: i for i, label in enumerate(labels)}
    y = meta["label"].map(label_to_idx).to_numpy(dtype=np.int64)

    fold_path = find_first_file("bird_group_folds_seed16.csv")
    if fold_path:
        fold_df = pd.read_csv(fold_path)
        if not {"file_id", "fold"}.issubset(fold_df.columns):
            raise ValueError(f"Fold file tidak memiliki file_id/fold: {fold_path}")
        fold_df["file_id"] = fold_df["file_id"].astype(str)
        fold_map = fold_df.drop_duplicates("file_id").set_index("file_id")["fold"]
        mapped = meta["file_id"].map(fold_map)
        if mapped.isna().any() or len(fold_df) != len(meta):
            raise ValueError("Saved seed16 folds tidak cocok dengan train metadata")
        meta["fold"] = mapped.astype(int)
        print("Using saved group folds:", fold_path)
    else:
        print("Saved folds unavailable; generating StratifiedGroupKFold(seed=16)")
        splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
        meta["fold"] = -1
        for fold, (_, val_idx) in enumerate(splitter.split(meta, y, groups=meta["source_group"])):
            meta.loc[val_idx, "fold"] = fold
        meta[["file_id", "label", "source_group", "fold"]].to_csv(
            os.path.join(OUT, "beats_generated_group_folds_seed16.csv"), index=False
        )
    if sorted(meta["fold"].unique().tolist()) != list(range(N_FOLDS)):
        raise ValueError(f"Fold IDs harus 0..3; ditemukan {sorted(meta['fold'].unique().tolist())}")

    print("Train rows:", len(meta), "| test rows:", len(sample))
    print("Class counts:", meta["label"].value_counts().to_dict())
    print("Fold counts:", meta["fold"].value_counts().sort_index().to_dict())

    def valid_embedding_cache(path, expected_rows):
        if not os.path.isfile(path):
            return False
        try:
            arr = np.load(path, mmap_mode="r")
            return arr.ndim == 2 and arr.shape[0] == expected_rows and arr.shape[1] >= 256
        except Exception:
            return False

    train_cache_ok = valid_embedding_cache(TRAIN_EMB_PATH, len(meta))
    test_cache_ok = valid_embedding_cache(TEST_EMB_PATH, len(sample))
    if train_cache_ok and test_cache_ok:
        X_train = np.asarray(np.load(TRAIN_EMB_PATH), dtype=np.float32)
        X_test = np.asarray(np.load(TEST_EMB_PATH), dtype=np.float32)
        print("Reusing cached train embeddings:", X_train.shape)
        print("Reusing cached test embeddings:", X_test.shape)
        print("Both embedding caches exist; no checkpoint/code download or audio pass needed.")
    else:
        checkpoint_path = ensure_checkpoint()
        model = load_beats_model(checkpoint_path)

        print("\nResolving audio paths...")
        train_paths = [resolve_train_audio(row) for _, row in meta.iterrows()]
        test_paths = [resolve_test_audio(fid) for fid in sample["file_id"].tolist()]
        print("Resolved train paths:", len(train_paths), "| test paths:", len(test_paths))

        print("\nPreparing train embeddings...")
        X_train = extract_embeddings(model, train_paths, TRAIN_EMB_PATH, "TRAIN")
        print("\nPreparing test embeddings...")
        X_test = extract_embeddings(model, test_paths, TEST_EMB_PATH, "TEST")
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("\nEmbedding shapes:", X_train.shape, X_test.shape)
    configs = [
        "logreg_C0.03", "logreg_C0.3", "logreg_C3.0",
        "rbf_svc_C0.3", "rbf_svc_C1.0", "rbf_svc_C3.0", "rbf_svc_C10.0",
    ]
    results = []
    predictions_by_config = {}

    print("\nEvaluating frozen-embedding classifiers using group OOF...")
    for config_name in configs:
        start = time.time()
        oof_probs = np.zeros((len(meta), len(labels)), dtype=np.float32)
        test_probs_sum = np.zeros((len(sample), len(labels)), dtype=np.float64)
        for fold in range(N_FOLDS):
            train_idx = np.where(meta["fold"].to_numpy() != fold)[0]
            val_idx = np.where(meta["fold"].to_numpy() == fold)[0]
            if len(np.unique(y[train_idx])) != len(labels):
                raise ValueError(f"Classifier train split fold {fold} tidak memiliki semua kelas")
            estimator = build_classifier(config_name)
            estimator.fit(X_train[train_idx], y[train_idx])
            oof_probs[val_idx] = aligned_predict_proba(estimator, X_train[val_idx], len(labels))
            test_probs_sum += aligned_predict_proba(estimator, X_test, len(labels))
            print(f"  {config_name}: fold {fold + 1}/{N_FOLDS} complete")

        test_probs = normalize_probs(test_probs_sum / N_FOLDS).astype(np.float32)
        oof_probs = normalize_probs(oof_probs).astype(np.float32)
        row = metric_row(config_name, y, oof_probs)
        row["elapsed_seconds"] = round(time.time() - start, 1)
        results.append(row)
        predictions_by_config[config_name] = {"oof": oof_probs, "test": test_probs}
        print(f"  RESULT {config_name}: OOF Macro F1={row['oof_macro_f1']:.5f} | acc={row['oof_accuracy']:.5f}")

    result_df = pd.DataFrame(results).sort_values(
        ["oof_macro_f1", "oof_accuracy"], ascending=False
    ).reset_index(drop=True)
    result_df.to_csv(CLASSIFIER_RESULTS_PATH, index=False)
    best_config = str(result_df.iloc[0]["classifier"])
    best_oof = predictions_by_config[best_config]["oof"]
    best_test = predictions_by_config[best_config]["test"]
    np.save(OOF_BEST_PATH, best_oof)
    np.save(TEST_BEST_PATH, best_test)

    print("\n" + "=" * 78)
    print("BEATs classifier OOF ranking")
    print(result_df.to_string(index=False))
    print("\nBest selected config:", best_config)
    print(classification_report(y, best_oof.argmax(axis=1), target_names=labels, digits=4, zero_division=0))
    print("OOF probabilities:", OOF_BEST_PATH)
    print("Test probabilities:", TEST_BEST_PATH)

    baseline_oof, baseline_test = load_previous_oof_and_test(meta, sample)
    blend_rows = []
    best_weight = 0.0
    best_blend_oof = None
    best_blend_test = None
    if baseline_oof is not None:
        baseline_oof_f1 = f1_score(y, baseline_oof.argmax(axis=1), average="macro")
        print(f"Existing triple OOF baseline F1 (reconstructed): {baseline_oof_f1:.5f}")
        for weight in np.round(np.arange(0.0, 0.501, 0.05), 2):
            blend_oof = normalize_probs((1.0 - weight) * baseline_oof + weight * best_oof).astype(np.float32)
            row = {
                "beats_weight": float(weight),
                "triple_baseline_weight": float(1.0 - weight),
                "oof_macro_f1": float(f1_score(y, blend_oof.argmax(axis=1), average="macro")),
                "oof_accuracy": float(accuracy_score(y, blend_oof.argmax(axis=1))),
            }
            blend_rows.append(row)
        blend_df = pd.DataFrame(blend_rows).sort_values(
            ["oof_macro_f1", "oof_accuracy"], ascending=False
        ).reset_index(drop=True)
        blend_df.to_csv(BLEND_RESULTS_PATH, index=False)
        best_weight = float(blend_df.iloc[0]["beats_weight"])
        best_blend_oof = normalize_probs(
            (1.0 - best_weight) * baseline_oof + best_weight * best_oof
        ).astype(np.float32)
        if baseline_test is not None:
            best_blend_test = normalize_probs(
                (1.0 - best_weight) * baseline_test + best_weight * best_test
            ).astype(np.float32)
            np.save(BEST_BLEND_TEST_PATH, best_blend_test)
        print("\nBEATs + triple OOF blend weight search (exploratory; selected on these OOF scores):")
        print(blend_df.head(11).to_string(index=False))
        print(f"Selected BEATs blend weight from OOF search: {best_weight:.2f}")
        if best_blend_test is not None:
            print("Saved test blend probabilities:", BEST_BLEND_TEST_PATH)
    else:
        print("\nPrevious triple OOF artifacts unavailable: skipping blend weight search.")

    metrics = {
        "checkpoint": CHECKPOINT_FILENAME,
        "checkpoint_repo": CHECKPOINT_REPO_ID,
        "duration_seconds": DURATION_SECONDS,
        "sample_rate": SAMPLE_RATE,
        "embedding_shape_train": list(X_train.shape),
        "embedding_shape_test": list(X_test.shape),
        "labels": labels,
        "folds": int(N_FOLDS),
        "seed": int(SEED),
        "best_classifier_by_oof_macro_f1": best_config,
        "best_classifier_oof_macro_f1": float(result_df.iloc[0]["oof_macro_f1"]),
        "best_classifier_oof_accuracy": float(result_df.iloc[0]["oof_accuracy"]),
        "best_beats_weight_in_triple_blend": best_weight if baseline_oof is not None else None,
        "oof_scores_are_selection_scores_not_unbiased_estimates": True,
        "submission_created": False,
    }
    with open(METRICS_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print("\n" + "=" * 78)
    print("BEATs EXPERIMENT COMPLETE — NO SUBMISSION CREATED")
    print("Train embeddings:", TRAIN_EMB_PATH)
    print("Test embeddings:", TEST_EMB_PATH)
    print("Classifier comparison:", CLASSIFIER_RESULTS_PATH)
    print("Best OOF probabilities:", OOF_BEST_PATH)
    print("Best test probabilities:", TEST_BEST_PATH)
    print("Blend search:", BLEND_RESULTS_PATH if baseline_oof is not None else "skipped")
    print("Metrics:", METRICS_JSON_PATH)
    print("=" * 78)


if __name__ == "__main__":
    main()


SCHEMATICS NPC DSC 2026 — BEATs Iter3+ AS2M Frozen Embedding GroupCV
Device: cuda
GPU: Tesla T4
Train/test audio length: 8.0 sec at 16000 Hz
Backbone: BEATs_iter3_plus_AS2M.pt (public pretrained checkpoint)
Output: /kaggle/working
Using saved group folds: /kaggle/input/notebooks/radithiafatir/testttrisan/bird_group_folds_seed16.csv
Train rows: 1826 | test rows: 814
Class counts: {'Bondol Jawa': 1124, 'Kutilang': 619, 'Cendrawasih': 83}
Fold counts: {0: 284, 1: 495, 2: 553, 3: 494}


BEATs_iter3_plus_AS2M.pt: reconstructing file:   0%|          |  0.00B /  361MB            

BEATs_iter3_plus_AS2M.pt: downloading bytes:           |  0.00B            

Downloading: https://raw.githubusercontent.com/microsoft/unilm/master/beats/BEATs.py
Downloading: https://raw.githubusercontent.com/microsoft/unilm/master/beats/backbone.py
Downloading: https://raw.githubusercontent.com/microsoft/unilm/master/beats/modules.py
BEATs cfg encoder dim: 768
BEATs encoder layers: 12
BEATs checkpoint: /kaggle/working/pretrained/BEATs_iter3_plus_AS2M.pt

Resolving audio paths...
Resolved train paths: 1826 | test paths: 814

Preparing train embeddings...
  TRAIN embeddings: 200/1826
  TRAIN embeddings: 400/1826
  TRAIN embeddings: 600/1826
  TRAIN embeddings: 800/1826
  TRAIN embeddings: 1000/1826
  TRAIN embeddings: 1200/1826
  TRAIN embeddings: 1400/1826
  TRAIN embeddings: 1600/1826
  TRAIN embeddings: 1800/1826
  TRAIN embeddings: 1826/1826
Saved TRAIN embeddings: (1826, 768) -> /kaggle/working/beats_iter3plus_as2m_8s_train_embeddings.npy; 83.2s

Preparing test embeddings...
  TEST embeddings: 200/814
  TEST embeddings: 400/814
  TEST embeddings: 600/814
  

In [12]:
# -*- coding: utf-8 -*-
"""
SCHEMATICS NPC DSC 2026 — BEATs Iter3+ AS2M partial fine-tuning (GroupCV)

Experiment design:
- Start from the same public BEATs_iter3_plus_AS2M.pt checkpoint used by the frozen-embedding run.
- Freeze the BEATs feature frontend and earlier encoder blocks; fine-tune only the last 2 Transformer
  blocks plus a new 3-class linear head. This is a conservative setting for only 1,826 labelled clips.
- Use the saved seed-16 source_group folds, class-weighted CrossEntropy, modest waveform augmentation,
  mixed precision on GPU, early stopping, and per-fold checkpoints.
- Save OOF/test probabilities and an exploratory blend search against the existing ResNet/B2/ConvNeXt
  baseline. No submission CSV is created and nothing is submitted.

Requires Internet in Kaggle only on the first run if BEATs source/checkpoint are not attached as inputs.
"""

import os
import sys
import glob
import json
import time
import random
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report

warnings.filterwarnings("ignore", category=FutureWarning)

# =============================================================================
# CONFIG
# =============================================================================
SEED = 16
N_FOLDS = 4
SAMPLE_RATE = 16000
DURATION_SECONDS = 8.0
NUM_SAMPLES = int(SAMPLE_RATE * DURATION_SECONDS)
BATCH_SIZE = 4
VAL_BATCH_SIZE = 4
NUM_WORKERS = 2
MAX_EPOCHS = 8
EARLY_STOP_PATIENCE = 3
UNFREEZE_LAST_N_LAYERS = 2
ENCODER_LR = 8e-6
HEAD_LR = 3e-4
WEIGHT_DECAY = 1e-4
USE_AMP = True
LABEL_SMOOTHING = 0.02
EXPERIMENT_VERSION = "beats_iter3plus_top2_ft_seed16_v1"

ROOT = "/kaggle/input/competitions/schematics-2026"
OUT = "/kaggle/working"
SOURCE_OUTPUT = "/kaggle/input/notebooks/radithiafatir/testttrisan"
CHECKPOINT_FILENAME = "BEATs_iter3_plus_AS2M.pt"
CHECKPOINT_REPO_ID = "lpepino/beats_ckpts"  # same public checkpoint mirror that worked in the frozen run
BEATS_CODE_DIR = os.path.join(OUT, "beats_official_code")

TRAIN_META_PATH = os.path.join(ROOT, "train", "metadata.csv")
SAMPLE_PATH = os.path.join(ROOT, "sample_submission.csv")
TRAIN_WAVE_CACHE = os.path.join(OUT, "beats_finetune_train_waveforms_8s_16k.npy")
TEST_WAVE_CACHE = os.path.join(OUT, "beats_finetune_test_waveforms_8s_16k.npy")
TRAIN_WAVE_MANIFEST = TRAIN_WAVE_CACHE + ".json"
TEST_WAVE_MANIFEST = TEST_WAVE_CACHE + ".json"
FOLD_CKPT_DIR = os.path.join(OUT, EXPERIMENT_VERSION, "fold_checkpoints")
OOF_PATH = os.path.join(OUT, "beats_finetune_top2_oof_probabilities.npy")
TEST_PROBS_PATH = os.path.join(OUT, "test_beats_finetune_top2_probabilities.npy")
FOLD_METRICS_PATH = os.path.join(OUT, "beats_finetune_top2_fold_metrics.csv")
BLEND_RESULTS_PATH = os.path.join(OUT, "beats_finetune_top2_vs_triple_oof_blend_search.csv")
BEST_BLEND_TEST_PATH = os.path.join(OUT, "test_beats_finetune_top2_best_oof_blend_probabilities.npy")
METRICS_JSON_PATH = os.path.join(OUT, "beats_finetune_top2_metrics.json")

EXPECTED_LABELS = ["Bondol Jawa", "Cendrawasih", "Kutilang"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(USE_AMP and DEVICE.type == "cuda")

os.makedirs(OUT, exist_ok=True)
os.makedirs(BEATS_CODE_DIR, exist_ok=True)
os.makedirs(FOLD_CKPT_DIR, exist_ok=True)


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Faster on T4; exact bitwise determinism is not required for this experiment.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def find_first_file(filename, search_dirs=None):
    """Search exact-name artifacts, prioritizing expected mounted inputs."""
    dirs = search_dirs or [OUT, SOURCE_OUTPUT, "/kaggle/input"]
    for directory in dirs:
        candidate = os.path.join(directory, filename)
        if os.path.isfile(candidate):
            return candidate
    for directory in dirs:
        if os.path.isdir(directory):
            matches = glob.glob(os.path.join(directory, "**", filename), recursive=True)
            if matches:
                return matches[0]
    return None


def download_file(url, destination):
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    tmp = destination + ".part"
    print(f"Downloading: {url}")
    try:
        import urllib.request
        urllib.request.urlretrieve(url, tmp)
        if os.path.getsize(tmp) < 1000:
            raise RuntimeError(f"Download looks too small ({os.path.getsize(tmp)} bytes)")
        os.replace(tmp, destination)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)


def ensure_beats_source():
    source_files = ["BEATs.py", "backbone.py", "modules.py"]
    candidate_dirs = [BEATS_CODE_DIR]
    for beats_py in glob.glob(os.path.join("/kaggle/input", "**", "BEATs.py"), recursive=True):
        candidate_dirs.append(os.path.dirname(beats_py))
    for directory in candidate_dirs:
        if all(os.path.isfile(os.path.join(directory, name)) for name in source_files):
            if directory not in sys.path:
                sys.path.insert(0, directory)
            return directory

    source_base = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"
    for name in source_files:
        destination = os.path.join(BEATS_CODE_DIR, name)
        if not os.path.isfile(destination):
            try:
                download_file(source_base + name, destination)
            except Exception as exc:
                raise RuntimeError(
                    f"Tidak bisa mengunduh kode resmi BEATs ({name}). Aktifkan Internet Kaggle atau "
                    "lampirkan folder input berisi BEATs.py, backbone.py, dan modules.py."
                ) from exc
    if BEATS_CODE_DIR not in sys.path:
        sys.path.insert(0, BEATS_CODE_DIR)
    return BEATS_CODE_DIR


def ensure_checkpoint():
    env_path = os.environ.get("BEATS_CHECKPOINT_PATH", "").strip()
    if env_path and os.path.isfile(env_path):
        return env_path

    existing = find_first_file(CHECKPOINT_FILENAME, search_dirs=["/kaggle/input", OUT])
    if existing and os.path.getsize(existing) > 100_000_000:
        return existing

    destination = os.path.join(OUT, "pretrained", CHECKPOINT_FILENAME)
    if os.path.isfile(destination) and os.path.getsize(destination) > 100_000_000:
        return destination

    try:
        from huggingface_hub import hf_hub_download
        downloaded = hf_hub_download(
            repo_id=CHECKPOINT_REPO_ID,
            filename=CHECKPOINT_FILENAME,
            local_dir=os.path.dirname(destination),
        )
        if os.path.isfile(downloaded) and os.path.getsize(downloaded) > 100_000_000:
            return downloaded
    except Exception as exc:
        print("huggingface_hub download failed; trying direct URL. Reason:", repr(exc))

    url = f"https://huggingface.co/{CHECKPOINT_REPO_ID}/resolve/main/{CHECKPOINT_FILENAME}"
    try:
        download_file(url, destination)
        return destination
    except Exception as exc:
        raise RuntimeError(
            "Checkpoint BEATs tidak ditemukan dan unduhan gagal. Aktifkan Internet Kaggle, atau "
            "tambahkan BEATs_iter3_plus_AS2M.pt sebagai Kaggle Input."
        ) from exc


def load_beats_backbone(checkpoint_path):
    ensure_beats_source()
    try:
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location="cpu")

    from BEATs import BEATs, BEATsConfig
    cfg = BEATsConfig(checkpoint["cfg"])
    backbone = BEATs(cfg)
    backbone.load_state_dict(checkpoint["model"], strict=True)
    backbone.to(DEVICE)
    backbone.predictor = None  # use sequence embeddings, not the original AudioSet prediction head

    # Official BEATs computes Kaldi filterbanks on CPU. Send them to the model device afterwards.
    original_preprocess = backbone.preprocess
    import types

    def preprocess_to_model_device(self, source, fbank_mean=15.41663, fbank_std=6.55582):
        source_cpu = source.detach().to("cpu")
        fbanks = original_preprocess(source_cpu, fbank_mean=fbank_mean, fbank_std=fbank_std)
        return fbanks.to(DEVICE, non_blocking=True)

    backbone.preprocess = types.MethodType(preprocess_to_model_device, backbone)
    print("BEATs encoder dim:", getattr(cfg, "encoder_embed_dim", "unknown"))
    print("BEATs encoder layers:", len(backbone.encoder.layers))
    print("BEATs checkpoint:", checkpoint_path)
    return backbone


def base_id(file_id):
    return os.path.splitext(os.path.basename(str(file_id)))[0]


def resolve_train_audio(row):
    raw = str(row.get("audio_path", ""))
    file_id = str(row["file_id"])
    raw_base = os.path.basename(raw)
    file_base = base_id(file_id)
    audio_dir = os.path.join(ROOT, "train", "audio")
    candidates = []
    if os.path.isabs(raw):
        candidates.append(raw)
    else:
        candidates.extend([os.path.join(ROOT, raw), os.path.join(ROOT, "train", raw)])
    candidates.extend([os.path.join(audio_dir, file_id), os.path.join(audio_dir, raw_base)])
    for ext in [".flac", ".wav", ".mp3", ".ogg", ".m4a"]:
        candidates.append(os.path.join(audio_dir, file_base + ext))
        candidates.append(os.path.join(audio_dir, base_id(raw_base) + ext))
    for path in candidates:
        if path and os.path.isfile(path):
            return path
    matches = glob.glob(os.path.join(audio_dir, file_base + ".*"))
    if matches:
        return matches[0]
    matches = glob.glob(os.path.join(audio_dir, base_id(raw_base) + ".*"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Audio train tidak ditemukan: file_id={file_id}; audio_path={raw}")


def resolve_test_audio(file_id):
    file_id = str(file_id)
    file_base = base_id(file_id)
    audio_dir = os.path.join(ROOT, "test", "audio")
    candidates = [os.path.join(audio_dir, file_id)]
    for ext in [".flac", ".wav", ".mp3", ".ogg", ".m4a"]:
        candidates.append(os.path.join(audio_dir, file_base + ext))
    for path in candidates:
        if os.path.isfile(path):
            return path
    matches = glob.glob(os.path.join(audio_dir, file_base + ".*"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Audio test tidak ditemukan: file_id={file_id}")


def ids_sha(ids):
    return hashlib.sha256("\n".join(map(str, ids)).encode("utf-8")).hexdigest()


def read_fixed_wave(path):
    try:
        y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    except Exception as exc:
        raise RuntimeError(f"Gagal membaca audio: {path}") from exc
    if y.size == 0:
        raise ValueError(f"Audio kosong: {path}")
    if len(y) < NUM_SAMPLES:
        y = np.pad(y, (0, NUM_SAMPLES - len(y)))
    else:
        y = y[:NUM_SAMPLES]
    return np.asarray(y, dtype=np.float32)


def prepare_wave_cache(paths, ids, cache_path, manifest_path, split_name):
    expected_hash = ids_sha(ids)
    if os.path.isfile(cache_path) and os.path.isfile(manifest_path):
        try:
            with open(manifest_path, "r", encoding="utf-8") as f:
                manifest = json.load(f)
            arr = np.load(cache_path, mmap_mode="r")
            if (manifest.get("ids_sha256") == expected_hash and
                    manifest.get("sample_rate") == SAMPLE_RATE and
                    manifest.get("num_samples") == NUM_SAMPLES and
                    manifest.get("dtype") == "float32" and
                    arr.dtype == np.float32 and
                    arr.shape == (len(paths), NUM_SAMPLES)):
                print(f"Reusing cached {split_name} waveforms: {arr.shape}")
                return
        except Exception as exc:
            print(f"Cache {split_name} tidak valid, akan dibuat ulang: {exc}")

    tmp_path = cache_path + ".tmp.npy"
    if os.path.exists(tmp_path):
        os.remove(tmp_path)
    arr = np.lib.format.open_memmap(
        tmp_path, mode="w+", dtype=np.float32, shape=(len(paths), NUM_SAMPLES)
    )
    start = time.time()
    for i, path in enumerate(paths):
        arr[i] = read_fixed_wave(path)
        if (i + 1) % 200 == 0 or i + 1 == len(paths):
            print(f"  {split_name} audio cache: {i+1}/{len(paths)}")
    arr.flush()
    del arr
    os.replace(tmp_path, cache_path)
    manifest = {
        "ids_sha256": expected_hash,
        "sample_rate": SAMPLE_RATE,
        "num_samples": NUM_SAMPLES,
        "rows": len(paths),
        "dtype": "float32",
    }
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    print(f"Saved {split_name} waveform cache: {cache_path}; {time.time()-start:.1f}s")


class CachedWaveDataset(Dataset):
    def __init__(self, cache_path, indices, labels=None, augment=False):
        self.cache_path = cache_path
        self.audio = np.load(cache_path, mmap_mode="r")
        self.indices = np.asarray(indices, dtype=np.int64)
        self.labels = None if labels is None else np.asarray(labels, dtype=np.int64)
        self.augment = bool(augment)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        idx = int(self.indices[item])
        wave = np.asarray(self.audio[idx], dtype=np.float32).copy()
        wave = torch.from_numpy(wave)
        if self.augment:
            # Conservative augmentation to avoid damaging bird calls.
            gain = float(np.random.uniform(0.85, 1.15))
            wave = wave * gain
            if np.random.random() < 0.35:
                wave = wave + torch.randn_like(wave) * float(np.random.uniform(0.0002, 0.0015))
        if self.labels is None:
            return wave, idx
        return wave, int(self.labels[idx]), idx


class BEATsFineTune(nn.Module):
    def __init__(self, backbone, embed_dim, n_classes):
        super().__init__()
        self.backbone = backbone
        self.pool_norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Dropout(p=0.25),
            nn.Linear(embed_dim, n_classes),
        )

    def forward(self, waveform):
        features, _ = self.backbone.extract_features(waveform)
        if features.ndim != 3:
            raise RuntimeError(f"Expected BEATs sequence output (B,T,D), got {tuple(features.shape)}")
        pooled = features.mean(dim=1)
        return self.head(self.pool_norm(pooled))


def prepare_trainable_model(checkpoint_path):
    backbone = load_beats_backbone(checkpoint_path)
    for parameter in backbone.parameters():
        parameter.requires_grad = False

    layers = backbone.encoder.layers
    if len(layers) < UNFREEZE_LAST_N_LAYERS:
        raise RuntimeError(f"Jumlah layer BEATs terlalu sedikit: {len(layers)}")
    unfrozen_ids = list(range(len(layers) - UNFREEZE_LAST_N_LAYERS, len(layers)))
    for layer_id in unfrozen_ids:
        for parameter in layers[layer_id].parameters():
            parameter.requires_grad = True

    embed_dim = int(getattr(backbone.cfg, "encoder_embed_dim", 768))
    model = BEATsFineTune(backbone, embed_dim, len(EXPECTED_LABELS)).to(DEVICE)

    encoder_params = [p for p in model.backbone.parameters() if p.requires_grad]
    head_params = list(model.pool_norm.parameters()) + list(model.head.parameters())
    param_groups = [
        {"params": encoder_params, "lr": ENCODER_LR},
        {"params": head_params, "lr": HEAD_LR},
    ]
    optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)

    prefixes = [f"backbone.encoder.layers.{i}." for i in unfrozen_ids]
    prefixes += ["pool_norm.", "head."]
    print("Fine-tuning encoder layer indices (0-based):", unfrozen_ids)
    print("Trainable parameter count:", sum(p.numel() for p in model.parameters() if p.requires_grad))
    return model, optimizer, unfrozen_ids, prefixes


def scaler_factory(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def amp_context():
    if AMP_ENABLED:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    # nullcontext keeps one code path for CPU fallback.
    from contextlib import nullcontext
    return nullcontext()


def predict_loader(model, loader):
    model.eval()
    all_probs = []
    all_indices = []
    with torch.no_grad():
        for batch in loader:
            # Supports both (wave, index) test batches and (wave, label, index) validation batches.
            waves = batch[0]
            indices = batch[-1]
            with amp_context():
                logits = model(waves)
            probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
            all_probs.append(probs)
            all_indices.append(np.asarray(indices, dtype=np.int64))
    if not all_probs:
        return np.zeros((0, len(EXPECTED_LABELS)), np.float32), np.zeros(0, np.int64)
    return np.concatenate(all_probs).astype(np.float32), np.concatenate(all_indices)


def get_partial_state(model, prefixes):
    state = model.state_dict()
    return {
        name: value.detach().cpu().clone()
        for name, value in state.items()
        if any(name.startswith(prefix) for prefix in prefixes)
    }


def load_partial_state(model, state):
    missing, unexpected = model.load_state_dict(state, strict=False)
    if unexpected:
        raise RuntimeError(f"Unexpected partial checkpoint keys: {unexpected[:5]}")
    return missing


def train_one_fold(model, optimizer, prefixes, train_loader, val_loader, y, val_indices, fold, checkpoint_path):
    # Reuse completed fold checkpoints when the experiment version matches.
    if os.path.isfile(checkpoint_path):
        try:
            saved = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
            if saved.get("version") == EXPERIMENT_VERSION and saved.get("fold") == int(fold):
                load_partial_state(model, saved["state"])
                val_probs, pred_idx = predict_loader(model, val_loader)
                if not np.array_equal(pred_idx, val_indices):
                    order = np.argsort(pred_idx)
                    pred_idx, val_probs = pred_idx[order], val_probs[order]
                score = f1_score(y[val_indices], val_probs.argmax(1), average="macro")
                print(f"Reused completed fold {fold} checkpoint; current OOF fold F1={score:.5f}")
                return float(saved.get("best_val_f1", score)), int(saved.get("best_epoch", 0)), val_probs, pred_idx
        except Exception as exc:
            print(f"Existing fold checkpoint cannot be reused; retraining fold {fold}: {exc}")

    criterion_weights = None
    # Derive weights from this fold's train indices by reading label from dataset samples isn't cheap;
    # the caller attaches train class weights to criterion before this function via attribute.
    class_weights = train_loader.class_weights.to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=1, min_lr=1e-7
    )
    scaler = scaler_factory(AMP_ENABLED)
    best_score = -1.0
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        # Keep dropout disabled inside the pretrained backbone to stabilize the frozen early layers.
        # Gradients still flow through the two trainable Transformer blocks.
        model.backbone.eval()
        model.head.train()
        model.pool_norm.train()
        running_loss = 0.0
        seen = 0
        start = time.time()

        for waves, labels, _indices in train_loader:
            optimizer.zero_grad(set_to_none=True)
            with amp_context():
                logits = model(waves)
                loss = criterion(logits.float(), labels.to(DEVICE, dtype=torch.long))
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
            batch_n = int(labels.shape[0])
            running_loss += float(loss.detach().cpu()) * batch_n
            seen += batch_n

        # Predict validation group, labels/indices are returned by the val dataset.
        model.eval()
        val_prob_chunks, val_idx_chunks, val_label_chunks = [], [], []
        with torch.no_grad():
            for waves, labels, indices in val_loader:
                with amp_context():
                    logits = model(waves)
                val_prob_chunks.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
                val_idx_chunks.append(np.asarray(indices, dtype=np.int64))
                val_label_chunks.append(np.asarray(labels, dtype=np.int64))
        val_probs = np.concatenate(val_prob_chunks).astype(np.float32)
        val_pred_idx = np.concatenate(val_idx_chunks)
        val_true = np.concatenate(val_label_chunks)
        val_f1 = float(f1_score(val_true, val_probs.argmax(axis=1), average="macro"))
        val_acc = float(accuracy_score(val_true, val_probs.argmax(axis=1)))
        avg_loss = running_loss / max(seen, 1)
        scheduler.step(val_f1)
        lr_encoder = optimizer.param_groups[0]["lr"]
        lr_head = optimizer.param_groups[1]["lr"]
        elapsed = time.time() - start
        print(
            f"Fold {fold} | Epoch {epoch:02d}/{MAX_EPOCHS} | loss={avg_loss:.4f} "
            f"val_macroF1={val_f1:.5f} acc={val_acc:.5f} "
            f"lr_enc={lr_encoder:.2e} lr_head={lr_head:.2e} time={elapsed:.1f}s"
        )
        history.append({
            "fold": int(fold), "epoch": int(epoch), "train_loss": avg_loss,
            "val_macro_f1": val_f1, "val_accuracy": val_acc,
            "lr_encoder": lr_encoder, "lr_head": lr_head, "seconds": elapsed,
        })

        if val_f1 > best_score + 1e-5:
            best_score = val_f1
            best_epoch = epoch
            best_state = get_partial_state(model, prefixes)
            torch.save({
                "version": EXPERIMENT_VERSION,
                "fold": int(fold),
                "best_epoch": int(best_epoch),
                "best_val_f1": float(best_score),
                "state": best_state,
                "history": history,
                "unfreeze_last_n_layers": UNFREEZE_LAST_N_LAYERS,
            }, checkpoint_path)
            print(f"  -> saved best partial checkpoint (epoch={epoch}, F1={val_f1:.5f})")
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print("  Early stopping.")
            break

    if best_state is None:
        raise RuntimeError(f"Fold {fold} did not produce a checkpoint")
    load_partial_state(model, best_state)

    # Re-evaluate best checkpoint to ensure returned OOF probabilities match the saved weights.
    val_prob_chunks, val_idx_chunks = [], []
    model.eval()
    with torch.no_grad():
        for waves, _labels, indices in val_loader:
            with amp_context():
                logits = model(waves)
            val_prob_chunks.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
            val_idx_chunks.append(np.asarray(indices, dtype=np.int64))
    val_probs = np.concatenate(val_prob_chunks).astype(np.float32)
    val_pred_idx = np.concatenate(val_idx_chunks)
    order = np.argsort(val_pred_idx)
    val_pred_idx, val_probs = val_pred_idx[order], val_probs[order]
    if not np.array_equal(val_pred_idx, val_indices):
        raise RuntimeError(f"Fold {fold}: validation index order mismatch")
    return best_score, best_epoch, val_probs, val_pred_idx


def normalize_probs(probs):
    probs = np.asarray(probs, dtype=np.float64)
    probs = np.clip(probs, 1e-9, None)
    return probs / probs.sum(axis=1, keepdims=True)


def load_previous_oof_and_test(meta, sample):
    names = {
        "resnet": "oof_resnet50_ensemble_tta3_seed16.npy",
        "b2": "oof_efficientnet_b2_audio8s_tta3_seed16.npy",
        "convnext": "oof_convnext_tiny_audio8s_tta3_seed16_pretrained_v4_resize224.npy",
    }
    found = {key: find_first_file(value) for key, value in names.items()}
    missing = [key for key, path in found.items() if path is None]
    if missing:
        print("Previous OOF files absent; skipping blend comparison:", missing)
        return None, None

    probs = {key: np.asarray(np.load(path), dtype=np.float64) for key, path in found.items()}
    expected = (len(meta), len(EXPECTED_LABELS))
    for key, p in probs.items():
        if p.shape != expected:
            print(f"OOF shape mismatch for {key}: {p.shape} vs {expected}")
            return None, None
        probs[key] = normalize_probs(p)
    baseline_oof = normalize_probs(
        0.55 * probs["resnet"] + 0.05 * probs["b2"] + 0.40 * probs["convnext"]
    ).astype(np.float32)

    test_file = "test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy"
    test_path = find_first_file(test_file)
    baseline_test = None
    if test_path:
        arr = np.asarray(np.load(test_path), dtype=np.float64)
        if arr.shape == (len(sample), len(EXPECTED_LABELS)):
            baseline_test = normalize_probs(arr).astype(np.float32)
        else:
            print("Baseline test probability shape mismatch:", arr.shape)
    else:
        print("Baseline test probability artifact absent; OOF blend only.")
    return baseline_oof, baseline_test


def main():
    seed_everything()
    print("=" * 78)
    print("SCHEMATICS NPC DSC 2026 — BEATs Iter3+ AS2M Partial Fine-Tuning GroupCV")
    print("Device:", DEVICE)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    print(f"Audio: {DURATION_SECONDS}s at {SAMPLE_RATE} Hz; AMP={AMP_ENABLED}")
    print(f"Fine-tuning: last {UNFREEZE_LAST_N_LAYERS} Transformer layers + linear head")
    print("No submission will be created or sent.")
    print("=" * 78)

    if not os.path.isfile(TRAIN_META_PATH) or not os.path.isfile(SAMPLE_PATH):
        raise FileNotFoundError(f"Competition input missing. Attach: {ROOT}")
    meta = pd.read_csv(TRAIN_META_PATH).copy()
    sample = pd.read_csv(SAMPLE_PATH).copy()
    required = {"file_id", "audio_path", "label", "source_group"}
    if required - set(meta.columns):
        raise ValueError(f"Train metadata missing columns: {required - set(meta.columns)}")
    if "file_id" not in sample.columns:
        raise ValueError("sample_submission.csv lacks file_id")

    meta["file_id"] = meta["file_id"].astype(str)
    meta["label"] = meta["label"].astype(str)
    meta["source_group"] = meta["source_group"].astype(str)
    sample["file_id"] = sample["file_id"].astype(str)
    labels = sorted(meta["label"].unique().tolist())
    if labels != EXPECTED_LABELS:
        raise ValueError(f"Label mismatch: {labels}; expected {EXPECTED_LABELS}")
    label_to_idx = {label: i for i, label in enumerate(labels)}
    y = meta["label"].map(label_to_idx).to_numpy(dtype=np.int64)

    fold_path = find_first_file("bird_group_folds_seed16.csv")
    if fold_path:
        fold_df = pd.read_csv(fold_path)
        if not {"file_id", "fold"}.issubset(fold_df.columns):
            raise ValueError(f"Fold file missing file_id/fold: {fold_path}")
        fold_df["file_id"] = fold_df["file_id"].astype(str)
        fold_map = fold_df.drop_duplicates("file_id").set_index("file_id")["fold"]
        mapped = meta["file_id"].map(fold_map)
        if mapped.isna().any() or len(fold_df) != len(meta):
            raise ValueError("Saved seed16 folds do not match train metadata")
        meta["fold"] = mapped.astype(int)
        print("Using saved group folds:", fold_path)
    else:
        print("Saved folds unavailable; generating StratifiedGroupKFold(seed=16)")
        splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
        meta["fold"] = -1
        for fold, (_, val_idx) in enumerate(splitter.split(meta, y, groups=meta["source_group"])):
            meta.loc[val_idx, "fold"] = fold
        meta[["file_id", "label", "source_group", "fold"]].to_csv(
            os.path.join(OUT, "beats_finetune_generated_group_folds_seed16.csv"), index=False
        )
    if sorted(meta["fold"].unique().tolist()) != list(range(N_FOLDS)):
        raise ValueError(f"Expected fold IDs 0..3; got {sorted(meta['fold'].unique().tolist())}")

    print("Train rows:", len(meta), "| test rows:", len(sample))
    print("Class counts:", meta["label"].value_counts().to_dict())
    print("Fold counts:", meta["fold"].value_counts().sort_index().to_dict())

    # Resolve audio only once, then cache fixed waveform tensors for repeated folds/epochs.
    print("\nResolving audio paths...")
    train_paths = [resolve_train_audio(row) for _, row in meta.iterrows()]
    test_paths = [resolve_test_audio(fid) for fid in sample["file_id"].tolist()]
    print("Resolved audio paths:", len(train_paths), "train |", len(test_paths), "test")
    prepare_wave_cache(train_paths, meta["file_id"].tolist(), TRAIN_WAVE_CACHE, TRAIN_WAVE_MANIFEST, "TRAIN")
    prepare_wave_cache(test_paths, sample["file_id"].tolist(), TEST_WAVE_CACHE, TEST_WAVE_MANIFEST, "TEST")

    checkpoint_path = ensure_checkpoint()
    oof = np.zeros((len(meta), len(labels)), dtype=np.float32)
    oof_filled = np.zeros(len(meta), dtype=bool)
    test_probs_sum = np.zeros((len(sample), len(labels)), dtype=np.float64)
    fold_metrics = []
    test_indices = np.arange(len(sample), dtype=np.int64)
    test_dataset = CachedWaveDataset(TEST_WAVE_CACHE, test_indices, labels=None, augment=False)
    test_loader = DataLoader(
        test_dataset, batch_size=VAL_BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=False,
        persistent_workers=(NUM_WORKERS > 0),
    )

    for fold in range(N_FOLDS):
        print("\n" + "-" * 72)
        print(f"BEATs fine-tune fold {fold}/{N_FOLDS-1}")
        val_idx = np.where(meta["fold"].to_numpy() == fold)[0]
        train_idx = np.where(meta["fold"].to_numpy() != fold)[0]
        counts = np.bincount(y[train_idx], minlength=len(labels)).astype(np.float32)
        class_weights = np.sqrt(counts.max() / np.maximum(counts, 1.0))
        class_weights = class_weights / class_weights.mean()
        print("Fold train/val rows:", len(train_idx), len(val_idx), "| class weights:", np.round(class_weights, 3).tolist())

        train_dataset = CachedWaveDataset(TRAIN_WAVE_CACHE, train_idx, labels=y, augment=True)
        val_dataset = CachedWaveDataset(TRAIN_WAVE_CACHE, val_idx, labels=y, augment=False)
        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=False,
            persistent_workers=(NUM_WORKERS > 0), drop_last=False,
        )
        # Dataset/loader attribute used by train_one_fold for fold-specific weighted loss.
        train_loader.class_weights = torch.tensor(class_weights, dtype=torch.float32)
        val_loader = DataLoader(
            val_dataset, batch_size=VAL_BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=False,
            persistent_workers=(NUM_WORKERS > 0), drop_last=False,
        )

        # Load a fresh copy of pretrained weights for every fold; fold training never shares updates.
        model, optimizer, unfrozen_ids, prefixes = prepare_trainable_model(checkpoint_path)
        fold_checkpoint_path = os.path.join(FOLD_CKPT_DIR, f"fold_{fold}_{EXPERIMENT_VERSION}.pt")
        start = time.time()
        best_score, best_epoch, val_probs, returned_idx = train_one_fold(
            model, optimizer, prefixes, train_loader, val_loader, y, val_idx,
            fold, fold_checkpoint_path,
        )
        if not np.array_equal(returned_idx, val_idx):
            order = np.argsort(returned_idx)
            returned_idx, val_probs = returned_idx[order], val_probs[order]
        oof[returned_idx] = val_probs
        oof_filled[returned_idx] = True

        test_probs, predicted_test_idx = predict_loader(model, test_loader)
        test_order = np.argsort(predicted_test_idx)
        test_probs = test_probs[test_order]
        predicted_test_idx = predicted_test_idx[test_order]
        if not np.array_equal(predicted_test_idx, test_indices):
            raise RuntimeError("Test prediction indices do not align to sample_submission")
        test_probs_sum += test_probs

        fold_true = y[val_idx]
        fold_pred = val_probs.argmax(axis=1)
        fold_f1 = float(f1_score(fold_true, fold_pred, average="macro"))
        fold_acc = float(accuracy_score(fold_true, fold_pred))
        fold_metrics.append({
            "fold": fold, "train_rows": len(train_idx), "val_rows": len(val_idx),
            "best_epoch": best_epoch, "best_val_macro_f1": best_score,
            "recomputed_val_macro_f1": fold_f1, "val_accuracy": fold_acc,
            "elapsed_seconds": round(time.time() - start, 1),
        })
        print(f"Fold {fold} best epoch={best_epoch}; OOF fold F1={fold_f1:.5f}; test probs collected")

        del model, optimizer, train_loader, val_loader, train_dataset, val_dataset
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if not oof_filled.all():
        missing = np.where(~oof_filled)[0][:10].tolist()
        raise RuntimeError(f"OOF is incomplete; missing row indices like {missing}")

    oof = normalize_probs(oof).astype(np.float32)
    test_probs = normalize_probs(test_probs_sum / N_FOLDS).astype(np.float32)
    np.save(OOF_PATH, oof)
    np.save(TEST_PROBS_PATH, test_probs)
    pd.DataFrame(fold_metrics).to_csv(FOLD_METRICS_PATH, index=False)

    overall_f1 = float(f1_score(y, oof.argmax(axis=1), average="macro"))
    overall_acc = float(accuracy_score(y, oof.argmax(axis=1)))
    print("\n" + "=" * 78)
    print("BEATs PARTIAL FINE-TUNING OOF RESULTS")
    print("=" * 78)
    print(f"Overall OOF Macro F1: {overall_f1:.5f}")
    print(f"Overall OOF Accuracy: {overall_acc:.5f}")
    print(pd.DataFrame(fold_metrics).to_string(index=False))
    print(classification_report(y, oof.argmax(axis=1), target_names=labels, digits=4, zero_division=0))
    print("OOF probabilities:", OOF_PATH)
    print("Test probabilities:", TEST_PROBS_PATH)

    baseline_oof, baseline_test = load_previous_oof_and_test(meta, sample)
    blend_rows = []
    selected_weight = 0.0
    if baseline_oof is not None:
        baseline_f1 = float(f1_score(y, baseline_oof.argmax(axis=1), average="macro"))
        print(f"Triple baseline OOF Macro F1: {baseline_f1:.5f}")
        for weight in np.round(np.arange(0.0, 0.401, 0.05), 2):
            blend_oof = normalize_probs((1.0-weight) * baseline_oof + weight * oof)
            blend_rows.append({
                "beats_finetune_weight": float(weight),
                "triple_baseline_weight": float(1.0-weight),
                "oof_macro_f1": float(f1_score(y, blend_oof.argmax(axis=1), average="macro")),
                "oof_accuracy": float(accuracy_score(y, blend_oof.argmax(axis=1))),
            })
        blend_df = pd.DataFrame(blend_rows).sort_values(
            ["oof_macro_f1", "oof_accuracy"], ascending=False
        ).reset_index(drop=True)
        blend_df.to_csv(BLEND_RESULTS_PATH, index=False)
        selected_weight = float(blend_df.iloc[0]["beats_finetune_weight"])
        print("\nBEATs fine-tune + triple baseline OOF blend search (exploratory only):")
        print(blend_df.to_string(index=False))
        print(f"OOF-selected BEATs fine-tune weight: {selected_weight:.2f}")
        if baseline_test is not None:
            blended_test = normalize_probs((1.0-selected_weight) * baseline_test + selected_weight * test_probs).astype(np.float32)
            np.save(BEST_BLEND_TEST_PATH, blended_test)
            print("Saved exploratory blended test probabilities:", BEST_BLEND_TEST_PATH)
    else:
        print("Baseline OOF/test artifacts missing; skipping ensemble blend search.")

    metrics = {
        "experiment_version": EXPERIMENT_VERSION,
        "backbone_checkpoint": CHECKPOINT_FILENAME,
        "finetune_last_n_encoder_layers": UNFREEZE_LAST_N_LAYERS,
        "epochs_max": MAX_EPOCHS,
        "batch_size": BATCH_SIZE,
        "encoder_lr": ENCODER_LR,
        "head_lr": HEAD_LR,
        "oof_macro_f1": overall_f1,
        "oof_accuracy": overall_acc,
        "fold_metrics": fold_metrics,
        "oof_best_blend_weight": selected_weight,
        "outputs": {
            "oof_probabilities": OOF_PATH,
            "test_probabilities": TEST_PROBS_PATH,
            "fold_metrics": FOLD_METRICS_PATH,
            "blend_search": BLEND_RESULTS_PATH,
            "best_oof_blend_test_probabilities": BEST_BLEND_TEST_PATH,
        },
        "submission_created": False,
    }
    with open(METRICS_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    print("\n" + "=" * 78)
    print("BEATs PARTIAL FINE-TUNING COMPLETE — NO SUBMISSION CREATED")
    print("Metrics JSON:", METRICS_JSON_PATH)
    print("=" * 78)


if __name__ == "__main__":
    main()


SCHEMATICS NPC DSC 2026 — BEATs Iter3+ AS2M Partial Fine-Tuning GroupCV
Device: cuda
GPU: Tesla T4
Audio: 8.0s at 16000 Hz; AMP=True
Fine-tuning: last 2 Transformer layers + linear head
No submission will be created or sent.
Using saved group folds: /kaggle/input/notebooks/radithiafatir/testttrisan/bird_group_folds_seed16.csv
Train rows: 1826 | test rows: 814
Class counts: {'Bondol Jawa': 1124, 'Kutilang': 619, 'Cendrawasih': 83}
Fold counts: {0: 284, 1: 495, 2: 553, 3: 494}

Resolving audio paths...
Resolved audio paths: 1826 train | 814 test
  TRAIN audio cache: 200/1826
  TRAIN audio cache: 400/1826
  TRAIN audio cache: 600/1826
  TRAIN audio cache: 800/1826
  TRAIN audio cache: 1000/1826
  TRAIN audio cache: 1200/1826
  TRAIN audio cache: 1400/1826
  TRAIN audio cache: 1600/1826
  TRAIN audio cache: 1800/1826
  TRAIN audio cache: 1826/1826
Saved TRAIN waveform cache: /kaggle/working/beats_finetune_train_waveforms_8s_16k.npy; 61.0s
  TEST audio cache: 200/814
  TEST audio cache: 400

In [13]:
import os
import numpy as np
import pandas as pd

ROOT = "/kaggle/input/competitions/schematics-2026"
SRC = "/kaggle/input/notebooks/radithiafatir/testttrisan"
OUT = "/kaggle/working"

N_UNKNOWN = 245
W_CONFIDENCE = 0.60
W_DISAGREEMENT = 0.40

# Blend 90% ensemble utama + 10% BEATs fine-tuning
blend_path = os.path.join(
    OUT, "test_beats_finetune_top2_best_oof_blend_probabilities.npy"
)

# Probabilitas model lama untuk menghitung disagreement dengan
# metode yang sama seperti kandidat Triple V4 sebelumnya.
crop_path = os.path.join(SRC, "tta_test_crop_probabilities.npz")
b2_path = os.path.join(SRC, "test_effnet_b2_audio8s_tta3.npy")
convnext_path = os.path.join(
    SRC, "test_convnext_tiny_pretrained_v4_resize224_tta3.npy"
)
sample_path = os.path.join(ROOT, "sample_submission.csv")

for path in [blend_path, crop_path, b2_path, convnext_path, sample_path]:
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"File tidak ditemukan: {path}\n"
            "Pastikan cell BEATs sebelumnya selesai dan input testttrisan terpasang."
        )

sample = pd.read_csv(sample_path)
n = len(sample)

P_BLEND = np.asarray(np.load(blend_path), dtype=np.float64)
P_B2 = np.asarray(np.load(b2_path), dtype=np.float64)
P_CONVNEXT = np.asarray(np.load(convnext_path), dtype=np.float64)

with np.load(crop_path) as z:
    P_R5 = np.asarray(z["test_r5"][:, [0, 2, 4], :], dtype=np.float64).mean(axis=1)
    P_R8 = np.asarray(z["test_r8"][:, [0, 2, 4], :], dtype=np.float64).mean(axis=1)

P_RESNET = 0.25 * P_R5 + 0.75 * P_R8

# Validasi dan normalisasi.
for name, p in {
    "BEATs blend": P_BLEND,
    "ResNet50": P_RESNET,
    "EfficientNet-B2": P_B2,
    "ConvNeXt-v4": P_CONVNEXT,
}.items():
    if p.shape != (n, 3):
        raise ValueError(f"{name}: shape {p.shape}, seharusnya {(n, 3)}")
    if not np.isfinite(p).all() or np.any(p < 0):
        raise ValueError(f"Probabilitas {name} mengandung nilai tidak valid.")

P_BLEND /= np.clip(P_BLEND.sum(axis=1, keepdims=True), 1e-12, None)

for name, p in {
    "ResNet50": P_RESNET,
    "EfficientNet-B2": P_B2,
    "ConvNeXt-v4": P_CONVNEXT,
}.items():
    p /= np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)

# Pertahankan definisi disagreement dari tiga model Triple V4.
models = [P_RESNET, P_B2, P_CONVNEXT]
pairwise_tv = []

for i in range(len(models)):
    for j in range(i + 1, len(models)):
        pairwise_tv.append(0.5 * np.abs(models[i] - models[j]).sum(axis=1))

disagreement = np.mean(pairwise_tv, axis=0)
confidence = P_BLEND.max(axis=1)

confidence_rank = pd.Series(confidence).rank(pct=True).to_numpy()
disagreement_rank = pd.Series(-disagreement).rank(pct=True).to_numpy()

unknown_score = (
    W_CONFIDENCE * confidence_rank
    + W_DISAGREEMENT * disagreement_rank
)

if not 0 < N_UNKNOWN < n:
    raise ValueError(f"Jumlah Unknown tidak valid: {N_UNKNOWN}")

unknown_idx = np.argsort(unknown_score)[:N_UNKNOWN]

labels = np.array(["Bondol Jawa", "Cendrawasih", "Kutilang"])
prediction = labels[P_BLEND.argmax(axis=1)]
prediction[unknown_idx] = "Unknown"

target_cols = [c for c in sample.columns if c != "file_id"]
if len(target_cols) != 1:
    raise ValueError(f"Kolom target tidak dikenali: {sample.columns.tolist()}")

submission = sample.copy()
submission[target_cols[0]] = prediction

output_path = os.path.join(
    OUT, "submission_beats_finetune_w10_conf60_dis40_u245.csv"
)
submission.to_csv(output_path, index=False)

# Verifikasi CSV, urutan ID, label, dan jumlah Unknown.
check = pd.read_csv(output_path)
assert len(check) == len(sample)
assert check["file_id"].equals(sample["file_id"])
assert check[target_cols[0]].notna().all()
assert (check[target_cols[0]] == "Unknown").sum() == N_UNKNOWN
assert set(check[target_cols[0]].unique()).issubset(
    {"Bondol Jawa", "Cendrawasih", "Kutilang", "Unknown"}
)

print("=" * 65)
print("BEATs FINE-TUNING 10% — KANDIDAT BARU")
print("=" * 65)
print("Blend: 90% Triple V4 + 10% BEATs fine-tuning")
print("Unknown:", int((prediction == "Unknown").sum()))
print("\nDistribusi label:")
print(pd.Series(prediction).value_counts().to_string())
print("\nCSV:", output_path)
print("Format valid. Belum disubmit.")

BEATs FINE-TUNING 10% — KANDIDAT BARU
Blend: 90% Triple V4 + 10% BEATs fine-tuning
Unknown: 245

Distribusi label:
Unknown        245
Kutilang       237
Bondol Jawa    214
Cendrawasih    118

CSV: /kaggle/working/submission_beats_finetune_w10_conf60_dis40_u245.csv
Format valid. Belum disubmit.


In [14]:
# -*- coding: utf-8 -*-
"""
SCHEMATICS NPC DSC 2026 — Triple V4 OOD diagnostics for NON-BIRD / NOISE Unknowns

Purpose:
- Preserve Triple V4 as the known-class classifier.
- Diagnose test novelty using frozen public BEATs embeddings + acoustic features.
- Compare OOF-known vs synthetic non-bird/noise proxies (noise types are only proxies).
- Rank OOD signals and inspect fixed-count sensitivity WITHOUT creating a submission.

Important limitations:
- The competition's true Unknown examples have no labels here.
- Synthetic noises are not representative of every real non-bird recording.
- Saved model artifacts are post-softmax probabilities, so raw-logit energy is NOT computed.
- OOF-known scores only validate behavior on known bird audio; synthetic proxy metrics are exploratory.
"""
import os
import sys
import glob
import time
import types
import random
import warnings
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore", category=FutureWarning)

# -----------------------------------------------------------------------------
# Config and paths
# -----------------------------------------------------------------------------
SEED = 16
N_FOLDS = 4
SR = 16000
DURATION = 8.0
NUM_SAMPLES = int(SR * DURATION)
BATCH_SIZE = 8
NUM_WORKERS = 2
USE_AMP = True
N_SYNTHETIC_NOISES = 256
KNN_K = 5
N_UNKNOWN_REFERENCE = 245

ROOT = "/kaggle/input/competitions/schematics-2026"
SRC = "/kaggle/input/notebooks/radithiafatir/testttrisan"
OUT = "/kaggle/working"
BEATS_CODE_DIR = os.path.join(OUT, "beats_official_code")
CHECKPOINT_NAME = "BEATs_iter3_plus_AS2M.pt"
CHECKPOINT_REPO_ID = "lpepino/beats_ckpts"

LABELS = ["Bondol Jawa", "Cendrawasih", "Kutilang"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUT, exist_ok=True)
os.makedirs(BEATS_CODE_DIR, exist_ok=True)

TRAIN_META_PATH = os.path.join(ROOT, "train", "metadata.csv")
SAMPLE_PATH = os.path.join(ROOT, "sample_submission.csv")
FOLD_PATH = os.path.join(SRC, "bird_group_folds_seed16.csv")

ARTIFACTS = {
    "oof_resnet": "oof_resnet50_ensemble_tta3_seed16.npy",
    "oof_b2": "oof_efficientnet_b2_audio8s_tta3_seed16.npy",
    "oof_convnext": "oof_convnext_tiny_audio8s_tta3_seed16_pretrained_v4_resize224.npy",
    "test_resnet_crops": "tta_test_crop_probabilities.npz",
    "test_b2": "test_effnet_b2_audio8s_tta3.npy",
    "test_convnext": "test_convnext_tiny_pretrained_v4_resize224_tta3.npy",
}
TRAIN_EMB_NAME = "beats_iter3plus_as2m_8s_train_embeddings.npy"
TEST_EMB_NAME = "beats_iter3plus_as2m_8s_test_embeddings.npy"
SYNTH_EMB_NAME = "beats_nonbird_synthetic_noise_embeddings.npy"


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def find_file(filename, dirs=None):
    dirs = dirs or [OUT, SRC, "/kaggle/input"]
    # Exact direct paths first.
    for d in dirs:
        p = os.path.join(d, filename)
        if os.path.isfile(p):
            return p
    # Recursive fallback for notebook outputs attached under a different input name.
    for d in dirs:
        if os.path.isdir(d):
            hits = glob.glob(os.path.join(d, "**", filename), recursive=True)
            if hits:
                return hits[0]
    return None


def download_file(url, destination):
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    tmp = destination + ".part"
    try:
        urllib.request.urlretrieve(url, tmp)
        if os.path.getsize(tmp) < 1000:
            raise RuntimeError(f"Unduhan terlalu kecil: {os.path.getsize(tmp)} bytes")
        os.replace(tmp, destination)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)


def ensure_beats_source():
    names = ["BEATs.py", "backbone.py", "modules.py"]
    for folder in [BEATS_CODE_DIR] + [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/BEATs.py", recursive=True)]:
        if all(os.path.isfile(os.path.join(folder, x)) for x in names):
            if folder not in sys.path:
                sys.path.insert(0, folder)
            return folder
    base = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"
    for name in names:
        dest = os.path.join(BEATS_CODE_DIR, name)
        if not os.path.isfile(dest):
            try:
                download_file(base + name, dest)
            except Exception as exc:
                raise RuntimeError(
                    "Kode resmi BEATs tidak tersedia. Aktifkan Internet atau lampirkan BEATs.py, "
                    "backbone.py, dan modules.py sebagai input Kaggle."
                ) from exc
    if BEATS_CODE_DIR not in sys.path:
        sys.path.insert(0, BEATS_CODE_DIR)
    return BEATS_CODE_DIR


def ensure_checkpoint():
    found = find_file(CHECKPOINT_NAME, [OUT, "/kaggle/input"])
    if found and os.path.getsize(found) > 100_000_000:
        return found
    dest = os.path.join(OUT, "pretrained", CHECKPOINT_NAME)
    if os.path.isfile(dest) and os.path.getsize(dest) > 100_000_000:
        return dest
    try:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(repo_id=CHECKPOINT_REPO_ID, filename=CHECKPOINT_NAME, local_dir=os.path.dirname(dest))
        if os.path.isfile(p) and os.path.getsize(p) > 100_000_000:
            return p
    except Exception as exc:
        print("HF download failed, trying direct public checkpoint URL:", repr(exc))
    url = f"https://huggingface.co/{CHECKPOINT_REPO_ID}/resolve/main/{CHECKPOINT_NAME}"
    try:
        download_file(url, dest)
        return dest
    except Exception as exc:
        raise RuntimeError(
            "Checkpoint BEATs tidak ditemukan. Aktifkan Internet atau tambahkan "
            "BEATs_iter3_plus_AS2M.pt sebagai Kaggle Input."
        ) from exc


def load_beats(checkpoint_path):
    ensure_beats_source()
    try:
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location="cpu")
    from BEATs import BEATs, BEATsConfig
    cfg = BEATsConfig(checkpoint["cfg"])
    model = BEATs(cfg)
    model.load_state_dict(checkpoint["model"], strict=True)
    model.eval().to(DEVICE)
    orig_preprocess = model.preprocess

    def preprocess_on_model_device(self, source, fbank_mean=15.41663, fbank_std=6.55582):
        fbanks = orig_preprocess(source.detach().to("cpu"), fbank_mean=fbank_mean, fbank_std=fbank_std)
        return fbanks.to(DEVICE, non_blocking=True)

    model.preprocess = types.MethodType(preprocess_on_model_device, model)
    if getattr(model, "predictor", None) is not None:
        model.predictor = None
    print("BEATs encoder dim:", getattr(cfg, "encoder_embed_dim", "unknown"),
          "| layers:", getattr(cfg, "encoder_layers", "unknown"))
    return model


def base_name(path):
    return os.path.splitext(os.path.basename(str(path)))[0]


def resolve_audio_paths(meta, is_test=False):
    audio_dir = os.path.join(ROOT, "test", "audio") if is_test else os.path.join(ROOT, "train", "audio")
    out = []
    for _, row in meta.iterrows():
        raw = str(row.get("audio_path", "")) if not is_test else ""
        fid = str(row["file_id"])
        fid_base = base_name(fid)
        cands = []
        if raw:
            if os.path.isabs(raw):
                cands.append(raw)
            else:
                cands.extend([os.path.join(ROOT, raw), os.path.join(ROOT, "train", raw)])
                cands.append(os.path.join(audio_dir, os.path.basename(raw)))
        cands.append(os.path.join(audio_dir, fid))
        for ext in [".flac", ".wav", ".mp3", ".ogg", ".m4a"]:
            cands.append(os.path.join(audio_dir, fid_base + ext))
        hit = next((p for p in cands if p and os.path.isfile(p)), None)
        if hit is None:
            matches = glob.glob(os.path.join(audio_dir, fid_base + ".*"))
            if matches:
                hit = matches[0]
        if hit is None:
            raise FileNotFoundError(f"Audio tidak ditemukan: file_id={fid}; audio_dir={audio_dir}; audio_path={raw}")
        out.append(hit)
    return out


def load_fixed_wave(path):
    y, _ = librosa.load(path, sr=SR, mono=True, duration=DURATION)
    if y.size == 0:
        raise ValueError(f"Audio kosong: {path}")
    if len(y) < NUM_SAMPLES:
        y = np.pad(y, (0, NUM_SAMPLES - len(y)))
    else:
        y = y[:NUM_SAMPLES]
    return np.asarray(y, dtype=np.float32)


def acoustic_features(wave):
    """Lightweight audio descriptors; diagnostic features, not a bird detector."""
    x = np.asarray(wave, dtype=np.float32)
    rms = float(np.sqrt(np.mean(x * x) + 1e-12))
    rms_db = float(20 * np.log10(rms + 1e-10))
    zcr = float(np.mean(np.abs(np.diff(np.signbit(x).astype(np.int8))))) if len(x) > 1 else 0.0

    n_fft, hop = 2048, 512
    if len(x) < n_fft:
        x = np.pad(x, (0, n_fft - len(x)))
    frames = np.lib.stride_tricks.sliding_window_view(x, n_fft)[::hop]
    win = np.hanning(n_fft).astype(np.float32)
    power = np.abs(np.fft.rfft(frames * win[None, :], axis=1)) ** 2 + 1e-12
    freqs = np.fft.rfftfreq(n_fft, d=1.0 / SR)
    frame_sum = power.sum(axis=1) + 1e-12
    p_norm = power / frame_sum[:, None]
    flatness = np.exp(np.mean(np.log(power), axis=1)) / (np.mean(power, axis=1) + 1e-12)
    entropy = -np.sum(p_norm * np.log(p_norm + 1e-12), axis=1) / np.log(power.shape[1])
    centroid = (power * freqs[None, :]).sum(axis=1) / frame_sum
    low = power[:, freqs < 500].sum(axis=1) / frame_sum
    high = power[:, freqs >= 2500].sum(axis=1) / frame_sum
    return np.array([
        rms, rms_db, zcr,
        float(np.median(flatness)), float(np.mean(flatness)),
        float(np.mean(entropy)), float(np.mean(centroid)),
        float(np.mean(low)), float(np.mean(high)),
        float(np.percentile(flatness, 90)),
    ], dtype=np.float32)

ACOUSTIC_NAMES = [
    "rms", "rms_db", "zero_crossing_rate", "spectral_flatness_median",
    "spectral_flatness_mean", "spectral_entropy_mean", "spectral_centroid_hz",
    "low_band_fraction_lt500hz", "high_band_fraction_ge2500hz", "spectral_flatness_p90",
]


class PathWaveDataset(Dataset):
    def __init__(self, paths): self.paths = list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        wave = load_fixed_wave(self.paths[idx])
        feat = acoustic_features(wave)
        return torch.from_numpy(wave), idx, torch.from_numpy(feat)


def extract_or_load_embeddings(model, paths, emb_path, split_name):
    # Look for an existing cache first, including notebook outputs attached as input.
    cache = find_file(os.path.basename(emb_path), [OUT, SRC, "/kaggle/input"])
    cached = None
    if cache:
        try:
            arr = np.load(cache, mmap_mode="r")
            if arr.ndim == 2 and arr.shape[0] == len(paths) and arr.shape[1] >= 256:
                cached = np.asarray(arr, dtype=np.float32)
                print(f"Reusing {split_name} BEATs embeddings {cached.shape}: {cache}")
        except Exception:
            cached = None

    dataset = PathWaveDataset(paths)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=False, persistent_workers=(NUM_WORKERS > 0))
    feats_all, emb_chunks = [], []
    model.eval()
    start = time.time()
    with torch.inference_mode():
        for batch_idx, (waves, indices, feats) in enumerate(loader):
            feats_all.append(feats.numpy().astype(np.float32))
            if cached is None:
                if DEVICE.type == "cuda" and USE_AMP:
                    with torch.autocast(device_type="cuda", dtype=torch.float16):
                        features, _ = model.extract_features(waves.float())
                else:
                    features, _ = model.extract_features(waves.float())
                if features.ndim != 3:
                    raise RuntimeError(f"Unexpected BEATs features shape: {tuple(features.shape)}")
                emb_chunks.append(features.float().mean(dim=1).cpu().numpy().astype(np.float32))
            done = min((batch_idx + 1) * BATCH_SIZE, len(dataset))
            if done % 200 < BATCH_SIZE or done == len(dataset):
                print(f"  {split_name}: {done}/{len(dataset)}")
    acoustics = np.concatenate(feats_all, axis=0)
    if cached is None:
        embeddings = np.concatenate(emb_chunks, axis=0)
        np.save(emb_path, embeddings)
        print(f"Saved {split_name} embeddings {embeddings.shape}: {emb_path}; {time.time()-start:.1f}s")
    else:
        embeddings = cached
        # Copy to canonical path if cached from an input, for reproducibility.
        if os.path.abspath(cache) != os.path.abspath(emb_path):
            np.save(emb_path, embeddings)
    return embeddings, acoustics


def make_synthetic_noise(n, train_rms, seed=SEED):
    """Artificial noise-only proxies; intentionally diverse but not equivalent to real unknowns."""
    rng = np.random.default_rng(seed)
    noises = np.zeros((n, NUM_SAMPLES), dtype=np.float32)
    rms_targets = np.quantile(np.asarray(train_rms), [0.10, 0.35, 0.60, 0.85])
    t = np.arange(NUM_SAMPLES, dtype=np.float64) / SR
    freqs = np.fft.rfftfreq(NUM_SAMPLES, 1.0 / SR)
    for i in range(n):
        kind = i % 4
        white = rng.normal(size=NUM_SAMPLES)
        if kind == 0:  # white noise
            x = white
        elif kind == 1:  # pink-ish noise (1/sqrt(f) amplitude)
            spec = np.fft.rfft(white)
            weight = 1.0 / np.sqrt(np.maximum(freqs, 1.0))
            weight[0] = 0.0
            x = np.fft.irfft(spec * weight, n=NUM_SAMPLES)
        elif kind == 2:  # low-pass / brown-ish noise
            spec = np.fft.rfft(white)
            weight = 1.0 / np.maximum(freqs, 20.0)
            weight[0] = 0.0
            x = np.fft.irfft(spec * weight, n=NUM_SAMPLES)
        else:  # mechanical-hum-like tones with broadband bed
            x = 0.45 * white
            for f in rng.choice([50, 60, 100, 120, 180, 240, 400, 800], size=3, replace=False):
                x += rng.uniform(0.1, 0.6) * np.sin(2 * np.pi * f * t + rng.uniform(0, 2*np.pi))
        x = x.astype(np.float32)
        x /= np.sqrt(np.mean(x * x) + 1e-12)
        target = max(float(rng.choice(rms_targets)), 1e-5)
        x *= target
        # Mild random gain variation around the chosen real-audio RMS quantile.
        x *= float(rng.uniform(0.65, 1.35))
        noises[i] = x
    return noises


def embed_wave_array(model, waves, batch_size=BATCH_SIZE):
    out = []
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(waves), batch_size):
            batch = torch.as_tensor(waves[start:start+batch_size], dtype=torch.float32)
            if DEVICE.type == "cuda" and USE_AMP:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    features, _ = model.extract_features(batch)
            else:
                features, _ = model.extract_features(batch)
            out.append(features.float().mean(dim=1).cpu().numpy().astype(np.float32))
    return np.concatenate(out, axis=0)


def normalize_probs(p):
    p = np.asarray(p, dtype=np.float64)
    if p.ndim != 2 or not np.isfinite(p).all() or np.any(p < 0):
        raise ValueError(f"Invalid probability array: shape={p.shape}")
    p = np.clip(p, 1e-12, None)
    return p / p.sum(axis=1, keepdims=True)


def entropy_rows(p):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-12, 1.0)
    return -np.sum(p * np.log(p), axis=1) / np.log(p.shape[1])


def pairwise_tv_mean(prob_list):
    vals = []
    for i in range(len(prob_list)):
        for j in range(i + 1, len(prob_list)):
            vals.append(0.5 * np.abs(prob_list[i] - prob_list[j]).sum(axis=1))
    return np.mean(vals, axis=0)


def rank_against(reference, values):
    reference = np.sort(np.asarray(reference, dtype=np.float64))
    values = np.asarray(values, dtype=np.float64)
    return np.searchsorted(reference, values, side="right") / max(len(reference), 1)


def fit_fold_embedding_scores(ref_embeddings, ref_y, query_embeddings):
    """Returns three novelty scores fitted only on known reference rows."""
    ref = np.asarray(ref_embeddings, dtype=np.float32)
    qry = np.asarray(query_embeddings, dtype=np.float32)
    ref_n = normalize(ref, norm="l2")
    qry_n = normalize(qry, norm="l2")
    classes = np.unique(ref_y)
    centers = []
    for cls in classes:
        c = ref_n[ref_y == cls].mean(axis=0)
        c /= max(np.linalg.norm(c), 1e-12)
        centers.append(c)
    centers = np.stack(centers, axis=0)
    proto_score = 1.0 - (qry_n @ centers.T).max(axis=1)

    nn = NearestNeighbors(n_neighbors=min(KNN_K, len(ref)), metric="cosine", algorithm="brute", n_jobs=-1)
    nn.fit(ref_n)
    knn_score = nn.kneighbors(qry_n, return_distance=True)[0].mean(axis=1)

    # Diagonal pooled within-class Mahalanobis approximation (regularized), explicitly not full covariance.
    class_centers = {cls: ref[ref_y == cls].mean(axis=0) for cls in classes}
    residuals = np.stack([ref[i] - class_centers[ref_y[i]] for i in range(len(ref))], axis=0)
    var = residuals.var(axis=0) + 1e-5
    floor = float(np.median(var)) * 0.05 + 1e-8
    var = np.maximum(var, floor)
    md_scores = []
    for cls in classes:
        d = qry - class_centers[cls][None, :]
        md = np.sqrt(np.maximum(np.mean((d * d) / var[None, :], axis=1), 0.0))
        md_scores.append(md)
    mahal_score = np.min(np.stack(md_scores, axis=1), axis=1)
    return proto_score.astype(np.float32), knn_score.astype(np.float32), mahal_score.astype(np.float32)


def main():
    seed_everything()
    print("=" * 78)
    print("SCHEMATICS NPC DSC 2026 — Triple V4 NON-BIRD / NOISE OOD DIAGNOSTICS")
    print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
    print("Unknown definition: non-bird, noise, or audio without bird sound")
    print("No submission will be created or sent.")
    print("=" * 78)

    for path in [TRAIN_META_PATH, SAMPLE_PATH, FOLD_PATH]:
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Required input missing: {path}")

    train_meta = pd.read_csv(TRAIN_META_PATH)
    sample = pd.read_csv(SAMPLE_PATH)
    train_meta["file_id"] = train_meta["file_id"].astype(str)
    sample["file_id"] = sample["file_id"].astype(str)
    train_meta["label"] = train_meta["label"].astype(str)
    if train_meta["file_id"].duplicated().any() or sample["file_id"].duplicated().any():
        raise ValueError("Duplicate file_id found in metadata/submission template")
    if sorted(train_meta["label"].unique().tolist()) != sorted(LABELS):
        raise ValueError(f"Unexpected labels: {train_meta['label'].unique().tolist()}")

    fold_df = pd.read_csv(FOLD_PATH)
    if not {"file_id", "fold"}.issubset(fold_df.columns):
        raise ValueError("Fold file must contain file_id and fold columns")
    fold_df["file_id"] = fold_df["file_id"].astype(str)
    if fold_df["file_id"].duplicated().any():
        raise ValueError("Duplicate file_id in folds file")
    fold_map = fold_df.set_index("file_id")["fold"]
    train_meta["fold"] = train_meta["file_id"].map(fold_map)
    if train_meta["fold"].isna().any() or len(fold_df) != len(train_meta):
        raise ValueError("Saved folds do not align exactly with train metadata")
    train_meta["fold"] = train_meta["fold"].astype(int)
    if sorted(train_meta["fold"].unique().tolist()) != list(range(N_FOLDS)):
        raise ValueError("Expected fold IDs 0,1,2,3")
    label_to_idx = {label: i for i, label in enumerate(LABELS)}
    y = train_meta["label"].map(label_to_idx).to_numpy(dtype=int)
    folds = train_meta["fold"].to_numpy(dtype=int)

    # Locate and load all existing Triple V4 arrays.
    pths = {key: find_file(name, [OUT, SRC, "/kaggle/input"]) for key, name in ARTIFACTS.items()}
    missing = [f"{key}: {ARTIFACTS[key]}" for key, p in pths.items() if p is None]
    if missing:
        raise FileNotFoundError("Missing Triple V4 artifacts; attach testttrisan as input:\n" + "\n".join(missing))

    oof_r = normalize_probs(np.load(pths["oof_resnet"]))
    oof_b = normalize_probs(np.load(pths["oof_b2"]))
    oof_c = normalize_probs(np.load(pths["oof_convnext"]))
    if any(p.shape != (len(train_meta), 3) for p in [oof_r, oof_b, oof_c]):
        raise ValueError("OOF probability shapes do not match train metadata")

    with np.load(pths["test_resnet_crops"]) as dat:
        r5 = np.asarray(dat["test_r5"][:, [0, 2, 4], :], dtype=np.float64).mean(axis=1)
        r8 = np.asarray(dat["test_r8"][:, [0, 2, 4], :], dtype=np.float64).mean(axis=1)
    test_r = normalize_probs(0.25 * r5 + 0.75 * r8)
    test_b = normalize_probs(np.load(pths["test_b2"]))
    test_c = normalize_probs(np.load(pths["test_convnext"]))
    n_test = len(sample)
    if any(p.shape != (n_test, 3) for p in [test_r, test_b, test_c]):
        raise ValueError("Test probability shapes do not match sample_submission")

    p_oof = normalize_probs(0.55 * oof_r + 0.05 * oof_b + 0.40 * oof_c)
    p_test = normalize_probs(0.55 * test_r + 0.05 * test_b + 0.40 * test_c)
    unc_oof = 1.0 - p_oof.max(axis=1)
    unc_test = 1.0 - p_test.max(axis=1)
    ent_oof = entropy_rows(p_oof)
    ent_test = entropy_rows(p_test)
    margin_oof = np.sort(p_oof, axis=1)[:, -1] - np.sort(p_oof, axis=1)[:, -2]
    margin_test = np.sort(p_test, axis=1)[:, -1] - np.sort(p_test, axis=1)[:, -2]
    dis_oof = pairwise_tv_mean([oof_r, oof_b, oof_c])
    dis_test = pairwise_tv_mean([test_r, test_b, test_c])

    print("Train rows:", len(train_meta), "| test rows:", n_test)
    print("Triple V4 OOF known-only Macro F1:", round(float(f1_score(y, p_oof.argmax(axis=1), average="macro")), 6))

    # Resolve source files and load official public BEATs pretrained encoder.
    train_paths = resolve_audio_paths(train_meta, is_test=False)
    test_paths = resolve_audio_paths(sample, is_test=True)
    checkpoint_path = ensure_checkpoint()
    beats = load_beats(checkpoint_path)

    train_emb_path = os.path.join(OUT, TRAIN_EMB_NAME)
    test_emb_path = os.path.join(OUT, TEST_EMB_NAME)
    train_emb, train_acoustic = extract_or_load_embeddings(beats, train_paths, train_emb_path, "TRAIN audio/features")
    test_emb, test_acoustic = extract_or_load_embeddings(beats, test_paths, test_emb_path, "TEST audio/features")
    if train_emb.shape[0] != len(train_meta) or test_emb.shape[0] != len(sample):
        raise ValueError("Embedding arrays do not align with metadata")

    # Fold-safe embedding novelty: each validation audio is scored against other group folds only.
    proto_oof = np.zeros(len(train_meta), dtype=np.float32)
    knn_oof = np.zeros(len(train_meta), dtype=np.float32)
    mahal_oof = np.zeros(len(train_meta), dtype=np.float32)
    proto_test_folds, knn_test_folds, mahal_test_folds = [], [], []
    proto_noise_folds, knn_noise_folds, mahal_noise_folds = [], [], []

    # Generate artificial noise-only proxies, using observed train RMS quantiles.
    syn_waves = make_synthetic_noise(N_SYNTHETIC_NOISES, train_acoustic[:, 0], seed=SEED)
    syn_noise_emb_path = os.path.join(OUT, SYNTH_EMB_NAME)
    synth_emb = None
    if os.path.isfile(syn_noise_emb_path):
        try:
            old = np.load(syn_noise_emb_path)
            if old.shape == (N_SYNTHETIC_NOISES, train_emb.shape[1]):
                synth_emb = old.astype(np.float32)
        except Exception:
            synth_emb = None
    if synth_emb is None:
        print("Embedding synthetic noise-only proxies:", N_SYNTHETIC_NOISES)
        synth_emb = embed_wave_array(beats, syn_waves)
        np.save(syn_noise_emb_path, synth_emb)
    syn_acoustic = np.stack([acoustic_features(w) for w in syn_waves]).astype(np.float32)

    for fold in range(N_FOLDS):
        tr_idx = np.flatnonzero(folds != fold)
        va_idx = np.flatnonzero(folds == fold)
        scores_va = fit_fold_embedding_scores(train_emb[tr_idx], y[tr_idx], train_emb[va_idx])
        proto_oof[va_idx], knn_oof[va_idx], mahal_oof[va_idx] = scores_va
        scores_te = fit_fold_embedding_scores(train_emb[tr_idx], y[tr_idx], test_emb)
        proto_test_folds.append(scores_te[0]); knn_test_folds.append(scores_te[1]); mahal_test_folds.append(scores_te[2])
        scores_syn = fit_fold_embedding_scores(train_emb[tr_idx], y[tr_idx], synth_emb)
        proto_noise_folds.append(scores_syn[0]); knn_noise_folds.append(scores_syn[1]); mahal_noise_folds.append(scores_syn[2])
        print(f"Fold {fold}: embedding novelty scores calculated (train reference={len(tr_idx)}, val={len(va_idx)})")

    proto_test = np.mean(proto_test_folds, axis=0)
    knn_test = np.mean(knn_test_folds, axis=0)
    mahal_test = np.mean(mahal_test_folds, axis=0)
    proto_noise = np.mean(proto_noise_folds, axis=0)
    knn_noise = np.mean(knn_noise_folds, axis=0)
    mahal_noise = np.mean(mahal_noise_folds, axis=0)

    # Score tables: raw signals and percentile ranks relative to held-out known birds.
    acoustic_oof = pd.DataFrame(train_acoustic, columns=ACOUSTIC_NAMES)
    acoustic_test = pd.DataFrame(test_acoustic, columns=ACOUSTIC_NAMES)
    acoustic_noise = pd.DataFrame(syn_acoustic, columns=ACOUSTIC_NAMES)

    oof_signals = {
        "uncertainty": unc_oof,
        "entropy": ent_oof,
        "disagreement_tv": dis_oof,
        "margin_low_is_ood": -margin_oof,
        "beats_cosine_proto_distance": proto_oof,
        "beats_knn_cosine_distance": knn_oof,
        "beats_diag_mahalanobis": mahal_oof,
    }
    test_signals = {
        "uncertainty": unc_test,
        "entropy": ent_test,
        "disagreement_tv": dis_test,
        "margin_low_is_ood": -margin_test,
        "beats_cosine_proto_distance": proto_test,
        "beats_knn_cosine_distance": knn_test,
        "beats_diag_mahalanobis": mahal_test,
    }
    noise_signals = {
        "beats_cosine_proto_distance": proto_noise,
        "beats_knn_cosine_distance": knn_noise,
        "beats_diag_mahalanobis": mahal_noise,
    }
    for col in ACOUSTIC_NAMES:
        oof_signals["acoustic_" + col] = acoustic_oof[col].to_numpy()
        test_signals["acoustic_" + col] = acoustic_test[col].to_numpy()
        noise_signals["acoustic_" + col] = acoustic_noise[col].to_numpy()

    test_rank = {k: rank_against(v, test_signals[k]) for k, v in oof_signals.items()}

    # Composite rankings for analysis ONLY. Top-ranked samples are more OOD by these heuristics.
    candidate_scores = {
        "triple_v4_uncertainty_disagreement_60_40": 0.60 * test_rank["uncertainty"] + 0.40 * test_rank["disagreement_tv"],
        "beats_embedding_only": (
            test_rank["beats_cosine_proto_distance"] + test_rank["beats_knn_cosine_distance"] + test_rank["beats_diag_mahalanobis"]
        ) / 3.0,
        "triple_plus_beats_embedding": (
            0.40 * test_rank["uncertainty"] + 0.20 * test_rank["disagreement_tv"]
            + 0.15 * test_rank["beats_cosine_proto_distance"]
            + 0.15 * test_rank["beats_knn_cosine_distance"]
            + 0.10 * test_rank["beats_diag_mahalanobis"]
        ),
        "all_signals_exploratory": (
            0.27 * test_rank["uncertainty"] + 0.18 * test_rank["disagreement_tv"]
            + 0.13 * test_rank["beats_cosine_proto_distance"]
            + 0.13 * test_rank["beats_knn_cosine_distance"]
            + 0.09 * test_rank["beats_diag_mahalanobis"]
            + 0.10 * test_rank["acoustic_spectral_flatness_mean"]
            + 0.10 * test_rank["acoustic_spectral_entropy_mean"]
        ),
    }

    # Synthetic proxy validation: AUROC and AP for known OOF (negative) vs noise-only proxy (positive).
    # Only embedding and acoustic signals are scored because the original CNN checkpoints/logits
    # are not included in the saved probability artifacts, so synthetic CNN probabilities cannot be generated here.
    y_proxy = np.r_[np.zeros(len(train_meta), dtype=int), np.ones(len(synth_emb), dtype=int)]
    eval_rows = []
    raw_proxy_signals = {
        "beats_cosine_proto_distance": (proto_oof, proto_noise),
        "beats_knn_cosine_distance": (knn_oof, knn_noise),
        "beats_diag_mahalanobis": (mahal_oof, mahal_noise),
    }
    for col in ACOUSTIC_NAMES:
        raw_proxy_signals["acoustic_" + col] = (train_acoustic[:, ACOUSTIC_NAMES.index(col)], syn_acoustic[:, ACOUSTIC_NAMES.index(col)])
    for name, (known_vals, noise_vals) in raw_proxy_signals.items():
        scores = np.r_[known_vals, noise_vals]
        try:
            auc_high = float(roc_auc_score(y_proxy, scores))
            auc_low = float(roc_auc_score(y_proxy, -scores))
            auc_best_dir = max(auc_high, auc_low)
            direction = "higher" if auc_high >= auc_low else "lower"
            ap = float(average_precision_score(y_proxy, scores if direction == "higher" else -scores))
        except Exception:
            auc_high, auc_low, auc_best_dir, direction, ap = np.nan, np.nan, np.nan, "unknown", np.nan
        eval_rows.append({
            "signal": name,
            "AUROC_unknown_if_higher": auc_high,
            "AUROC_unknown_if_lower": auc_low,
            "AUROC_best_direction_on_synthetic_proxy": auc_best_dir,
            "direction_selected_on_synthetic_proxy": direction,
            "average_precision_best_direction": ap,
        })
    proxy_eval = pd.DataFrame(eval_rows).sort_values("AUROC_best_direction_on_synthetic_proxy", ascending=False)
    print("\n" + "=" * 78)
    print("SYNTHETIC NON-BIRD/NOISE PROXY EVALUATION")
    print("Known examples = fold-held-out training bird clips; positives = synthetic noise-only waveforms.")
    print("These metrics may overestimate real-world OOD performance; they are NOT test-label validation.")
    print(proxy_eval.round(4).to_string(index=False))

    # Test score table with raw signals, percentile ranks, per-model class, and candidate rank scores.
    class_pred = np.asarray(LABELS, dtype=object)[p_test.argmax(axis=1)]
    score_table = sample[["file_id"]].copy()
    score_table["triple_v4_pred_known_class"] = class_pred
    score_table["triple_v4_confidence"] = p_test.max(axis=1)
    score_table["triple_v4_uncertainty"] = unc_test
    score_table["triple_v4_entropy_norm"] = ent_test
    score_table["triple_v4_disagreement_tv"] = dis_test
    score_table["beats_cosine_proto_distance"] = proto_test
    score_table["beats_knn_cosine_distance"] = knn_test
    score_table["beats_diag_mahalanobis"] = mahal_test
    for col in ACOUSTIC_NAMES:
        score_table[col] = test_acoustic[:, ACOUSTIC_NAMES.index(col)]
    for k, r in test_rank.items():
        score_table["rank_" + k] = r
    for k, score in candidate_scores.items():
        score_table["ood_rank_score_" + k] = score
    score_path = os.path.join(OUT, "triple_v4_nonbird_ood_test_scores.csv")
    score_table.to_csv(score_path, index=False)

    # Overlap and class-count sensitivity: no submission files written.
    masks = {}
    for name, score in candidate_scores.items():
        ix = np.argsort(score, kind="mergesort")[-min(N_UNKNOWN_REFERENCE, n_test):]
        mask = np.zeros(n_test, dtype=bool); mask[ix] = True
        masks[name] = mask
    overlap_rows = []
    methods = list(masks)
    for a in methods:
        for b in methods:
            overlap_rows.append({"method_a": a, "method_b": b, "top245_overlap": int((masks[a] & masks[b]).sum())})
    overlap_df = pd.DataFrame(overlap_rows)

    count_rows = []
    for count in [150, 200, 245, 300, 350]:
        if count >= n_test:
            continue
        for method, score in candidate_scores.items():
            ix = np.argsort(score, kind="mergesort")[-count:]
            pred = class_pred.copy()
            pred[ix] = "Unknown"
            counts = pd.Series(pred).value_counts().to_dict()
            count_rows.append({
                "method": method,
                "unknown_count": count,
                "unknown_fraction": count / n_test,
                "pred_unknown": int((pred == "Unknown").sum()),
                "pred_bondol": int((pred == "Bondol Jawa").sum()),
                "pred_cendrawasih": int((pred == "Cendrawasih").sum()),
                "pred_kutilang": int((pred == "Kutilang").sum()),
            })
    counts_df = pd.DataFrame(count_rows)

    # Train OOF table lets user compare distributions of known bird clips.
    oof_table = train_meta[["file_id", "label", "source_group", "fold"]].copy()
    oof_table["triple_v4_confidence"] = p_oof.max(axis=1)
    oof_table["triple_v4_uncertainty"] = unc_oof
    oof_table["triple_v4_entropy_norm"] = ent_oof
    oof_table["triple_v4_disagreement_tv"] = dis_oof
    oof_table["beats_cosine_proto_distance"] = proto_oof
    oof_table["beats_knn_cosine_distance"] = knn_oof
    oof_table["beats_diag_mahalanobis"] = mahal_oof
    for col in ACOUSTIC_NAMES:
        oof_table[col] = train_acoustic[:, ACOUSTIC_NAMES.index(col)]
    oof_path = os.path.join(OUT, "triple_v4_nonbird_ood_oof_known_scores.csv")
    oof_table.to_csv(oof_path, index=False)

    proxy_eval_path = os.path.join(OUT, "triple_v4_synthetic_noise_proxy_eval.csv")
    overlap_path = os.path.join(OUT, "triple_v4_nonbird_ood_top245_overlap.csv")
    counts_path = os.path.join(OUT, "triple_v4_nonbird_ood_count_sensitivity.csv")
    proxy_eval.to_csv(proxy_eval_path, index=False)
    overlap_df.to_csv(overlap_path, index=False)
    counts_df.to_csv(counts_path, index=False)

    print("\n" + "=" * 78)
    print("TEST SCORE SUMMARY (no submission created)")
    print("=" * 78)
    for name, mask in masks.items():
        print(f"{name}: top-{N_UNKNOWN_REFERENCE} overlap selection distribution:", pd.Series(np.where(mask, "Unknown", class_pred)).value_counts().to_dict())
    print("\nTop 20 test files by fused OOD score:")
    fused_col = "ood_rank_score_all_signals_exploratory"
    print(score_table.sort_values(fused_col, ascending=False)[["file_id", "triple_v4_pred_known_class", "triple_v4_confidence", "beats_cosine_proto_distance", "beats_knn_cosine_distance", "beats_diag_mahalanobis", fused_col]].head(20).to_string(index=False))
    print("\nArtifacts saved:")
    for p in [score_path, oof_path, proxy_eval_path, overlap_path, counts_path]:
        print(" -", p)
    print("No submission CSV was created.")


if __name__ == "__main__":
    main()


SCHEMATICS NPC DSC 2026 — Triple V4 NON-BIRD / NOISE OOD DIAGNOSTICS
Device: cuda | GPU: Tesla T4
Unknown definition: non-bird, noise, or audio without bird sound
No submission will be created or sent.
Train rows: 1826 | test rows: 814
Triple V4 OOF known-only Macro F1: 0.942038
BEATs encoder dim: 768 | layers: 12
Reusing TRAIN audio/features BEATs embeddings (1826, 768): /kaggle/working/beats_iter3plus_as2m_8s_train_embeddings.npy
  TRAIN audio/features: 200/1826
  TRAIN audio/features: 400/1826
  TRAIN audio/features: 600/1826
  TRAIN audio/features: 800/1826
  TRAIN audio/features: 1000/1826
  TRAIN audio/features: 1200/1826
  TRAIN audio/features: 1400/1826
  TRAIN audio/features: 1600/1826
  TRAIN audio/features: 1800/1826
  TRAIN audio/features: 1826/1826
Reusing TEST audio/features BEATs embeddings (814, 768): /kaggle/working/beats_iter3plus_as2m_8s_test_embeddings.npy
  TEST audio/features: 200/814
  TEST audio/features: 400/814
  TEST audio/features: 600/814
  TEST audio/featu

In [15]:
import pandas as pd
W = "/kaggle/working/"
oof = pd.read_csv(W + "triple_v4_nonbird_ood_oof_known_scores.csv")
te  = pd.read_csv(W + "triple_v4_nonbird_ood_test_scores.csv")
cols = ["beats_knn_cosine_distance", "beats_cosine_proto_distance", "beats_diag_mahalanobis"]
rows = []
for c in ["Bondol Jawa", "Cendrawasih", "Kutilang"]:
    o = oof[oof.label == c]
    t = te[te.triple_v4_pred_known_class == c]
    for col in cols:
        frac = (t[col] > o[col].quantile(0.95)).mean()
        rows.append(dict(kelas=c, skor=col, n_test=len(t), n_oof=len(o),
                         frac_di_atas_p95=round(frac, 3),
                         est_unknown_min=round(max(frac - 0.05, 0) / 0.95, 3)))
print(pd.DataFrame(rows).to_string(index=False))
print(te.triple_v4_pred_known_class.value_counts(normalize=True).round(3))

      kelas                        skor  n_test  n_oof  frac_di_atas_p95  est_unknown_min
Bondol Jawa   beats_knn_cosine_distance     321   1124             0.227            0.187
Bondol Jawa beats_cosine_proto_distance     321   1124             0.100            0.052
Bondol Jawa      beats_diag_mahalanobis     321   1124             0.103            0.056
Cendrawasih   beats_knn_cosine_distance     182     83             0.060            0.011
Cendrawasih beats_cosine_proto_distance     182     83             0.011            0.000
Cendrawasih      beats_diag_mahalanobis     182     83             0.033            0.000
   Kutilang   beats_knn_cosine_distance     311    619             0.129            0.083
   Kutilang beats_cosine_proto_distance     311    619             0.035            0.000
   Kutilang      beats_diag_mahalanobis     311    619             0.029            0.000
triple_v4_pred_known_class
Bondol Jawa    0.394
Kutilang       0.382
Cendrawasih    0.224
Name: prop

In [16]:
import os
import numpy as np
import pandas as pd
import librosa

from IPython.display import display, Audio, Markdown

OUT = "/kaggle/working"
score_path = os.path.join(
    OUT, "triple_v4_nonbird_ood_test_scores.csv"
)

scores = pd.read_csv(score_path)
sample_df = pd.read_csv(SAMPLE_PATH)

base_col = "ood_rank_score_triple_v4_uncertainty_disagreement_60_40"
fused_col = "ood_rank_score_all_signals_exploratory"
N_UNKNOWN = 245

required = {"file_id", base_col, fused_col}
missing = required - set(scores.columns)
if missing:
    raise ValueError(f"Kolom skor tidak tersedia: {missing}")

scores["file_id"] = scores["file_id"].astype(str)
sample_df["file_id"] = sample_df["file_id"].astype(str)

base_top = set(scores.nlargest(N_UNKNOWN, base_col)["file_id"])
fused_top = set(scores.nlargest(N_UNKNOWN, fused_col)["file_id"])

common = base_top & fused_top
fused_only = fused_top - base_top
base_only = base_top - fused_top

print("Irisan kedua metode:", len(common), "dari", N_UNKNOWN)
print(
    "Jaccard overlap:",
    round(len(common) / max(1, len(base_top | fused_top)), 4)
)

# Pilih empat contoh per kelompok untuk audit awal.
groups = {
    "Dipilih kedua metode": (
        scores[scores["file_id"].isin(common)]
        .nlargest(4, fused_col)["file_id"].tolist()
    ),
    "Dipilih embedding/fusi saja": (
        scores[scores["file_id"].isin(fused_only)]
        .nlargest(4, fused_col)["file_id"].tolist()
    ),
    "Dipilih metode lama saja": (
        scores[scores["file_id"].isin(base_only)]
        .nlargest(4, base_col)["file_id"].tolist()
    ),
    "Kontrol: skor OOD rendah": (
        scores[~scores["file_id"].isin(fused_top)]
        .nsmallest(4, fused_col)["file_id"].tolist()
    ),
}

score_idx = scores.set_index("file_id")
sample_idx = sample_df.set_index("file_id")

for group_name, file_ids in groups.items():
    display(Markdown(f"## {group_name}"))

    for fid in file_ids:
        row = score_idx.loc[fid]
        meta = sample_idx.loc[[fid]].reset_index()

        # Fungsi resolver berasal dari notebook diagnostik sebelumnya.
        audio_path = resolve_audio_paths(meta, is_test=True)[0]
        y, sr = librosa.load(
            audio_path, sr=16000, mono=True, duration=8.0
        )

        display(Markdown(
            f"**File:** `{fid}`  \n"
            f"Prediksi kelas: **{row['triple_v4_pred_known_class']}**  \n"
            f"Confidence: `{row['triple_v4_confidence']:.3f}`  \n"
            f"BEATs centroid distance: "
            f"`{row['beats_cosine_proto_distance']:.4f}`  \n"
            f"Skor OOD gabungan: `{row[fused_col]:.4f}`"
        ))
        display(Audio(y, rate=sr))

print(
    "\nAudit selesai. Tidak ada label, CSV submission, "
    "atau file kompetisi yang diubah."
)

Irisan kedua metode: 131 dari 245
Jaccard overlap: 0.3649


## Dipilih kedua metode

**File:** `b4e2fe496b12e1dc4f9eeb61c3270b3fcdf03d1d6d290b89aa9e3b588a1999cb`  
Prediksi kelas: **Cendrawasih**  
Confidence: `0.515`  
BEATs centroid distance: `0.2218`  
Skor OOD gabungan: `0.9613`

**File:** `7abedc358bde04a408b7bfb8f5988d0f2db994d179b9bfbfa3034fa16b5dba0a`  
Prediksi kelas: **Kutilang**  
Confidence: `0.482`  
BEATs centroid distance: `0.2162`  
Skor OOD gabungan: `0.9532`

**File:** `ad423b401553b164269585cc030b2a8b8f47596083024153c59e4c12f00b2b74`  
Prediksi kelas: **Bondol Jawa**  
Confidence: `0.408`  
BEATs centroid distance: `0.2750`  
Skor OOD gabungan: `0.9381`

**File:** `05833a303c260eec62ea3cff3a6c64aea2d679c575ecaf1771ee795ebca26b58`  
Prediksi kelas: **Bondol Jawa**  
Confidence: `0.507`  
BEATs centroid distance: `0.2568`  
Skor OOD gabungan: `0.9336`

... (12 outputs left out)

**File:** `e4ddf7f6fbbe8c00a139dd64e2dbe1b6e4eaae08492fc09c0652645bf8910d47`  
Prediksi kelas: **Bondol Jawa**  
Confidence: `0.969`  
BEATs centroid distance: `0.1011`  
Skor OOD gabungan: `0.2267`

**File:** `8e3f6360bab5df0a285f07acd3affacefba74303bf737c21af5b7d4e3a9415cc`  
Prediksi kelas: **Kutilang**  
Confidence: `0.959`  
BEATs centroid distance: `0.0927`  
Skor OOD gabungan: `0.2458`

**File:** `4d4d2abf9ccb62c5cf83676a3a021a35d2a708362c7ffcf3aaece24b06057194`  
Prediksi kelas: **Kutilang**  
Confidence: `0.971`  
BEATs centroid distance: `0.1197`  
Skor OOD gabungan: `0.2537`


Audit selesai. Tidak ada label, CSV submission, atau file kompetisi yang diubah.


In [17]:
import os
import zipfile
import numpy as np
import pandas as pd
import librosa
import soundfile as sf

ROOT = "/kaggle/input/competitions/schematics-2026"
OUT = "/kaggle/working"

score_path = os.path.join(
    OUT, "triple_v4_nonbird_ood_test_scores.csv"
)
sample_path = os.path.join(ROOT, "sample_submission.csv")

scores = pd.read_csv(score_path)
sample_df = pd.read_csv(sample_path)

scores["file_id"] = scores["file_id"].astype(str)
sample_df["file_id"] = sample_df["file_id"].astype(str)

base_col = (
    "ood_rank_score_triple_v4_uncertainty_disagreement_60_40"
)
fused_col = "ood_rank_score_all_signals_exploratory"
N_UNKNOWN = 245
N_PER_GROUP = 5

base_top = set(scores.nlargest(N_UNKNOWN, base_col)["file_id"])
fused_top = set(scores.nlargest(N_UNKNOWN, fused_col)["file_id"])

common = base_top & fused_top
fused_only = fused_top - base_top
base_only = base_top - fused_top

groups = {
    "01_both_methods": (
        scores[scores.file_id.isin(common)]
        .nlargest(N_PER_GROUP, fused_col)
    ),
    "02_embedding_only": (
        scores[scores.file_id.isin(fused_only)]
        .nlargest(N_PER_GROUP, fused_col)
    ),
    "03_old_method_only": (
        scores[scores.file_id.isin(base_only)]
        .nlargest(N_PER_GROUP, base_col)
    ),
    "04_low_ood_control": (
        scores[~scores.file_id.isin(fused_top)]
        .nsmallest(N_PER_GROUP, fused_col)
    ),
}

sample_idx = sample_df.set_index("file_id")
audio_dir = os.path.join(OUT, "triple_v4_ood_audio_audit")
os.makedirs(audio_dir, exist_ok=True)

manifest_rows = []

for group_name, group_df in groups.items():
    for _, row in group_df.iterrows():
        fid = row["file_id"]

        if fid not in sample_idx.index:
            print("ID tidak ditemukan di sample_submission:", fid)
            continue

        meta = sample_idx.loc[[fid]].reset_index()

        # Fungsi ini sudah digunakan pada notebook diagnostik sebelumnya.
        audio_path = resolve_audio_paths(meta, is_test=True)[0]

        # Simpan klip 8 detik mono pada 16 kHz.
        audio, sr = librosa.load(
            audio_path, sr=16000, mono=True, duration=8.0
        )

        filename = f"{group_name}_{fid[:12]}.wav"
        wav_path = os.path.join(audio_dir, filename)
        sf.write(wav_path, audio, sr, subtype="PCM_16")

        entry = row.to_dict()
        entry["audit_group"] = group_name
        entry["audio_filename"] = filename
        manifest_rows.append(entry)

manifest_path = os.path.join(audio_dir, "audit_manifest.csv")
pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)

zip_path = os.path.join(
    OUT, "triple_v4_ood_audio_audit.zip"
)

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    for filename in os.listdir(audio_dir):
        full_path = os.path.join(audio_dir, filename)
        z.write(full_path, arcname=filename)

print("Klip berhasil diekspor:", len(manifest_rows))
print("ZIP:", zip_path)
print("Manifest:", manifest_path)
print("Tidak ada submission dibuat.")

Klip berhasil diekspor: 20
ZIP: /kaggle/working/triple_v4_ood_audio_audit.zip
Manifest: /kaggle/working/triple_v4_ood_audio_audit/audit_manifest.csv
Tidak ada submission dibuat.


In [19]:
import os
import numpy as np
import pandas as pd

OUT = "/kaggle/working"
SCORE_PATH = os.path.join(
    OUT, "triple_v4_nonbird_ood_test_scores.csv"
)

N_UNKNOWN = 245

df = pd.read_csv(SCORE_PATH)

required = {
    "file_id",
    "triple_v4_pred_known_class",
    "triple_v4_confidence",
    "rank_uncertainty",
    "rank_disagreement_tv",
}
missing = required - set(df.columns)

if missing:
    raise ValueError(f"Kolom tidak ditemukan: {missing}")

if len(df) != 814:
    print(f"Peringatan: jumlah baris test adalah {len(df)}, bukan 814.")

# Bobot pertama adalah komponen uncertainty,
# bobot kedua adalah disagreement antar-model.
ratios = [
    (0.50, 0.50),
    (0.60, 0.40),
    (0.70, 0.30),
    (0.75, 0.25),
    (0.80, 0.20),
]

rows = []

for w_unc, w_dis in ratios:
    # Formula ranking yang sama dengan eksperimen Triple V4.
    score = (
        w_unc * df["rank_uncertainty"].to_numpy()
        + w_dis * df["rank_disagreement_tv"].to_numpy()
    )

    # Skor tinggi dipilih sebagai kandidat Unknown.
    selected_idx = np.argsort(score, kind="mergesort")[-N_UNKNOWN:]
    selected = df.iloc[selected_idx]
    remaining = df.drop(df.index[selected_idx])

    selected_counts = selected["triple_v4_pred_known_class"].value_counts()
    remaining_counts = remaining["triple_v4_pred_known_class"].value_counts()

    selected_cend = selected[
        selected["triple_v4_pred_known_class"] == "Cendrawasih"
    ]

    rows.append({
        "uncertainty_weight": w_unc,
        "disagreement_weight": w_dis,
        "unknown_count": len(selected),
        "mean_conf_unknown_candidates":
            selected["triple_v4_confidence"].mean(),
        "median_conf_unknown_candidates":
            selected["triple_v4_confidence"].median(),

        # Prediksi kelas sebelum diganti menjadi Unknown.
        "pred_cendrawasih_reassigned_unknown":
            int((selected["triple_v4_pred_known_class"] == "Cendrawasih").sum()),
        "pred_kutilang_reassigned_unknown":
            int((selected["triple_v4_pred_known_class"] == "Kutilang").sum()),
        "pred_bondol_reassigned_unknown":
            int((selected["triple_v4_pred_known_class"] == "Bondol Jawa").sum()),

        "mean_conf_reassigned_cendrawasih":
            selected_cend["triple_v4_confidence"].mean()
            if len(selected_cend) else np.nan,

        # Jumlah prediksi kelas yang tersisa setelah seleksi.
        "cendrawasih_remaining":
            int((remaining["triple_v4_pred_known_class"] == "Cendrawasih").sum()),
        "kutilang_remaining":
            int((remaining["triple_v4_pred_known_class"] == "Kutilang").sum()),
        "bondol_remaining":
            int((remaining["triple_v4_pred_known_class"] == "Bondol Jawa").sum()),
    })

summary = pd.DataFrame(rows)

output_path = os.path.join(
    OUT, "triple_v4_ratio_sensitivity_245.csv"
)
summary.to_csv(output_path, index=False)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print("=" * 90)
print("TRIPLE V4 — AUTOMATED RATIO SENSITIVITY")
print("=" * 90)
print(summary.round(4).to_string(index=False))
print("\nSaved:", output_path)
print("Tidak ada model dilatih ulang atau submission dibuat.")

TRIPLE V4 — AUTOMATED RATIO SENSITIVITY
 uncertainty_weight  disagreement_weight  unknown_count  mean_conf_unknown_candidates  median_conf_unknown_candidates  pred_cendrawasih_reassigned_unknown  pred_kutilang_reassigned_unknown  pred_bondol_reassigned_unknown  mean_conf_reassigned_cendrawasih  cendrawasih_remaining  kutilang_remaining  bondol_remaining
               0.50                 0.50            245                        0.5240                          0.5278                                   66                                72                             107                            0.5070                    116                 239               214
               0.60                 0.40            245                        0.5191                          0.5185                                   70                                76                              99                            0.5080                    112                 235               222
            

In [20]:
import os
import pandas as pd

path = "/kaggle/working/triple_v4_nonbird_ood_oof_known_scores.csv"

oof = pd.read_csv(path)

print("Shape:", oof.shape)
print("\nKolom tersedia:")
for col in oof.columns:
    print("-", col)

print("\nLima baris pertama:")
display(oof.head())

Shape: (1826, 21)

Kolom tersedia:
- file_id
- label
- source_group
- fold
- triple_v4_confidence
- triple_v4_uncertainty
- triple_v4_entropy_norm
- triple_v4_disagreement_tv
- beats_cosine_proto_distance
- beats_knn_cosine_distance
- beats_diag_mahalanobis
- rms
- rms_db
- zero_crossing_rate
- spectral_flatness_median
- spectral_flatness_mean
- spectral_entropy_mean
- spectral_centroid_hz
- low_band_fraction_lt500hz
- high_band_fraction_ge2500hz
- spectral_flatness_p90

Lima baris pertama:


,file_id,label,source_group,fold,triple_v4_confidence,triple_v4_uncertainty,triple_v4_entropy_norm,triple_v4_disagreement_tv,beats_cosine_proto_distance,beats_knn_cosine_distance,beats_diag_mahalanobis,rms,rms_db,zero_crossing_rate,spectral_flatness_median,spectral_flatness_mean,spectral_entropy_mean,spectral_centroid_hz,low_band_fraction_lt500hz,high_band_fraction_ge2500hz,spectral_flatness_p90
0,313fef340a5c0be86a6c3ec29bb0fc5b684accd321f994...,Bondol Jawa,d20bbb9d3f16cea2e0988846afe752fa7233d9de8cffd7...,2,0.936524,0.063476,0.251039,0.097189,0.198319,0.055438,1.228804,0.052915,-25.528467,0.320026,0.232157,0.222823,0.852079,2298.4430,0.139101,0.365647,0.261079
1,9f0d58ab5eb8bdad788915ba9053c9a23b2709461ee290...,Bondol Jawa,100f6ca56523c6b10f0da48fca4a3100f7e2fad1a2c238...,0,0.600115,0.399885,0.659550,0.671849,0.127451,0.086417,0.938688,0.107415,-19.378670,0.358511,0.195725,0.167663,0.803908,2822.0510,0.079335,0.564573,0.250918
2,f18d5438129a6c8fabe09f8d108c44b0fa40880cf0ffcb...,Kutilang,81b28e5fe98ace507d968e64f7cf87d7c4af112b8d3d19...,2,0.974882,0.025118,0.121994,0.068947,0.139325,0.094933,0.980746,0.115078,-18.780160,0.151704,0.032325,0.034150,0.660129,991.0136,0.553916,0.141492,0.053191
3,ec4ea61e4451227b2ee1e4046296b515e4f58761ea4097...,Bondol Jawa,d20bbb9d3f16cea2e0988846afe752fa7233d9de8cffd7...,2,0.975008,0.024992,0.117782,0.058913,0.146042,0.110097,0.991827,0.063528,-23.940650,0.450769,0.076894,0.077878,0.761752,3660.1106,0.207195,0.685010,0.107523
4,d0bd6e1839642188508060a7bb5c29c1a8251ada1a48f0...,Kutilang,0d9c30be6d4b5d92c8384b9e5fd3cf908e4b8a965c68a9...,0,0.977768,0.022232,0.107820,0.060579,0.142582,0.079767,0.927286,0.097072,-20.258139,0.343143,0.170872,0.154936,0.776872,2638.3003,0.071868,0.603505,0.258534


In [21]:
import os
import numpy as np
import pandas as pd

OUT = "/kaggle/working"

OOF_PATH = os.path.join(
    OUT, "triple_v4_nonbird_ood_oof_known_scores.csv"
)

N_UNKNOWN_TEST = 245
N_TEST = 814
REJECT_FRACTION = N_UNKNOWN_TEST / N_TEST

df = pd.read_csv(OOF_PATH)

# Kolom yang diperlukan untuk membandingkan semua metode
required = [
    "label",
    "triple_v4_confidence",
    "triple_v4_uncertainty",
    "triple_v4_disagreement_tv",
    "beats_cosine_proto_distance",
    "beats_knn_cosine_distance",
    "beats_diag_mahalanobis",
]

missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom yang diperlukan tidak ditemukan: {missing_cols}")

# Periksa missing value dan nilai non-finite.
print("Jumlah baris OOF:", len(df))
print("\nMissing values per kolom:")
print(df[required].isna().sum().to_string())

numeric_cols = [c for c in required if c != "label"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

valid_mask = (
    df["label"].notna()
    & np.isfinite(df[numeric_cols]).all(axis=1)
)

valid = df.loc[valid_mask].copy()

n_dropped = len(df) - len(valid)
print(f"\nBaris valid: {len(valid)} / {len(df)}")
print("Baris yang dikeluarkan karena nilai tidak valid:", n_dropped)

if len(valid) < 0.98 * len(df):
    raise ValueError(
        "Lebih dari 2% baris tidak valid. Periksa missing values "
        "sebelum melanjutkan supaya evaluasi tidak bias."
    )

if n_dropped:
    print("PERINGATAN: hasil dihitung hanya pada baris valid.")

classes = ["Bondol Jawa", "Cendrawasih", "Kutilang"]
observed = set(valid["label"].unique())
if not observed.issubset(set(classes)):
    raise ValueError(f"Label di luar tiga kelas known: {observed - set(classes)}")

# Pastikan arah semua skor konsisten:
# skor lebih tinggi = lebih berpotensi ditandai sebagai Unknown.
score = pd.DataFrame(index=valid.index)

score["uncertainty_rank"] = valid[
    "triple_v4_uncertainty"
].rank(pct=True)

score["disagreement_rank"] = valid[
    "triple_v4_disagreement_tv"
].rank(pct=True)

score["confidence_low_rank"] = (
    -valid["triple_v4_confidence"]
).rank(pct=True)

score["beats_centroid_rank"] = valid[
    "beats_cosine_proto_distance"
].rank(pct=True)

score["beats_knn_rank"] = valid[
    "beats_knn_cosine_distance"
].rank(pct=True)

score["beats_mahalanobis_rank"] = valid[
    "beats_diag_mahalanobis"
].rank(pct=True)

score["beats_mean_distance_rank"] = score[
    [
        "beats_centroid_rank",
        "beats_knn_rank",
        "beats_mahalanobis_rank",
    ]
].mean(axis=1)

# Formula rank-based. Nilai tinggi diperlakukan sebagai kandidat Unknown.
for w_unc, w_dis in [
    (0.50, 0.50),
    (0.60, 0.40),
    (0.70, 0.30),
    (0.75, 0.25),
    (0.80, 0.20),
]:
    name = f"triple_{int(w_unc*100)}_{int(w_dis*100)}"
    score[name] = (
        w_unc * score["uncertainty_rank"]
        + w_dis * score["disagreement_rank"]
    )

score["uncertainty_only"] = score["uncertainty_rank"]
score["disagreement_only"] = score["disagreement_rank"]
score["confidence_only"] = score["confidence_low_rank"]
score["beats_centroid_only"] = score["beats_centroid_rank"]
score["beats_knn_only"] = score["beats_knn_rank"]
score["beats_mahalanobis_only"] = score["beats_mahalanobis_rank"]
score["beats_all_three_equal"] = score["beats_mean_distance_rank"]

# Eksperimen gabungan eksploratif: Triple 60/40 + embedding BEATs.
score["triple60_plus_beats10"] = (
    0.90 * score["triple_60_40"]
    + 0.10 * score["beats_mean_distance_rank"]
)
score["triple60_plus_beats20"] = (
    0.80 * score["triple_60_40"]
    + 0.20 * score["beats_mean_distance_rank"]
)

# Tolak proporsi OOF yang sama dengan proporsi kandidat Unknown pada test.
n_reject = round(len(valid) * REJECT_FRACTION)
n_reject = max(1, min(n_reject, len(valid) - 1))

summary_rows = []
class_rows = []
selection_rows = []

true_labels = valid["label"]

for method in score.columns:
    # Stable sorting memastikan hasil dapat direproduksi.
    selected_idx = score[method].sort_values(
        ascending=False, kind="mergesort"
    ).head(n_reject).index

    rejected_labels = true_labels.loc[selected_idx]
    rejected_counts = rejected_labels.value_counts().reindex(
        classes, fill_value=0
    )
    total_counts = true_labels.value_counts().reindex(
        classes, fill_value=0
    )

    class_rates = {}

    for cls in classes:
        total_cls = int(total_counts[cls])
        rejected_cls = int(rejected_counts[cls])
        rate = rejected_cls / total_cls if total_cls else np.nan
        class_rates[cls] = rate

        class_rows.append({
            "method": method,
            "class": cls,
            "total_oof_examples": total_cls,
            "rejected_as_unknown": rejected_cls,
            "false_rejection_rate": rate,
            "share_of_rejected": rejected_cls / n_reject,
        })

    rates = [v for v in class_rates.values() if np.isfinite(v)]

    summary_rows.append({
        "method": method,
        "n_oof": len(valid),
        "n_rejected_as_unknown": n_reject,
        "rejection_fraction": n_reject / len(valid),
        "bondol_false_rejection_rate": class_rates["Bondol Jawa"],
        "cendrawasih_false_rejection_rate": class_rates["Cendrawasih"],
        "kutilang_false_rejection_rate": class_rates["Kutilang"],
        "max_min_class_rejection_gap_pp": (
            (max(rates) - min(rates)) * 100 if rates else np.nan
        ),
        "mean_confidence_of_rejected": valid.loc[
            selected_idx, "triple_v4_confidence"
        ].mean(),
    })

    for idx in selected_idx:
        selection_rows.append({
            "method": method,
            "file_id": valid.loc[idx, "file_id"],
            "label": valid.loc[idx, "label"],
            "source_group": valid.loc[idx, "source_group"],
            "fold": valid.loc[idx, "fold"],
            "triple_v4_confidence": valid.loc[idx, "triple_v4_confidence"],
            "ood_score_rank": score.loc[idx, method],
        })

summary = pd.DataFrame(summary_rows).sort_values(
    "max_min_class_rejection_gap_pp"
)
class_detail = pd.DataFrame(class_rows)
selections = pd.DataFrame(selection_rows)

summary_path = os.path.join(
    OUT, "triple_v4_oof_false_rejection_summary.csv"
)
class_path = os.path.join(
    OUT, "triple_v4_oof_false_rejection_by_class.csv"
)
selection_path = os.path.join(
    OUT, "triple_v4_oof_false_rejection_selected_rows.csv"
)

summary.to_csv(summary_path, index=False)
class_detail.to_csv(class_path, index=False)
selections.to_csv(selection_path, index=False)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print("\n" + "=" * 110)
print("TRIPLE V4 — OOF FALSE-REJECTION SUMMARY")
print("=" * 110)
print(summary.round(4).to_string(index=False))

print("\n" + "=" * 110)
print("FALSE-REJECTION RATE PER CLASS — TRIPLE V4 60/40 DAN BEATs")
print("=" * 110)

chosen_methods = [
    "triple_60_40",
    "triple_75_25",
    "beats_centroid_only",
    "beats_knn_only",
    "beats_mahalanobis_only",
    "beats_all_three_equal",
    "triple60_plus_beats10",
    "triple60_plus_beats20",
]

print(
    class_detail[class_detail["method"].isin(chosen_methods)]
    .round(4)
    .to_string(index=False)
)

print("\nFile hasil:")
print(summary_path)
print(class_path)
print(selection_path)
print("\nDiagnostik OOF selesai. Tidak ada submission dibuat.")

Jumlah baris OOF: 1826

Missing values per kolom:
label                          0
triple_v4_confidence           0
triple_v4_uncertainty          0
triple_v4_disagreement_tv      0
beats_cosine_proto_distance    0
beats_knn_cosine_distance      0
beats_diag_mahalanobis         0

Baris valid: 1826 / 1826
Baris yang dikeluarkan karena nilai tidak valid: 0

TRIPLE V4 — OOF FALSE-REJECTION SUMMARY
                  method  n_oof  n_rejected_as_unknown  rejection_fraction  bondol_false_rejection_rate  cendrawasih_false_rejection_rate  kutilang_false_rejection_rate  max_min_class_rejection_gap_pp  mean_confidence_of_rejected
     beats_centroid_rank   1826                    550              0.3012                       0.2998                            0.4699                         0.2811                         18.8781                       0.8661
     beats_centroid_only   1826                    550              0.3012                       0.2998                            0.4699    

In [22]:
import os
import numpy as np
import pandas as pd

PATH = "/kaggle/working/triple_v4_nonbird_ood_oof_known_scores.csv"
OUT = "/kaggle/working"

df = pd.read_csv(PATH)

required = [
    "fold", "label",
    "triple_v4_uncertainty",
    "triple_v4_disagreement_tv",
    "beats_cosine_proto_distance",
    "beats_knn_cosine_distance",
    "beats_diag_mahalanobis",
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Kolom tidak ditemukan: {missing}")

df = df.dropna(subset=required).copy()

REJECT_FRACTION = 245 / 814
CLASSES = ["Bondol Jawa", "Cendrawasih", "Kutilang"]

records = []

for fold, g in df.groupby("fold", sort=True):
    g = g.copy()

    # Ranking dihitung di dalam fold agar skala skor antarfold
    # tidak langsung menentukan ranking keseluruhan.
    rank = pd.DataFrame(index=g.index)

    rank["Triple V4 60/40"] = (
        0.60 * g["triple_v4_uncertainty"].rank(pct=True)
        + 0.40 * g["triple_v4_disagreement_tv"].rank(pct=True)
    )

    rank["BEATs centroid"] = g[
        "beats_cosine_proto_distance"
    ].rank(pct=True)

    rank["BEATs Mahalanobis"] = g[
        "beats_diag_mahalanobis"
    ].rank(pct=True)

    rank["BEATs kNN"] = g[
        "beats_knn_cosine_distance"
    ].rank(pct=True)

    n_reject = round(len(g) * REJECT_FRACTION)

    for method in rank.columns:
        selected_idx = rank[method].nlargest(n_reject).index
        selected = g.loc[selected_idx]

        for cls in CLASSES:
            class_rows = g[g["label"] == cls]
            rejected = int((selected["label"] == cls).sum())
            total = len(class_rows)

            records.append({
                "fold": int(fold),
                "method": method,
                "fold_rows": len(g),
                "rejected_total": len(selected),
                "class": cls,
                "class_total": total,
                "class_rejected": rejected,
                "false_rejection_rate": (
                    rejected / total if total else np.nan
                ),
            })

result = pd.DataFrame(records)

summary = (
    result.groupby(["method", "class"])["false_rejection_rate"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)

print("=" * 90)
print("OOF FOLD STABILITY — CLASS FALSE-REJECTION RATE")
print("=" * 90)
print(summary.round(4).to_string(index=False))

path = os.path.join(OUT, "triple_v4_ood_fold_stability.csv")
result.to_csv(path, index=False)
print("\nDetail tersimpan:", path)

OOF FOLD STABILITY — CLASS FALSE-REJECTION RATE
           method       class   mean    std    min    max
BEATs Mahalanobis Bondol Jawa 0.3196 0.0453 0.2564 0.3545
BEATs Mahalanobis Cendrawasih 0.5341 0.1201 0.3750 0.6667
BEATs Mahalanobis    Kutilang 0.2220 0.0477 0.1627 0.2727
   BEATs centroid Bondol Jawa 0.2914 0.0322 0.2479 0.3248
   BEATs centroid Cendrawasih 0.5265 0.1105 0.4286 0.6667
   BEATs centroid    Kutilang 0.2836 0.0246 0.2528 0.3106
        BEATs kNN Bondol Jawa 0.2646 0.0766 0.1940 0.3471
        BEATs kNN Cendrawasih 0.7613 0.1354 0.6250 0.8966
        BEATs kNN    Kutilang 0.2677 0.0831 0.2135 0.3916
  Triple V4 60/40 Bondol Jawa 0.4050 0.1957 0.2462 0.6752
  Triple V4 60/40 Cendrawasih 0.6680 0.4344 0.0857 1.0000
  Triple V4 60/40    Kutilang 0.1902 0.1864 0.0227 0.3706

Detail tersimpan: /kaggle/working/triple_v4_ood_fold_stability.csv


In [23]:
import os
import numpy as np
import pandas as pd

ROOT = "/kaggle/input/competitions/schematics-2026"
OUT = "/kaggle/working"

SCORE_PATH = os.path.join(
    OUT, "triple_v4_nonbird_ood_test_scores.csv"
)
SAMPLE_PATH = os.path.join(ROOT, "sample_submission.csv")

N_UNKNOWN = 245

scores = pd.read_csv(SCORE_PATH)
sample = pd.read_csv(SAMPLE_PATH)

required = {
    "file_id",
    "triple_v4_pred_known_class",
    "beats_cosine_proto_distance",
}
missing = required - set(scores.columns)

if missing:
    raise ValueError(f"Kolom skor tidak tersedia: {missing}")

scores["file_id"] = scores["file_id"].astype(str)
sample["file_id"] = sample["file_id"].astype(str)

if scores["file_id"].duplicated().any():
    raise ValueError("Ada file_id duplikat pada skor OOD.")

if sample["file_id"].duplicated().any():
    raise ValueError("Ada file_id duplikat pada sample_submission.")

if len(scores) != len(sample):
    raise ValueError(
        f"Jumlah skor ({len(scores)}) tidak cocok "
        f"dengan jumlah test ({len(sample)})."
    )

if set(scores["file_id"]) != set(sample["file_id"]):
    raise ValueError("Daftar file_id skor dan sample_submission berbeda.")

distance = pd.to_numeric(
    scores["beats_cosine_proto_distance"], errors="coerce"
)

if not np.isfinite(distance).all():
    raise ValueError("Ada skor centroid yang tidak valid.")

if not 0 < N_UNKNOWN < len(sample):
    raise ValueError("Jumlah Unknown tidak valid.")

# Urutkan jarak tertinggi sebagai kandidat paling OOD.
selection = scores.assign(
    beats_centroid_distance=distance
).sort_values(
    ["beats_centroid_distance", "file_id"],
    ascending=[False, True],
    kind="mergesort",
)

unknown_ids = set(selection.head(N_UNKNOWN)["file_id"])

# Pertahankan prediksi kelas dari Triple V4.
pred_map = scores.set_index("file_id")[
    "triple_v4_pred_known_class"
]
prediction = sample["file_id"].map(pred_map)

if prediction.isna().any():
    raise ValueError("Ada prediksi Triple V4 yang tidak ditemukan.")

prediction = prediction.astype(str)
prediction[sample["file_id"].isin(unknown_ids)] = "Unknown"

target_cols = [c for c in sample.columns if c != "file_id"]

if len(target_cols) != 1:
    raise ValueError(
        f"Kolom target tidak dikenali: {sample.columns.tolist()}"
    )

submission = sample.copy()
submission[target_cols[0]] = prediction

output_path = os.path.join(
    OUT, "submission_triple_v4_beats_centroid_u245.csv"
)
submission.to_csv(output_path, index=False)

# Simpan daftar 245 pilihan untuk audit.
selected = selection.head(N_UNKNOWN).copy()
selected["selected_as_unknown"] = True

audit_path = os.path.join(
    OUT, "triple_v4_beats_centroid_selected_245.csv"
)
selected.to_csv(audit_path, index=False)

# Validasi akhir.
check = pd.read_csv(output_path)

assert len(check) == len(sample)
assert check["file_id"].equals(sample["file_id"])
assert check[target_cols[0]].notna().all()
assert (check[target_cols[0]] == "Unknown").sum() == N_UNKNOWN
assert set(check[target_cols[0]].unique()).issubset(
    {"Bondol Jawa", "Cendrawasih", "Kutilang", "Unknown"}
)

print("=" * 65)
print("TRIPLE V4 + BEATs CENTROID — KANDIDAT OFFLINE")
print("=" * 65)
print("Classifier: Triple V4, tanpa perubahan")
print("Strategi Unknown: jarak centroid BEATs tertinggi")
print("Jumlah Unknown:", int((prediction == "Unknown").sum()))

print("\nDistribusi prediksi awal pada 245 file terpilih:")
print(
    selected["triple_v4_pred_known_class"]
    .value_counts()
    .to_string()
)

print("\nDistribusi final:")
print(pd.Series(prediction).value_counts().to_string())

print("\nCSV kandidat:", output_path)
print("Daftar pilihan:", audit_path)
print("Status: BELUM DISUBMIT")

TRIPLE V4 + BEATs CENTROID — KANDIDAT OFFLINE
Classifier: Triple V4, tanpa perubahan
Strategi Unknown: jarak centroid BEATs tertinggi
Jumlah Unknown: 245

Distribusi prediksi awal pada 245 file terpilih:
triple_v4_pred_known_class
Bondol Jawa    118
Cendrawasih     73
Kutilang        54

Distribusi final:
file_id
Kutilang       257
Unknown        245
Bondol Jawa    203
Cendrawasih    109

CSV kandidat: /kaggle/working/submission_triple_v4_beats_centroid_u245.csv
Daftar pilihan: /kaggle/working/triple_v4_beats_centroid_selected_245.csv
Status: BELUM DISUBMIT


In [24]:
import pandas as pd
W = "/kaggle/working/"
best = pd.read_csv(W + "offline_candidate_triple_v4_c60_d40_u245.csv").set_index("file_id")["label"]
new  = pd.read_csv(W + "submission_triple_v4_beats_centroid_u245.csv").set_index("file_id")["label"].reindex(best.index)
ub, un = best.eq("Unknown"), new.eq("Unknown")
print("irisan Unknown:", int((ub & un).sum()), "dari 245")
print("label dikenal yang berbeda (harus 0):", int((best[~ub & ~un] != new[~ub & ~un]).sum()))

irisan Unknown: 84 dari 245
label dikenal yang berbeda (harus 0): 0


In [25]:
# -*- coding: utf-8 -*-
"""SCHEMATICS NPC DSC 2026 — Triple V4 LOCO + confidence-shift diagnostics.

This notebook/script does NOT train the CNNs and does NOT create a submission.
It uses existing Triple V4 probability artifacts plus frozen BEATs embeddings.

Part A: leave-one-class-out (LOCO) validation on known training birds.
  For each of the three classes, exclude that class entirely from fitting. A
  source_group-separated fold is used for evaluation. The held-out species is
  treated as a pseudo-Unknown and compared with the other two known species.
  Methods: logistic-regression max-probability/entropy/margin and BEATs
  cosine-prototype, kNN cosine, and diagonal Mahalanobis distance.

Part B: class-wise Triple V4 OOF-vs-test confidence/margin comparison.
  It reports predicted-class proportions, confidence quantiles, margins,
  entropy, soft probability mass, and how many test predictions fall below
  OOF known-audio thresholds. This is a diagnostic, NOT an estimate of the true
  Unknown count: test class priors may differ and true test labels are absent.

Caveats:
- LOCO is only a proxy: a withheld training species is not necessarily similar
  to the unseen species present in test.
- The probabilities saved by the old models are post-softmax, so raw-logit
  energy is not available and is not fabricated here.
- No labels are inferred for test; no submission CSV is created or sent.
"""

import os
import sys
import glob
import time
import types
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score

warnings.filterwarnings("ignore", category=FutureWarning)

# =============================================================================
# CONFIG
# =============================================================================
SEED = 16
N_FOLDS = 4
LABELS = ["Bondol Jawa", "Cendrawasih", "Kutilang"]
SR = 16000
DURATION = 8.0
NUM_SAMPLES = int(SR * DURATION)
BATCH_SIZE = 8
NUM_WORKERS = 2
USE_AMP = True
KNN_K = 5
LOGREG_C = 0.03

ROOT = "/kaggle/input/competitions/schematics-2026"
SRC = "/kaggle/input/notebooks/radithiafatir/testttrisan"
OUT = "/kaggle/working"
BEATS_CODE_DIR = os.path.join(OUT, "beats_official_code")
CHECKPOINT_NAME = "BEATs_iter3_plus_AS2M.pt"
CHECKPOINT_REPO_ID = "lpepino/beats_ckpts"

TRAIN_META_PATH = os.path.join(ROOT, "train", "metadata.csv")
SAMPLE_PATH = os.path.join(ROOT, "sample_submission.csv")
FOLD_PATH = os.path.join(SRC, "bird_group_folds_seed16.csv")
TRAIN_EMB_NAME = "beats_iter3plus_as2m_8s_train_embeddings.npy"
TEST_EMB_NAME = "beats_iter3plus_as2m_8s_test_embeddings.npy"

ARTIFACTS = {
    "oof_resnet": "oof_resnet50_ensemble_tta3_seed16.npy",
    "oof_b2": "oof_efficientnet_b2_audio8s_tta3_seed16.npy",
    "oof_convnext": "oof_convnext_tiny_audio8s_tta3_seed16_pretrained_v4_resize224.npy",
    "test_resnet_crops": "tta_test_crop_probabilities.npz",
    "test_b2": "test_effnet_b2_audio8s_tta3.npy",
    "test_convnext": "test_convnext_tiny_pretrained_v4_resize224_tta3.npy",
}

os.makedirs(OUT, exist_ok=True)
os.makedirs(BEATS_CODE_DIR, exist_ok=True)


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def find_file(filename, dirs=None):
    """Locate an artifact by exact filename in working/input directories."""
    dirs = dirs or [OUT, SRC, "/kaggle/input"]
    for folder in dirs:
        direct = os.path.join(folder, filename)
        if os.path.isfile(direct):
            return direct
    for folder in dirs:
        if os.path.isdir(folder):
            try:
                hits = glob.glob(os.path.join(folder, "**", filename), recursive=True)
                if hits:
                    return hits[0]
            except Exception:
                pass
    return None


def normalize_probs(p):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, 1e-12, None)
    return p / p.sum(axis=1, keepdims=True)


def entropy_norm(p):
    p = normalize_probs(p)
    return -(p * np.log(p)).sum(axis=1) / np.log(p.shape[1])


def resolve_audio_paths(meta, is_test=False):
    audio_dir = os.path.join(ROOT, "test", "audio") if is_test else os.path.join(ROOT, "train", "audio")
    out = []
    for _, row in meta.iterrows():
        raw = str(row.get("audio_path", "")) if not is_test else ""
        fid = str(row["file_id"])
        fid_base = os.path.splitext(os.path.basename(fid))[0]
        candidates = []
        if raw and raw.lower() != "nan":
            if os.path.isabs(raw):
                candidates.append(raw)
            else:
                candidates.extend([os.path.join(ROOT, raw), os.path.join(ROOT, "train", raw)])
                candidates.append(os.path.join(audio_dir, os.path.basename(raw)))
        candidates.append(os.path.join(audio_dir, fid))
        for ext in [".flac", ".wav", ".mp3", ".ogg", ".m4a"]:
            candidates.append(os.path.join(audio_dir, fid_base + ext))
        hit = next((p for p in candidates if p and os.path.isfile(p)), None)
        if hit is None:
            matches = glob.glob(os.path.join(audio_dir, fid_base + ".*"))
            if matches:
                hit = matches[0]
        if hit is None:
            raise FileNotFoundError(
                f"Audio tidak ditemukan: file_id={fid}; audio_dir={audio_dir}; audio_path={raw}"
            )
        out.append(hit)
    return out


def load_fixed_wave(path):
    wave, _ = librosa.load(path, sr=SR, mono=True, duration=DURATION)
    if wave.size == 0:
        raise ValueError(f"Audio kosong: {path}")
    if len(wave) < NUM_SAMPLES:
        wave = np.pad(wave, (0, NUM_SAMPLES - len(wave)))
    else:
        wave = wave[:NUM_SAMPLES]
    return np.asarray(wave, dtype=np.float32)


class PathWaveDataset(Dataset):
    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        return torch.from_numpy(load_fixed_wave(self.paths[idx]))


def download_file(url, destination):
    import urllib.request
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    temporary = destination + ".part"
    try:
        urllib.request.urlretrieve(url, temporary)
        if os.path.getsize(temporary) < 1000:
            raise RuntimeError(f"Unduhan terlalu kecil: {os.path.getsize(temporary)} bytes")
        os.replace(temporary, destination)
    finally:
        if os.path.exists(temporary):
            os.remove(temporary)


def ensure_beats_source():
    names = ["BEATs.py", "backbone.py", "modules.py"]
    for folder in [BEATS_CODE_DIR] + [
        os.path.dirname(p) for p in glob.glob("/kaggle/input/**/BEATs.py", recursive=True)
    ]:
        if all(os.path.isfile(os.path.join(folder, name)) for name in names):
            if folder not in sys.path:
                sys.path.insert(0, folder)
            return folder
    base_url = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"
    for name in names:
        target = os.path.join(BEATS_CODE_DIR, name)
        if not os.path.isfile(target):
            try:
                download_file(base_url + name, target)
            except Exception as exc:
                raise RuntimeError(
                    "Kode resmi BEATs tidak ditemukan. Aktifkan Internet atau lampirkan "
                    "BEATs.py, backbone.py, dan modules.py sebagai input Kaggle."
                ) from exc
    if BEATS_CODE_DIR not in sys.path:
        sys.path.insert(0, BEATS_CODE_DIR)
    return BEATS_CODE_DIR


def ensure_checkpoint():
    existing = find_file(CHECKPOINT_NAME, [OUT, SRC, "/kaggle/input"])
    if existing and os.path.getsize(existing) > 100_000_000:
        return existing
    destination = os.path.join(OUT, "pretrained", CHECKPOINT_NAME)
    if os.path.isfile(destination) and os.path.getsize(destination) > 100_000_000:
        return destination
    try:
        from huggingface_hub import hf_hub_download
        found = hf_hub_download(
            repo_id=CHECKPOINT_REPO_ID,
            filename=CHECKPOINT_NAME,
            local_dir=os.path.dirname(destination),
        )
        if os.path.isfile(found) and os.path.getsize(found) > 100_000_000:
            return found
    except Exception as exc:
        print("HF checkpoint download failed; trying direct public URL:", repr(exc))
    url = f"https://huggingface.co/{CHECKPOINT_REPO_ID}/resolve/main/{CHECKPOINT_NAME}"
    download_file(url, destination)
    return destination


def load_beats(checkpoint_path):
    ensure_beats_source()
    try:
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location="cpu")
    from BEATs import BEATs, BEATsConfig
    cfg = BEATsConfig(checkpoint["cfg"])
    model = BEATs(cfg)
    model.load_state_dict(checkpoint["model"], strict=True)
    model.eval().to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    device = next(model.parameters()).device
    original_preprocess = model.preprocess

    def preprocess_on_device(self, source, fbank_mean=15.41663, fbank_std=6.55582):
        fbanks = original_preprocess(source.detach().to("cpu"), fbank_mean=fbank_mean, fbank_std=fbank_std)
        return fbanks.to(device, non_blocking=True)

    model.preprocess = types.MethodType(preprocess_on_device, model)
    if getattr(model, "predictor", None) is not None:
        model.predictor = None
    print("Loaded BEATs encoder dim:", getattr(cfg, "encoder_embed_dim", "unknown"),
          "| layers:", getattr(cfg, "encoder_layers", "unknown"))
    return model


def extract_embeddings(model, paths, cache_path, split_name):
    cache_hit = find_file(os.path.basename(cache_path), [OUT, SRC, "/kaggle/input"])
    if cache_hit:
        try:
            arr = np.load(cache_hit, mmap_mode="r")
            if arr.ndim == 2 and arr.shape[0] == len(paths) and arr.shape[1] >= 256:
                arr = np.asarray(arr, dtype=np.float32)
                if os.path.abspath(cache_hit) != os.path.abspath(cache_path):
                    np.save(cache_path, arr)
                print(f"Reusing {split_name} embeddings {arr.shape}: {cache_hit}")
                return arr
        except Exception as exc:
            print(f"Cache could not be loaded ({cache_hit}):", repr(exc))

    dataset = PathWaveDataset(paths)
    loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=False, persistent_workers=(NUM_WORKERS > 0),
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    chunks = []
    start = time.time()
    model.eval()
    with torch.inference_mode():
        for batch_index, waves in enumerate(loader):
            waves = waves.to(device, non_blocking=True)
            if device.type == "cuda" and USE_AMP:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    features, _ = model.extract_features(waves.float())
            else:
                features, _ = model.extract_features(waves.float())
            if features.ndim != 3:
                raise RuntimeError(f"Unexpected BEATs feature shape: {tuple(features.shape)}")
            chunks.append(features.float().mean(dim=1).cpu().numpy().astype(np.float32))
            done = min((batch_index + 1) * BATCH_SIZE, len(dataset))
            if done % 200 < BATCH_SIZE or done == len(dataset):
                print(f"  {split_name}: {done}/{len(dataset)}")
    embeddings = np.concatenate(chunks, axis=0)
    np.save(cache_path, embeddings)
    print(f"Saved {split_name} embeddings {embeddings.shape} -> {cache_path}; {time.time()-start:.1f}s")
    return embeddings


def load_or_get_embeddings(train_meta, sample):
    train_path = find_file(TRAIN_EMB_NAME, [OUT, SRC, "/kaggle/input"])
    test_path = find_file(TEST_EMB_NAME, [OUT, SRC, "/kaggle/input"])
    train_cached = None
    test_cached = None
    if train_path:
        try:
            a = np.load(train_path, mmap_mode="r")
            if a.ndim == 2 and a.shape[0] == len(train_meta) and a.shape[1] >= 256:
                train_cached = np.asarray(a, dtype=np.float32)
                print("Found train BEATs embedding cache:", train_path, train_cached.shape)
        except Exception as exc:
            print("Could not read train embedding cache:", repr(exc))
    if test_path:
        try:
            a = np.load(test_path, mmap_mode="r")
            if a.ndim == 2 and a.shape[0] == len(sample) and a.shape[1] >= 256:
                test_cached = np.asarray(a, dtype=np.float32)
                print("Found test BEATs embedding cache:", test_path, test_cached.shape)
        except Exception as exc:
            print("Could not read test embedding cache:", repr(exc))

    if train_cached is not None and test_cached is not None:
        return train_cached, test_cached

    print("At least one embedding cache is missing; extracting only missing embeddings.")
    train_paths = resolve_audio_paths(train_meta, is_test=False) if train_cached is None else None
    test_paths = resolve_audio_paths(sample, is_test=True) if test_cached is None else None
    checkpoint_path = ensure_checkpoint()
    model = load_beats(checkpoint_path)
    if train_cached is None:
        train_cached = extract_embeddings(
            model, train_paths, os.path.join(OUT, TRAIN_EMB_NAME), "TRAIN BEATs embeddings"
        )
    if test_cached is None:
        test_cached = extract_embeddings(
            model, test_paths, os.path.join(OUT, TEST_EMB_NAME), "TEST BEATs embeddings"
        )
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return train_cached, test_cached


def make_embedding_scores(x_train, y_train, x_query, known_classes):
    """Score queries against the two known classes in one LOCO/fold run."""
    # Logistic-regression maximum probability: lower max probability => more OOD.
    classifier = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=LOGREG_C, max_iter=2000, class_weight="balanced", random_state=SEED),
    )
    classifier.fit(x_train, y_train)
    probabilities = classifier.predict_proba(x_query)
    max_probability = probabilities.max(axis=1)
    entropy = -(np.clip(probabilities, 1e-12, 1.0) * np.log(np.clip(probabilities, 1e-12, 1.0))).sum(axis=1)
    entropy_norm = entropy / np.log(max(probabilities.shape[1], 2))
    sorted_prob = np.sort(probabilities, axis=1)
    margin = sorted_prob[:, -1] - sorted_prob[:, -2] if probabilities.shape[1] > 1 else max_probability

    # Cosine prototype distance: nearest of the two class centroids in normalized space.
    xtr_norm = normalize(np.asarray(x_train, dtype=np.float32), norm="l2")
    xq_norm = normalize(np.asarray(x_query, dtype=np.float32), norm="l2")
    centers = []
    for cls in known_classes:
        center = xtr_norm[np.asarray(y_train) == cls].mean(axis=0)
        center = normalize(center.reshape(1, -1), norm="l2")[0]
        centers.append(center)
    centers = np.stack(centers, axis=0)
    cosine_sim = xq_norm @ centers.T
    centroid_distance = 1.0 - cosine_sim.max(axis=1)

    # kNN cosine distance to known reference embeddings.
    k = min(KNN_K, len(xtr_norm))
    knn = NearestNeighbors(n_neighbors=k, metric="cosine", algorithm="brute", n_jobs=-1)
    knn.fit(xtr_norm)
    knn_distances, _ = knn.kneighbors(xq_norm, return_distance=True)
    knn_distance = knn_distances.mean(axis=1)

    # Diagonal Mahalanobis distance to nearest class mean, fitted on known training data only.
    x_train = np.asarray(x_train, dtype=np.float64)
    x_query = np.asarray(x_query, dtype=np.float64)
    global_var = np.var(x_train, axis=0)
    floor = max(float(np.median(global_var)) * 1e-4, 1e-8)
    variance = np.maximum(global_var, floor)
    md_by_class = []
    for cls in known_classes:
        mu = x_train[np.asarray(y_train) == cls].mean(axis=0)
        d = x_query - mu[None, :]
        md = np.sqrt(np.maximum(np.mean((d * d) / variance[None, :], axis=1), 0.0))
        md_by_class.append(md)
    mahalanobis = np.min(np.stack(md_by_class, axis=1), axis=1)

    pred_label = classifier.predict(x_query)
    return {
        "max_probability": max_probability,
        "probability_entropy": entropy_norm,
        "inverse_margin": 1.0 - margin,
        "beats_centroid_distance": centroid_distance,
        "beats_knn_distance": knn_distance,
        "beats_diag_mahalanobis": mahalanobis,
        "classifier_predicted_label": pred_label,
        "classifier_known_accuracy": np.nan,  # filled in by the caller
    }


def safe_auc(y_true, score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    try:
        return float(roc_auc_score(y_true, score))
    except Exception:
        return np.nan


def run_loco(train_meta, embeddings, label_to_idx):
    """Group-folded leave-one-class-out validation, with scores higher = more unknown."""
    label_idx = train_meta["label"].map(label_to_idx).to_numpy(dtype=int)
    folds = train_meta["fold"].to_numpy(dtype=int)
    x = np.asarray(embeddings, dtype=np.float32)
    if len(x) != len(train_meta):
        raise ValueError("Training embedding rows do not align with metadata")

    records = []
    fold_records = []
    metric_direction = {
        "max_probability": -1.0,  # transform max probability into OOD score
        "probability_entropy": 1.0,
        "inverse_margin": 1.0,
        "beats_centroid_distance": 1.0,
        "beats_knn_distance": 1.0,
        "beats_diag_mahalanobis": 1.0,
    }

    for heldout_class in range(len(LABELS)):
        class_name = LABELS[heldout_class]
        print("\n" + "-" * 75)
        print(f"LOCO pseudo-Unknown class: {class_name}")
        print("Fit classifier on only the other two classes; test on held-out groups.")
        for fold in range(N_FOLDS):
            train_idx = np.flatnonzero((folds != fold) & (label_idx != heldout_class))
            val_idx = np.flatnonzero(folds == fold)
            if len(train_idx) == 0 or len(val_idx) == 0:
                raise ValueError(f"Empty train/validation split for heldout={class_name}, fold={fold}")
            known_classes = [i for i in range(len(LABELS)) if i != heldout_class]
            ytrain = label_idx[train_idx]
            yval = label_idx[val_idx]
            if set(np.unique(ytrain)) != set(known_classes):
                raise ValueError(
                    f"Fold {fold}, heldout {class_name}: train lacks one of known classes; "
                    f"got {np.unique(ytrain)}"
                )

            scores = make_embedding_scores(x[train_idx], ytrain, x[val_idx], known_classes)
            known_mask = yval != heldout_class
            known_acc = accuracy_score(yval[known_mask], scores["classifier_predicted_label"][known_mask]) if known_mask.any() else np.nan
            fold_records.append({
                "heldout_class": class_name,
                "fold": fold,
                "n_train_known": len(train_idx),
                "n_val_known": int(known_mask.sum()),
                "n_val_pseudo_unknown": int((~known_mask).sum()),
                "known_two_class_accuracy": known_acc,
            })

            for j, idx in enumerate(val_idx):
                actual_label = LABELS[yval[j]]
                is_unknown = bool(yval[j] == heldout_class)
                row = {
                    "file_id": train_meta.iloc[idx]["file_id"],
                    "source_group": train_meta.iloc[idx]["source_group"],
                    "fold": fold,
                    "heldout_class": class_name,
                    "actual_label": actual_label,
                    "is_pseudo_unknown": is_unknown,
                    "max_probability": float(scores["max_probability"][j]),
                    "probability_entropy": float(scores["probability_entropy"][j]),
                    "inverse_margin": float(scores["inverse_margin"][j]),
                    "beats_centroid_distance": float(scores["beats_centroid_distance"][j]),
                    "beats_knn_distance": float(scores["beats_knn_distance"][j]),
                    "beats_diag_mahalanobis": float(scores["beats_diag_mahalanobis"][j]),
                    "classifier_predicted_label": LABELS[int(scores["classifier_predicted_label"][j])],
                }
                records.append(row)
            print(
                f"  Fold {fold}: train={len(train_idx)}, val={len(val_idx)}, "
                f"pseudo-Unknown={int((~known_mask).sum())}, known accuracy={known_acc:.4f}"
            )

    detail = pd.DataFrame(records)
    fold_detail = pd.DataFrame(fold_records)
    methods = {
        "max_probability_as_ood": 1.0 - detail["max_probability"].to_numpy(),
        "probability_entropy": detail["probability_entropy"].to_numpy(),
        "inverse_margin": detail["inverse_margin"].to_numpy(),
        "beats_centroid_distance": detail["beats_centroid_distance"].to_numpy(),
        "beats_knn_distance": detail["beats_knn_distance"].to_numpy(),
        "beats_diag_mahalanobis": detail["beats_diag_mahalanobis"].to_numpy(),
    }
    y_unknown = detail["is_pseudo_unknown"].astype(int).to_numpy()
    summaries = []

    # Aggregate metrics per withheld class, and overall pooled LOCO metrics.
    for method_name, values in methods.items():
        summaries.append({
            "heldout_class": "ALL_PSEUDO_UNKNOWN_CLASSES",
            "method": method_name,
            "AUROC_unknown_higher": safe_auc(y_unknown, values),
            "average_precision": float(average_precision_score(y_unknown, values)),
            "TPR_at_5pct_known_FPR": np.nan,
            "TPR_at_10pct_known_FPR": np.nan,
        })
        for cls_name in LABELS:
            mask = detail["heldout_class"].eq(cls_name).to_numpy()
            y_cls = y_unknown[mask]
            s_cls = values[mask]
            known_scores = s_cls[y_cls == 0]
            unknown_scores = s_cls[y_cls == 1]
            tpr5 = tpr10 = np.nan
            if len(known_scores) and len(unknown_scores):
                threshold5 = np.quantile(known_scores, 0.95)
                threshold10 = np.quantile(known_scores, 0.90)
                tpr5 = float(np.mean(unknown_scores >= threshold5))
                tpr10 = float(np.mean(unknown_scores >= threshold10))
            summaries.append({
                "heldout_class": cls_name,
                "method": method_name,
                "AUROC_unknown_higher": safe_auc(y_cls, s_cls),
                "average_precision": float(average_precision_score(y_cls, s_cls)) if len(np.unique(y_cls)) == 2 else np.nan,
                "TPR_at_5pct_known_FPR": tpr5,
                "TPR_at_10pct_known_FPR": tpr10,
            })

    summary = pd.DataFrame(summaries)
    # Add an unweighted macro-average across the three held-out species. The
    # pooled row is also reported, but its AUROC can be influenced by score-scale
    # differences between the three separate binary LOCO experiments.
    class_rows = summary[summary["heldout_class"].isin(LABELS)]
    macro_row = {"heldout_class": "MACRO_AVG_HELDOUT_CLASS"}
    for col in ["method", "AUROC_unknown_higher", "average_precision", "TPR_at_5pct_known_FPR", "TPR_at_10pct_known_FPR"]:
        if col == "method":
            continue
        macro_row[col] = np.nan
    macro_rows = []
    for method_name in methods.keys():
        sub = class_rows[class_rows["method"] == method_name]
        macro_rows.append({
            "heldout_class": "MACRO_AVG_HELDOUT_CLASS",
            "method": method_name,
            "AUROC_unknown_higher": float(sub["AUROC_unknown_higher"].mean()),
            "average_precision": float(sub["average_precision"].mean()),
            "TPR_at_5pct_known_FPR": float(sub["TPR_at_5pct_known_FPR"].mean()),
            "TPR_at_10pct_known_FPR": float(sub["TPR_at_10pct_known_FPR"].mean()),
        })
    summary = pd.concat([summary, pd.DataFrame(macro_rows)], ignore_index=True)
    return detail, fold_detail, summary


def load_triple_v4_probabilities(train_meta, sample):
    paths = {k: find_file(v, [OUT, SRC, "/kaggle/input"]) for k, v in ARTIFACTS.items()}
    missing = [f"{k}: {ARTIFACTS[k]}" for k, v in paths.items() if v is None]
    if missing:
        raise FileNotFoundError(
            "Triple V4 probability artifacts tidak ditemukan. Lampirkan output notebook testttrisan "
            "sebagai input Kaggle. File hilang:\n" + "\n".join(missing)
        )
    oof_r = normalize_probs(np.load(paths["oof_resnet"]))
    oof_b = normalize_probs(np.load(paths["oof_b2"]))
    oof_c = normalize_probs(np.load(paths["oof_convnext"]))
    with np.load(paths["test_resnet_crops"]) as z:
        test_r5 = np.asarray(z["test_r5"][:, [0, 2, 4], :], dtype=np.float64).mean(axis=1)
        test_r8 = np.asarray(z["test_r8"][:, [0, 2, 4], :], dtype=np.float64).mean(axis=1)
    test_r = normalize_probs(0.25 * test_r5 + 0.75 * test_r8)
    test_b = normalize_probs(np.load(paths["test_b2"]))
    test_c = normalize_probs(np.load(paths["test_convnext"]))

    expected_oof = (len(train_meta), len(LABELS))
    expected_test = (len(sample), len(LABELS))
    for name, array in [("OOF ResNet", oof_r), ("OOF B2", oof_b), ("OOF ConvNeXt", oof_c)]:
        if array.shape != expected_oof:
            raise ValueError(f"{name} probability shape {array.shape}; expected {expected_oof}")
    for name, array in [("Test ResNet", test_r), ("Test B2", test_b), ("Test ConvNeXt", test_c)]:
        if array.shape != expected_test:
            raise ValueError(f"{name} probability shape {array.shape}; expected {expected_test}")

    # Triple V4 fixed known-class blend, same as earlier OOD diagnostics.
    p_oof = normalize_probs(0.55 * oof_r + 0.05 * oof_b + 0.40 * oof_c)
    p_test = normalize_probs(0.55 * test_r + 0.05 * test_b + 0.40 * test_c)
    return p_oof, p_test


def quantile_summary(values, prefix):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if not len(values):
        return {f"{prefix}_n": 0}
    out = {f"{prefix}_n": int(len(values)), f"{prefix}_mean": float(np.mean(values)), f"{prefix}_std": float(np.std(values))}
    for q in [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]:
        out[f"{prefix}_q{int(q*100):02d}"] = float(np.quantile(values, q))
    return out


def analyze_confidence_calibration(train_meta, sample, y_idx, p_oof, p_test):
    oof_pred = p_oof.argmax(axis=1)
    test_pred = p_test.argmax(axis=1)
    oof_conf = p_oof.max(axis=1)
    test_conf = p_test.max(axis=1)
    oof_sorted = np.sort(p_oof, axis=1)
    test_sorted = np.sort(p_test, axis=1)
    oof_margin = oof_sorted[:, -1] - oof_sorted[:, -2]
    test_margin = test_sorted[:, -1] - test_sorted[:, -2]
    oof_ent = entropy_norm(p_oof)
    test_ent = entropy_norm(p_test)

    pred_dist_rows = []
    class_rows = []
    bin_rows = []
    q_rows = []

    # Predicted-class distribution and confidence summaries (OOF compared with test).
    for split_name, meta, probs, pred, conf, margin, ent in [
        ("OOF_known", train_meta, p_oof, oof_pred, oof_conf, oof_margin, oof_ent),
        ("TEST_unlabeled", sample, p_test, test_pred, test_conf, test_margin, test_ent),
    ]:
        for class_idx, class_name in enumerate(LABELS):
            mask = pred == class_idx
            n = int(mask.sum())
            row = {
                "split": split_name,
                "predicted_class": class_name,
                "count": n,
                "share": float(n / len(meta)),
                "mean_max_probability": float(conf[mask].mean()) if n else np.nan,
                "median_max_probability": float(np.median(conf[mask])) if n else np.nan,
                "mean_margin_top1_top2": float(margin[mask].mean()) if n else np.nan,
                "mean_normalized_entropy": float(ent[mask].mean()) if n else np.nan,
                "mean_probability_for_class": float(probs[mask, class_idx].mean()) if n else np.nan,
            }
            row.update(quantile_summary(conf[mask], "maxprob"))
            pred_dist_rows.append(row)

            # Confidence bins within predictions to this class.
            edges = [0.0, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0000001]
            cats = pd.cut(conf[mask], bins=edges, right=False, include_lowest=True)
            for interval, cnt in pd.Series(cats).value_counts(sort=False).items():
                bin_rows.append({
                    "split": split_name,
                    "predicted_class": class_name,
                    "confidence_bin": str(interval),
                    "count": int(cnt),
                    "share_within_predicted_class": float(cnt / n) if n else np.nan,
                })

    # Class-wise reference distributions from OOF: true class rows, plus correctly classified subset.
    for class_idx, class_name in enumerate(LABELS):
        true_mask = y_idx == class_idx
        oof_correct_mask = true_mask & (oof_pred == class_idx)
        test_pred_mask = test_pred == class_idx

        ref_true_prob = p_oof[true_mask, class_idx]
        ref_correct_prob = p_oof[oof_correct_mask, class_idx]
        test_class_prob = p_test[test_pred_mask, class_idx]

        # Use both references; all-true-class OOF includes known-class confusions,
        # correctly-classified reference is a narrower/high-confidence diagnostic.
        q_ref = {}
        for label, arr in [("OOF_true_class_prob", ref_true_prob), ("OOF_correct_prob", ref_correct_prob)]:
            q = np.quantile(arr, [0.05, 0.10, 0.25, 0.50]) if len(arr) else [np.nan] * 4
            q_ref[label] = q
        row = {
            "class": class_name,
            "train_true_count": int(true_mask.sum()),
            "train_true_share": float(true_mask.mean()),
            "oof_predicted_count": int((oof_pred == class_idx).sum()),
            "oof_predicted_share": float((oof_pred == class_idx).mean()),
            "oof_true_class_recall": float(oof_correct_mask.sum() / true_mask.sum()) if true_mask.sum() else np.nan,
            "test_predicted_count": int(test_pred_mask.sum()),
            "test_predicted_share": float(test_pred_mask.mean()),
            "test_soft_probability_mass": float(p_test[:, class_idx].sum()),
            "test_mean_probability_for_predicted_class": float(test_class_prob.mean()) if len(test_class_prob) else np.nan,
            "test_median_probability_for_predicted_class": float(np.median(test_class_prob)) if len(test_class_prob) else np.nan,
        }
        for ref_name, q in q_ref.items():
            for name, val in zip(["q05", "q10", "q25", "q50"], q):
                row[f"{ref_name}_{name}"] = float(val) if np.isfinite(val) else np.nan
            arr_test = test_class_prob
            for name, val in zip(["q05", "q10", "q25", "q50"], q):
                row[f"test_share_below_{ref_name}_{name}"] = float(np.mean(arr_test < val)) if len(arr_test) and np.isfinite(val) else np.nan
        q_rows.append(row)

        # OOF distribution conditioned on true label vs TEST distribution conditioned on predicted label.
        class_rows.append({
            "class": class_name,
            "train_true_share": float(true_mask.mean()),
            "oof_true_class_mean_probability": float(ref_true_prob.mean()) if len(ref_true_prob) else np.nan,
            "oof_true_class_median_probability": float(np.median(ref_true_prob)) if len(ref_true_prob) else np.nan,
            "oof_correct_only_mean_probability": float(ref_correct_prob.mean()) if len(ref_correct_prob) else np.nan,
            "test_predicted_count": int(test_pred_mask.sum()),
            "test_predicted_share": float(test_pred_mask.mean()),
            "test_predicted_class_mean_probability": float(test_class_prob.mean()) if len(test_class_prob) else np.nan,
            "test_predicted_class_median_probability": float(np.median(test_class_prob)) if len(test_class_prob) else np.nan,
            "test_soft_probability_mass": float(p_test[:, class_idx].sum()),
            "reference_count_if_all_test_known_same_training_prior": float(len(sample) * true_mask.mean()),
        })

    # Known OOF overall correct-class confidence distribution as a reference.
    oof_true_pred = p_oof[np.arange(len(p_oof)), y_idx]
    oof_correct = oof_pred == y_idx
    test_mass = p_test.sum(axis=0)
    diagnostics = {
        "n_train": int(len(train_meta)),
        "n_test": int(len(sample)),
        "triple_v4_oof_macro_f1_known_only": float(f1_score(y_idx, oof_pred, average="macro")),
        "triple_v4_oof_accuracy_known_only": float(accuracy_score(y_idx, oof_pred)),
        "oof_known_class_correct_rate": float(oof_correct.mean()),
        "oof_true_class_probability_mean": float(oof_true_pred.mean()),
        "test_soft_probability_mass_total": float(test_mass.sum()),
        "warning": "Test class prior can differ from training; prior-scaled counts are diagnostic only, not Unknown-count estimates.",
    }

    return (
        pd.DataFrame(pred_dist_rows), pd.DataFrame(class_rows), pd.DataFrame(bin_rows),
        pd.DataFrame(q_rows), diagnostics,
    )


def main():
    seed_everything()
    print("=" * 86)
    print("SCHEMATICS NPC DSC 2026 — Triple V4 LOCO + CONFIDENCE CALIBRATION")
    print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"),
          "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
    print("Unknown definition: bird species absent from the three training classes")
    print("No CNN training; no submission is created or sent.")
    print("=" * 86)

    for required_path in [TRAIN_META_PATH, SAMPLE_PATH, FOLD_PATH]:
        if not os.path.isfile(required_path):
            raise FileNotFoundError(f"Required input missing: {required_path}")

    train_meta = pd.read_csv(TRAIN_META_PATH)
    sample = pd.read_csv(SAMPLE_PATH)
    for df in [train_meta, sample]:
        df["file_id"] = df["file_id"].astype(str)
    train_meta["label"] = train_meta["label"].astype(str)
    if "source_group" not in train_meta.columns:
        raise ValueError("train metadata does not have source_group column")
    if train_meta["file_id"].duplicated().any() or sample["file_id"].duplicated().any():
        raise ValueError("Duplicate file_id found")
    if sorted(train_meta["label"].unique().tolist()) != sorted(LABELS):
        raise ValueError(f"Unexpected training labels: {train_meta['label'].unique().tolist()}")

    fold_df = pd.read_csv(FOLD_PATH)
    if not {"file_id", "fold"}.issubset(fold_df.columns):
        raise ValueError("Saved fold file must contain file_id, fold")
    fold_df["file_id"] = fold_df["file_id"].astype(str)
    if fold_df["file_id"].duplicated().any() or len(fold_df) != len(train_meta):
        raise ValueError("Saved group folds do not align one-to-one with train metadata")
    train_meta["fold"] = train_meta["file_id"].map(fold_df.set_index("file_id")["fold"])
    if train_meta["fold"].isna().any():
        raise ValueError("Some train metadata file_id values do not have a fold")
    train_meta["fold"] = train_meta["fold"].astype(int)
    if sorted(train_meta["fold"].unique().tolist()) != list(range(N_FOLDS)):
        raise ValueError(f"Expected fold IDs 0..{N_FOLDS-1}")
    group_fold_counts = train_meta.groupby("source_group")["fold"].nunique()
    leakage_groups = int((group_fold_counts > 1).sum())
    if leakage_groups:
        raise ValueError(
            f"Leakage guard failed: {leakage_groups} source_group values span multiple folds"
        )
    print("Train rows:", len(train_meta), "| test rows:", len(sample))
    print("Class counts:", train_meta["label"].value_counts().to_dict())
    print("Fold counts:", train_meta["fold"].value_counts().sort_index().to_dict())
    print("Source-group leakage across folds:", leakage_groups)

    label_to_idx = {label: i for i, label in enumerate(LABELS)}
    y_idx = train_meta["label"].map(label_to_idx).to_numpy(dtype=int)

    embeddings, test_embeddings = load_or_get_embeddings(train_meta, sample)
    print("Embedding shapes:", embeddings.shape, test_embeddings.shape)

    # Part A: LOCO open-set proxy benchmark.
    loco_detail, loco_fold_detail, loco_summary = run_loco(train_meta, embeddings, label_to_idx)
    loco_detail_path = os.path.join(OUT, "triple_v4_beats_loco_detail.csv")
    loco_fold_path = os.path.join(OUT, "triple_v4_beats_loco_fold_metrics.csv")
    loco_summary_path = os.path.join(OUT, "triple_v4_beats_loco_summary.csv")
    loco_detail.to_csv(loco_detail_path, index=False)
    loco_fold_detail.to_csv(loco_fold_path, index=False)
    loco_summary.to_csv(loco_summary_path, index=False)

    print("\n" + "=" * 86)
    print("LEAVE-ONE-CLASS-OUT OOD SUMMARY (scores higher = more pseudo-Unknown)")
    print("=" * 86)
    show = loco_summary.copy()
    print(show.round(4).to_string(index=False))
    print("\nFold known-pair accuracy:")
    print(loco_fold_detail.groupby("heldout_class")["known_two_class_accuracy"].agg(["mean", "std", "min", "max"]).round(4).to_string())

    # Part B: fixed Triple V4 OOF-vs-test confidence and class distribution.
    p_oof, p_test = load_triple_v4_probabilities(train_meta, sample)
    oof_pred = p_oof.argmax(axis=1)
    print("\nTriple V4 OOF known-only Macro F1:", round(float(f1_score(y_idx, oof_pred, average="macro")), 6))
    print("Triple V4 OOF known-only accuracy:", round(float(accuracy_score(y_idx, oof_pred)), 6))

    pred_dist, class_shift, confidence_bins, class_quantile, diagnostics = analyze_confidence_calibration(
        train_meta, sample, y_idx, p_oof, p_test
    )
    pred_dist_path = os.path.join(OUT, "triple_v4_oof_vs_test_predicted_class_confidence.csv")
    class_shift_path = os.path.join(OUT, "triple_v4_oof_vs_test_class_distribution.csv")
    bins_path = os.path.join(OUT, "triple_v4_oof_vs_test_confidence_bins.csv")
    quantile_path = os.path.join(OUT, "triple_v4_oof_vs_test_class_confidence_quantiles.csv")
    import json
    diagnostics_path = os.path.join(OUT, "triple_v4_oof_vs_test_confidence_metrics.json")
    pred_dist.to_csv(pred_dist_path, index=False)
    class_shift.to_csv(class_shift_path, index=False)
    confidence_bins.to_csv(bins_path, index=False)
    class_quantile.to_csv(quantile_path, index=False)
    with open(diagnostics_path, "w", encoding="utf-8") as f:
        json.dump(diagnostics, f, indent=2, ensure_ascii=False)

    print("\n" + "=" * 86)
    print("TRIPLE V4 OOF VS TEST — PREDICTED-CLASS CONFIDENCE")
    print("=" * 86)
    cols1 = ["split", "predicted_class", "count", "share", "mean_max_probability", "median_max_probability", "mean_margin_top1_top2", "mean_normalized_entropy"]
    print(pred_dist[cols1].round(4).to_string(index=False))

    print("\n" + "=" * 86)
    print("CLASS DISTRIBUTION / PROBABILITY-MASS SHIFT (diagnostic, not Unknown estimate)")
    print("=" * 86)
    print(class_shift.round(4).to_string(index=False))

    print("\n" + "=" * 86)
    print("TEST CONFIDENCE BELOW OOF CLASS-CONDITIONAL QUANTILES")
    print("=" * 86)
    important_cols = [
        "class", "train_true_count", "oof_predicted_count", "test_predicted_count",
        "test_predicted_share", "test_soft_probability_mass",
        "test_share_below_OOF_true_class_prob_q10", "test_share_below_OOF_true_class_prob_q25",
        "test_share_below_OOF_correct_prob_q10", "test_share_below_OOF_correct_prob_q25",
    ]
    important_cols = [c for c in important_cols if c in class_quantile.columns]
    print(class_quantile[important_cols].round(4).to_string(index=False))

    print("\nSaved diagnostics:")
    for p in [loco_detail_path, loco_fold_path, loco_summary_path, pred_dist_path, class_shift_path, bins_path, quantile_path, diagnostics_path]:
        print(" -", p)
    print("\nReminder: LOCO is a proxy, and test class priors may differ from train.")
    print("No submission CSV was created or sent.")


if __name__ == "__main__":
    main()


SCHEMATICS NPC DSC 2026 — Triple V4 LOCO + CONFIDENCE CALIBRATION
Device: cuda | GPU: Tesla T4
Unknown definition: bird species absent from the three training classes
No CNN training; no submission is created or sent.
Train rows: 1826 | test rows: 814
Class counts: {'Bondol Jawa': 1124, 'Kutilang': 619, 'Cendrawasih': 83}
Fold counts: {0: 284, 1: 495, 2: 553, 3: 494}
Source-group leakage across folds: 0
Found train BEATs embedding cache: /kaggle/working/beats_iter3plus_as2m_8s_train_embeddings.npy (1826, 768)
Found test BEATs embedding cache: /kaggle/working/beats_iter3plus_as2m_8s_test_embeddings.npy (814, 768)
Embedding shapes: (1826, 768) (814, 768)

---------------------------------------------------------------------------
LOCO pseudo-Unknown class: Bondol Jawa
Fit classifier on only the other two classes; test on held-out groups.
  Fold 0: train=535, val=284, pseudo-Unknown=117, known accuracy=0.9820
  Fold 1: train=521, val=495, pseudo-Unknown=314, known accuracy=0.9945
  Fold 2

In [26]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

PATH = "/kaggle/working/triple_v4_beats_loco_detail.csv"
OUT = "/kaggle/working"

df = pd.read_csv(PATH)

required = {
    "heldout_class", "fold", "is_pseudo_unknown",
    "max_probability", "beats_knn_distance"
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Kolom tidak ditemukan: {missing}")

# Pastikan label pseudo-Unknown terbaca sebagai boolean dengan benar.
if df["is_pseudo_unknown"].dtype != bool:
    df["is_pseudo_unknown"] = (
        df["is_pseudo_unknown"].astype(str).str.strip().str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
    )

if df["is_pseudo_unknown"].isna().any():
    raise ValueError("Kolom is_pseudo_unknown mengandung nilai tak dikenali.")

df["max_probability"] = pd.to_numeric(
    df["max_probability"], errors="coerce"
)
df["beats_knn_distance"] = pd.to_numeric(
    df["beats_knn_distance"], errors="coerce"
)
df = df.dropna(
    subset=["max_probability", "beats_knn_distance", "heldout_class", "fold"]
).copy()

# Semakin tinggi skor, semakin besar dugaan pseudo-Unknown.
df["inverse_maxprob"] = 1.0 - df["max_probability"]

# Ranking percentile di dalam setiap skenario heldout-class dan fold.
group_cols = ["heldout_class", "fold"]

df["rank_maxprob"] = df.groupby(group_cols)["inverse_maxprob"].rank(
    pct=True, method="average"
)
df["rank_knn"] = df.groupby(group_cols)["beats_knn_distance"].rank(
    pct=True, method="average"
)

def safe_auc(y, s):
    return roc_auc_score(y, s) if len(np.unique(y)) == 2 else np.nan

rows = []

for w_knn in np.round(np.arange(0.0, 1.01, 0.1), 2):
    # 0% = inverse max-probability saja; 100% = kNN saja.
    score = (
        (1.0 - w_knn) * df["rank_maxprob"].to_numpy()
        + w_knn * df["rank_knn"].to_numpy()
    )

    y = df["is_pseudo_unknown"].astype(int).to_numpy()
    pooled_auc = safe_auc(y, score)
    pooled_ap = average_precision_score(y, score)

    per_class_auc = []
    per_class_ap = []
    class_tpr5 = []

    tmp = df[["heldout_class", "fold", "is_pseudo_unknown"]].copy()
    tmp["score"] = score

    # AUROC dan AP per spesies yang disisihkan.
    for heldout_class, g in tmp.groupby("heldout_class"):
        yy = g["is_pseudo_unknown"].astype(int).to_numpy()
        ss = g["score"].to_numpy()

        per_class_auc.append(safe_auc(yy, ss))
        per_class_ap.append(average_precision_score(yy, ss))

        # Ambang 5% false-positive rate dihitung pada known di tiap fold.
        hits = 0
        total_unknown = 0

        for _, fg in g.groupby("fold"):
            known_scores = fg.loc[~fg["is_pseudo_unknown"], "score"].to_numpy()
            unknown_scores = fg.loc[fg["is_pseudo_unknown"], "score"].to_numpy()

            if len(known_scores) == 0 or len(unknown_scores) == 0:
                continue

            threshold = np.quantile(known_scores, 0.95)
            hits += int((unknown_scores >= threshold).sum())
            total_unknown += len(unknown_scores)

        class_tpr5.append(
            hits / total_unknown if total_unknown else np.nan
        )

    rows.append({
        "knn_weight": w_knn,
        "maxprob_weight": 1.0 - w_knn,
        "pooled_auroc": pooled_auc,
        "pooled_average_precision": pooled_ap,
        "macro_heldout_class_auroc": np.nanmean(per_class_auc),
        "macro_heldout_class_ap": np.nanmean(per_class_ap),
        "macro_TPR_at_5pct_known_FPR": np.nanmean(class_tpr5),
    })

result = pd.DataFrame(rows).sort_values(
    ["macro_heldout_class_auroc", "macro_TPR_at_5pct_known_FPR"],
    ascending=False
)

out_path = os.path.join(OUT, "triple_v4_loco_maxprob_knn_weight_search.csv")
result.to_csv(out_path, index=False)

print("=" * 100)
print("LOCO — MAX PROBABILITY VS BEATS kNN RANK FUSION")
print("=" * 100)
print(result.round(4).to_string(index=False))
print("\nSaved:", out_path)
print("\nDiagnostik saja; belum ada submission dibuat.")

LOCO — MAX PROBABILITY VS BEATS kNN RANK FUSION
 knn_weight  maxprob_weight  pooled_auroc  pooled_average_precision  macro_heldout_class_auroc  macro_heldout_class_ap  macro_TPR_at_5pct_known_FPR
        0.5             0.5        0.6849                    0.4827                     0.7636                  0.4912                       0.2587
        0.6             0.4        0.6742                    0.4742                     0.7633                  0.4913                       0.2541
        0.7             0.3        0.6595                    0.4622                     0.7550                  0.4894                       0.2780
        0.4             0.6        0.6885                    0.4850                     0.7501                  0.4897                       0.2490
        0.8             0.2        0.6427                    0.4476                     0.7426                  0.4888                       0.3288
        0.3             0.7        0.6859                    0.4

In [27]:
import os
import numpy as np
import pandas as pd

ROOT = "/kaggle/input/competitions/schematics-2026"
OUT = "/kaggle/working"

SCORE_PATH = os.path.join(
    OUT, "triple_v4_nonbird_ood_test_scores.csv"
)
SAMPLE_PATH = os.path.join(ROOT, "sample_submission.csv")

N_UNKNOWN = 245
W_UNCERTAINTY = 0.50
W_KNN = 0.50

scores = pd.read_csv(SCORE_PATH)
sample = pd.read_csv(SAMPLE_PATH)

required = {
    "file_id",
    "triple_v4_pred_known_class",
    "triple_v4_confidence",
    "beats_knn_cosine_distance",
}
missing = required - set(scores.columns)
if missing:
    raise ValueError(f"Kolom tidak ditemukan: {missing}")

scores["file_id"] = scores["file_id"].astype(str)
sample["file_id"] = sample["file_id"].astype(str)

if scores["file_id"].duplicated().any():
    raise ValueError("file_id duplikat pada skor test.")

if set(scores["file_id"]) != set(sample["file_id"]):
    raise ValueError("file_id tidak cocok dengan sample_submission.")

if len(sample) != 814:
    print(f"Peringatan: jumlah baris test = {len(sample)}, bukan 814.")

confidence = pd.to_numeric(
    scores["triple_v4_confidence"], errors="coerce"
)
knn_distance = pd.to_numeric(
    scores["beats_knn_cosine_distance"], errors="coerce"
)

if not np.isfinite(confidence).all():
    raise ValueError("Confidence mengandung nilai yang tidak valid.")
if not np.isfinite(knn_distance).all():
    raise ValueError("KNN distance mengandung nilai yang tidak valid.")

# Semakin kecil confidence, semakin tinggi uncertainty.
# Semakin besar jarak kNN, semakin kuat dugaan OOD.
uncertainty_rank = (
    pd.Series(-confidence).rank(pct=True).to_numpy()
)
knn_rank = (
    pd.Series(knn_distance).rank(pct=True).to_numpy()
)

ood_score = (
    W_UNCERTAINTY * uncertainty_rank
    + W_KNN * knn_rank
)

scores["ood_score_conf_knn50"] = ood_score

# Pilih 245 skor tertinggi; ties dipecahkan deterministik.
selected = scores.sort_values(
    ["ood_score_conf_knn50", "file_id"],
    ascending=[False, True],
    kind="mergesort",
).head(N_UNKNOWN)

unknown_ids = set(selected["file_id"])

# Pertahankan prediksi kelas Triple V4 untuk file lainnya.
pred_map = scores.set_index("file_id")["triple_v4_pred_known_class"]
prediction = sample["file_id"].map(pred_map)

if prediction.isna().any():
    raise ValueError("Ada prediksi kelas yang hilang.")

prediction = prediction.astype(str)
prediction[sample["file_id"].isin(unknown_ids)] = "Unknown"

target_cols = [c for c in sample.columns if c != "file_id"]
if len(target_cols) != 1:
    raise ValueError(f"Kolom target tidak dikenal: {sample.columns.tolist()}")

submission = sample.copy()
submission[target_cols[0]] = prediction

output_path = os.path.join(
    OUT, "submission_triple_v4_conf_knn50_u245.csv"
)
submission.to_csv(output_path, index=False)

audit_path = os.path.join(
    OUT, "triple_v4_conf_knn50_selected_245.csv"
)
selected.to_csv(audit_path, index=False)

# Validasi akhir
check = pd.read_csv(output_path)
assert len(check) == len(sample)
assert check["file_id"].equals(sample["file_id"])
assert check[target_cols[0]].notna().all()
assert (check[target_cols[0]] == "Unknown").sum() == N_UNKNOWN
assert set(check[target_cols[0]].unique()).issubset(
    {"Bondol Jawa", "Cendrawasih", "Kutilang", "Unknown"}
)

print("=" * 65)
print("TRIPLE V4 + BEATs kNN 50/50 — KANDIDAT OFFLINE")
print("=" * 65)
print("Jumlah Unknown:", int((prediction == "Unknown").sum()))
print("\nDistribusi kelas awal pada 245 kandidat:")
print(selected["triple_v4_pred_known_class"].value_counts().to_string())
print("\nDistribusi prediksi final:")
print(pd.Series(prediction).value_counts().to_string())

# Bandingkan pilihan dengan metode Triple V4 lama, jika tersedia.
old_col = "ood_rank_score_triple_v4_uncertainty_disagreement_60_40"
if old_col in scores.columns:
    old_top = set(
        scores.nlargest(N_UNKNOWN, old_col)["file_id"]
    )
    overlap = len(unknown_ids & old_top)
    print(f"\nOverlap dengan metode lama: {overlap}/{N_UNKNOWN}")
    print(f"Jaccard overlap: {overlap / len(unknown_ids | old_top):.4f}")

print("\nCSV kandidat:", output_path)
print("Audit pilihan:", audit_path)
print("Validasi format: LULUS")
print("Status: BELUM DISUBMIT")

TRIPLE V4 + BEATs kNN 50/50 — KANDIDAT OFFLINE
Jumlah Unknown: 245

Distribusi kelas awal pada 245 kandidat:
triple_v4_pred_known_class
Cendrawasih    105
Bondol Jawa     92
Kutilang        48

Distribusi prediksi final:
file_id
Kutilang       263
Unknown        245
Bondol Jawa    229
Cendrawasih     77

Overlap dengan metode lama: 157/245
Jaccard overlap: 0.4715

CSV kandidat: /kaggle/working/submission_triple_v4_conf_knn50_u245.csv
Audit pilihan: /kaggle/working/triple_v4_conf_knn50_selected_245.csv
Validasi format: LULUS
Status: BELUM DISUBMIT
